# Multi-Label Persuasion-CATEGORY Classifier -> Donation-Outcome Analysis (Phase 3 + 4)

**Environment: Kaggle Notebook, accelerator = GPU `T4 x2`, Internet ON.**

**This is the category-level pivot of the strategy-level pipeline.** The
project's taxonomy has 41 fine-grained strategies grouped into 11 coarser
categories. Every earlier revision of this notebook predicted the 41 (later
39) strategies directly. This revision predicts the **11 categories**
instead, discarding fine-grained strategy prediction entirely -- the
trade-off being no "which specific strategy" answer, in exchange for every
class having far more training examples than the rarest strategies ever did.

This notebook **fine-tunes text encoders** (RoBERTa base *and* large, ELECTRA
and TOD-BERT, at two context widths and two seeds, head-to-head) plus
TF-IDF -> XGBoost / logistic members, then searches a **zoo of ensembling and
stacking schemes** over all of them, to tag each persuader turn with the
**set** of persuasion categories it uses -- and rebuilds the
donation-probability analysis on top of those multi-label category features.

Everything that can be chosen is chosen on **grouped, cross-fitted validation**
and never on test: which encoder, which context width, which decision rule,
whether to force at least one label per turn, how to combine the members, and
whether a level-2 stacker beats a plain blend at all.

It follows the engineering conventions of the donation-intent classifiers in
`Intention/Classifier/` (`v2 - speaker-aware-encoding`, `v4 - data_aug+left_trunc`,
`v8 - hierarchical+aug`): one `CONFIG` block, a hand-written PyTorch training loop,
speaker-role embeddings read from the `[Persuader]`/`[Persuadee]` markers,
left truncation so the turn being classified is never cut, post-split
augmentation for starved classes, a class-balanced / focal-style loss, and every
baseline computed on the *same* test split. The section layout and the Phase-4
statistics follow `kaggle_phase3_4_pipeline.ipynb`.

---

## Task

One head on a shared, speaker-aware encoder: 11-way **multi-label** (sigmoid)
-- which of the 11 taxonomy categories does this turn use? A turn belongs to
a category iff any of its (raw, gold-labelled) fine-grained strategies does,
via the static strategy -> category map (`S2C`). No fine-grained strategy
prediction anywhere in this revision, and no separate category *head* on top
of a strategy head either -- category is simply this pipeline's one and only
target, reusing every generic piece of machinery (loss, decode-rule search,
ensemble zoo, per-label reporting) that earlier revisions wrote against a
variable called `STRATS`.

Two features that don't exist for a live, turn-by-turn deployment are never
built at all: an `[A:<annotator>]` rater-ID tag on the input text (a labeling
artifact, not a property of the conversation), and whole-dialogue
(past-**and-future**) probability smoothing at decode time.

## Data

The Phase-2 **human-corrected** multi-label ground truth for all **10,600
persuader turns** -- `multilabel_merged.jsonl` (4 annotator segments, 1,017
dialogues). Categories are not separately annotated -- they are a fixed
lookup from the same fine-grained strategy labels.

Optional joins, auto-discovered and **degraded gracefully if absent**:

| file | adds | without it |
|---|---|---|
| `persuader_turns.csv` | the <=5-turn `context`, and the normalized dialogue-level donation outcome (`binary_label_norm`, `modifier_norm`) | turns are classified without dialogue context; **Phase 4 is skipped** |
| `taxonomy_multilabel.json` | the taxonomy | bundled fallback (verified identical) |
| `Manual_Label_normalized.csv` | persuadee sentiment / engagement covariates | those covariates are set to 0 |

## Split

**Dialogue-level** 70/15/15, stratified on the dialogue's aggregated category
vector (`MultilabelStratifiedKFold`, falls back to a grouped random split). Never
turn-level -- context overlap between turns of one conversation would leak.

## Class imbalance

Three layers: (1) **Asymmetric Loss** (Ben-Baruch et al. 2021) -- the multi-label
analogue of v8's class-balanced focal loss; it down-weights easy negatives and
hard-clips very-low-probability negatives; (2) optional per-label
inverse-frequency `pos_weight` for the plain BCE path; (3) post-split **EDA
augmentation** of turns carrying a rare category, applied to the *current turn
only* (never the context), label-safety-guarded. With only 11 categories over
10,600 turns, none of these are expected to matter as much as they did at
strategy granularity -- every category should be comfortably supported.

## Context width is measured, not assumed

The 5-turn context is inherited from Petrova et al.'s *prompt* design. For a
fine-tuned classifier it is a liability as much as an asset, because the
context contains previous persuader turns, which themselves used persuasive
language -- exactly what the classifier keys on. The notebook **trains
encoders at several context widths**, reports the ablation (S8e), and lets
the ensemble search weight or drop each width on validation, rather than
assuming either way.

## Decision rule

Predicting a label set is not just a matter of running the encoder: the
threshold rule matters as much as the model. This notebook fits several
candidate rules on validation -- a plain 0.5 cut, a single micro-F1-optimal
threshold, a cardinality-matched threshold, and coordinate ascent on micro-F1
that may switch a hopeless label off -- and picks between them on grouped,
complexity-penalised cross-fitted validation. The rule that ships in most
multi-label code (tune each label's own F1) is retained purely as a reported
baseline.

## Evaluation

micro / macro / weighted / samples **F1**, **Hamming loss**, **Jaccard**
(samples), **subset accuracy**, per-category F1, and a
**per-annotator-segment** breakdown (the four segments differ sharply in label
density, so `annotator` is a rater variable), all on the held-out test split,
against a **prior** baseline and a **TF-IDF word+char -> one-vs-rest logistic
regression** baseline.

## Phase 4

Sections 13-17 rebuild the donation-probability analysis, entirely at
category granularity: per-dialogue category co-occurrence features -> chi2
association tests -> logistic Models 1-4 (McFadden pseudo-R2 vs the paper's
single-label ~0.015-0.08) -> per-category multivariate effect model (15b) ->
conditional/deferred modifier model. The strategy-level Guilt-Induction
re-test (Section 16 in earlier revisions) has no faithful category-level
equivalent and is not run here -- see Section 16's markdown for why.


## 0b. Kaggle setup & dependencies

**Add Input -> Datasets** and attach `DS_Persuasion_Multi_strategy_Capstone`
(or any dataset with the same files at any depth):

```
multilabel_merged.jsonl        <- REQUIRED   (turn -> strategy-set ground truth)
persuader_turns.csv            <- strongly recommended: context + donation outcome (Phase 4)
taxonomy_multilabel.json       <- optional (bundled fallback is verified identical)
Manual_Label_normalized.csv    <- optional: persuadee sentiment/engagement covariates
multilabel_merged.csv          <- not read (flat/CSV mirror of the .jsonl; kept for reference)
multilabel_merged_wide.csv     <- not read (one-hot mirror; kept for reference)
guideline_multi_strategy.md    <- not read (annotation guideline; human reference only)
turnset_fingerprint.json       <- not read (provenance checksum; human reference only)
```

Discovery is fuzzy, recursive and case-insensitive over `search_roots`
(`/kaggle/input` first), so the exact dataset slug Kaggle mounts it under and the
folder layout inside it do not matter -- verified against this dataset's actual
(flat) layout. `CONFIG["dataset_hint"]` names the attached dataset purely so a
missing-file error message can tell you what to check; it is not required for
discovery to work. The install cell never raises: anything it cannot fetch
degrades to a documented fallback.


In [ ]:
# =============================================================
# 0b. INSTALL / UPGRADE DEPENDENCIES  (never fatal)
# =============================================================
import subprocess, sys, importlib.util

def _pip(args, label):
    """pip install that reports instead of raising -- an offline Kaggle run must
    still reach the training loop, just with the documented fallbacks."""
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       check=True, timeout=900)
        print(f"  ok       {label}")
        return True
    except Exception as exc:
        print(f"  SKIPPED  {label}  ({type(exc).__name__}) -- degrading gracefully")
        return False

# Only install what is actually missing. Kaggle's image already ships a
# GPU-matched torch/transformers pair; force-upgrading transformers is how this
# notebook used to break, so we leave it alone unless it is genuinely absent.
_NEEDED = [
    ("iterstrat",       "iterative-stratification", "stratified split -> grouped-random fallback"),
    ("sentencepiece",   "sentencepiece",            "DeBERTa-v3 tokenizer -> encoder auto-skipped"),
    ("google.protobuf", "protobuf",                 "DeBERTa-v3 slow-tokenizer conversion"),
    ("statsmodels",     "statsmodels",              "Phase 4 odds-ratios / FDR -> section skipped"),
    ("xgboost",         "xgboost",                  "S8b XGBoost member + S8c stacker -> both skipped"),
    ("nltk",            "nltk",                     "VADER sentiment -> tiny lexicon fallback"),
]
for mod, pkg, why in _NEEDED:
    if importlib.util.find_spec(mod.split(".")[0]) is None:
        print(f"installing {pkg}  ({why})")
        _pip([pkg], pkg)
    else:
        print(f"  present  {pkg}")

import transformers, sklearn
print(f"\ntransformers {transformers.__version__} | scikit-learn {sklearn.__version__}")
print("dependency check done.")


## 1. Imports & global config

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set **before** torch
initialises its CUDA allocator -- the same fragmentation-OOM mitigation the
`v8 - hierarchical+aug` notebook adopted after DeBERTa-v3 OOM'd on a T4.

Everything you would want to change lives in the single `CONFIG` dict below.


In [ ]:
# =============================================================
# 1. IMPORTS & GLOBAL CONFIG
# =============================================================
import os
# Set BEFORE torch initialises its CUDA allocator (the v8 notebook's OOM fix).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence fork warnings
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import re, json, glob, gc, time, math, warnings, random, zipfile, zlib, inspect
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             hamming_loss, jaccard_score, log_loss)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU  = torch.cuda.device_count()
print(f"Using device: {DEVICE}  |  visible GPUs: {N_GPU}")
print(f"torch {torch.__version__} (built for CUDA {torch.version.cuda})")
if DEVICE == "cuda":
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")

# ---- CONFIG: the only block you should need to touch -----------------
CONFIG = {
    # Fuzzy recursive search-roots -- same convention as the Intention/Classifier
    # notebooks, so this works unmodified once the files are a Kaggle input.
    # /kaggle/input is walked recursively, so the exact mounted dataset slug
    # (e.g. /kaggle/input/ds-persuasion-multi-strategy-capstone/...) never
    # needs to be spelled out here -- verified against a flat mount of
    # DS_Persuasion_Multi_strategy_Capstone's actual file layout.
    "search_roots": ["/kaggle/input", "/kaggle/working", "/workspace", "/data", "."],
    "dataset_hint": "DS_Persuasion_Multi_strategy_Capstone",  # cosmetic: named in error messages only
    "out_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs",

    # File discovery tokens (case-insensitive substring match on the filename).
    # Only `labels` is required; everything else degrades gracefully.
    "files": {
        "labels":   {"contains": ["multilabel_merged"], "suffix": ".jsonl"},
        "turns":    {"contains": ["persuader_turns"],    "suffix": ".csv"},
        "manual":   {"contains": ["manual_label"],       "suffix": ".csv"},
        "taxonomy": {"contains": ["taxonomy_multilabel"],"suffix": ".json"},
    },

    # Column mapping for persuader_turns.csv. Only this dict needs editing if the
    # headers change -- nothing downstream references raw column names.
    "column_map": {
        "turn_id": "turn_id", "dialogue_id": "dialogue_id",
        "context": "context", "text": "text",
        "donated": "binary_label_norm", "modifier": "modifier_norm",
    },
    "modifier_classes": ["none", "deferred", "conditional"],

    # ---- encoders benchmarked head-to-head, then ensembled --------------
    # Per-entry keys override the global defaults. DeBERTa-v3 needs a lower LR and
    # a smaller batch (the same finding as the v6/v8 OOM + NaN post-mortems).
    # Each entry may override context_turns / max_length / lr / batch_size / seed.
    #
    # The ensemble's diversity axis is now CONTEXT WIDTH, not just architecture.
    # Reason (section 8e measures it on your own data): the <=5-turn context
    # contains *previous persuader turns*, which themselves used strategies, and
    # their language bleeds into the current turn's prediction. On a controlled
    # ablation over this corpus, removing context improved 29 of 33 evaluable
    # strategies -- gratitude_and_appreciation went 0.54 -> 0.86 F1, and several
    # rare strategies went from never-predicted to F1 0.3-0.6. Rather than assume
    # either way for a transformer (whose attention can suppress the context a
    # bag-of-words model cannot), the notebook TRAINS several widths and lets
    # grouped cross-fitted validation pick, with the greedy ensemble free to drop
    # whichever loses. Short-context runs are also much faster.
    #
    # The list below is ORDERED BY EXPECTED VALUE PER MINUTE, because
    # `time_budget_min_total` stops the loop when the session budget is spent --
    # so anything that gets dropped is dropped from the bottom. Set
    # "enabled": False on an entry to skip it without deleting the recipe.
    #
    # Diversity axes, all three of which the ensemble exploits:
    #   architecture  roberta-base / TOD-BERT / ELECTRA / DeBERTa-v3 / roberta-large
    #   context width 0 / 1 / 5 turns  (section 8e ablates it)
    #   seed          roberta-ctx0 is trained twice, at seed 42 and 1337
    # Seed-only diversity is the cheapest decorrelation there is: same recipe,
    # different initialisation of the heads and a different data order, and the
    # errors are only ~0.85 correlated. The stacker in 8c gets a real vote out of it.
    #
    # DeBERTa-v3 is back ON, but re-tuned. The previous run's post-mortem said it
    # UNDERFITTED (train loss 0.037 vs RoBERTa's 0.011, val still climbing) -- a
    # 12-layer LLRD chain at decay 0.90 leaves its bottom layer on 2e-5*0.9^11 =
    # 7e-6, which for DeBERTa-v3's 128k-token embedding is close to frozen. It now
    # gets lr 2.5e-5 with llrd_decay 0.95 (per-encoder override) instead of a
    # lower LR, which is the fix for underfitting rather than a cause of it.
    # ---- encoders -------------------------------------------------------
    # Trimmed and re-ordered for a MACRO-F1 objective, using what the v4 run
    # measured:
    #   * roberta-large scored 0.6621 test micro alone -- more than v4's entire
    #     five-member ensemble minus a hair -- and held 40% of the winning
    #     blend. It goes FIRST so a budget overrun can never drop it.
    #   * The greedy blend dropped roberta-ctx5, todbert-ctx5, deberta-ctx0 and
    #     roberta-ctx0-s7 entirely, i.e. ~160 min of GPU bought nothing.
    #     deberta-ctx0 (0.6016, 33 min) and the two ctx5 runs (worst width,
    #     slowest) are gone.
    #   * More transformer capacity mostly improves labels that are ALREADY at
    #     F1 0.70-0.90, which is worth almost nothing to a macro average. The
    #     budget freed here is spent on the rare tail instead -- heavier
    #     augmentation, a positive-weighted loss, and the two zero-shot cue
    #     members in S8b -- which is where the 25 starved labels live.
    # Set "enabled": False on an entry to skip it without deleting the recipe.
    "encoders": [
        {"name": "robertaL-ctx0", "hf_id": "roberta-large", "context_turns": 0,
         "max_length": 128, "lr": 1e-5, "warmup_ratio": 0.10, "batch_size": 16,
         "eval_batch_size": 32, "llrd_decay": 0.95, "proto": True},
        # roberta-large again at ctx1. In v4 roberta-large was the best single
        # member by a clear margin (0.6621 test micro, vs 0.6532 for the best
        # base model) and ctx1 was the best width among the base encoders, so
        # this is the strongest recipe the run has evidence for. It is also the
        # single most expensive entry (~95 min) -- set "enabled": False here
        # first if the session needs to be shorter.
        {"name": "robertaL-ctx1", "hf_id": "roberta-large", "context_turns": 1,
         "max_length": 192, "lr": 1e-5, "warmup_ratio": 0.10, "batch_size": 16,
         "eval_batch_size": 32, "llrd_decay": 0.95, "proto": True},
        {"name": "electra-ctx0", "hf_id": "google/electra-base-discriminator",
         "context_turns": 0, "max_length": 128, "lr": 3e-5, "warmup_ratio": 0.10},
        {"name": "todbert-ctx0", "hf_id": "TODBERT/TOD-BERT-JNT-V1",
         "context_turns": 0, "max_length": 128, "proto": True},
        {"name": "roberta-ctx1", "hf_id": "roberta-base", "context_turns": 1,
         "max_length": 192, "proto": True},
        {"name": "roberta-ctx0", "hf_id": "roberta-base", "context_turns": 0, "max_length": 128},
        {"name": "roberta-ctx0-s7", "hf_id": "roberta-base", "context_turns": 0,
         "max_length": 128, "seed": 1337},
        # v4 measurements, kept so the reasons stay visible:
        # {"name": "deberta-ctx0",  ... 0.6016 test, 33 min, dropped by the blend}
        # {"name": "roberta-ctx5",  ... 0.6448 test, 36 min, dropped by the blend}
        # {"name": "todbert-ctx5",  ... 0.6158 test, 35 min, dropped by the blend}
    ],

    # ---- input construction --------------------------------------------
    "context_turns": 5,      # prior turns shown with each turn (Petrova et al. use up to 5)
    "max_length": 320,       # token budget; truncation_side="left" keeps the current turn
                             # (p95 of context+turn is ~260 subwords, so 320 rarely truncates)
    "use_speaker_roles": True,   # learned [Persuader]/[Persuadee] role embeddings

    # NOTE: an earlier version of this notebook prepended an "[A:<annotator>]"
    # rater-ID tag here as a feature (+0.031 micro-F1 measured on this corpus).
    # Removed: the deployed use case classifies a live, turn-by-turn
    # conversation that will never have a human-labeler ID, so conditioning on
    # one is a feature the production model could never actually receive.
    # See Section 8b's `meta_feats` for the second (numeric) leak of the same
    # variable, also removed.

    # ---- split ---------------------------------------------------------
    "train_frac": 0.70, "val_frac": 0.15, "test_frac": 0.15,

    # ---- training (manual loop, shared across encoders) ---------------
    "batch_size": 32,            # TOTAL batch; DataParallel splits it across both T4s
    "eval_batch_size": 64,       # conservative: an eval OOM would cost a full re-train
    "num_epochs": 16,      # every encoder was still improving at epoch 10
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,
    "adam_eps": 1e-6,            # raised from 1e-8 -- DeBERTa-v3 NaN-loss mitigation
    "dropout": 0.15,
    # 5, not 4: the macro selection metric is noisier than the micro one (it is
    # an average over labels with 1-3 validation positives), so a short patience
    # stops on noise rather than on convergence.
    "early_stopping_patience": 5,
    "checkpoint_dir_name": "checkpoints",
    # 0 by default on purpose. With >0, several worker-backed DataLoaders are
    # alive at once (train/val/test/full), forked workers inherit another
    # loader's live iterator, and every worker exit prints
    # "AssertionError: can only test a child process" -- thousands of lines of
    # benign noise that buried the real output last run. Tokenisation is ~3% of
    # a step here, so the workers were never buying much anyway. If you do raise
    # this, only the TRAIN loader uses workers (see the dl() helper below).
    "num_workers": 0,

    # fp16 autocast + GradScaler. Turing (T4) has no bf16, so fp16 it is; the
    # scaler skips any step whose gradients overflow, which doubles as the
    # cheapest insurance against DeBERTa-v3's fp16 NaNs.
    "use_amp": True,
    # Turn on only if you hit OOM -- ~30% slower, ~40% less activation memory.
    "grad_checkpointing": False,
    # Wall-clock guard so one slow encoder cannot eat the whole Kaggle session.
    "time_budget_min_per_encoder": 150,
    # Guard for the WHOLE encoder loop. With nine encoders configured, a single
    # over-running member could otherwise leave no session left for the
    # ensemble, the stacker and Phase 4 -- all of which are cheap but must run.
    # Section 8 stops starting new encoders once this is spent and says which
    # ones it skipped; the `encoders` list is ordered so that what gets dropped
    # is the least valuable member.
    "time_budget_min_total": 420,

    # ---- loss ----------------------------------------------------------
    # "asl" -> Asymmetric Loss (default; multi-label analogue of v8's CB-focal)
    # "bce" -> plain BCEWithLogits + inverse-frequency pos_weight (previous default)
    "loss_type": "asl",
    "asl_gamma_neg": 3.0, "asl_gamma_pos": 0.0, "asl_clip": 0.02,
    "use_pos_weight": True, "pos_weight_cap": 30.0,
    # Apply the per-label positive weight to the ASYMMETRIC loss too (v4 built
    # pos_weight and then only used it for loss_type == "bce", so under the
    # default "asl" the rare tail got no explicit up-weighting at all). Plain
    # ASL is right for micro-F1 and wrong for macro, where a 2-example strategy
    # carries 1/41 of the score by itself. Cap 30 keeps the gradient finite:
    # uncapped, door_in_the_face would ask for a weight of ~3,800.
    "asl_use_pos_weight": True,
    "strategy_loss_weight": 1.0,

    # ---- head / pooling ------------------------------------------------
    # "mean" (v4 style) | "meanmax" | "attn" (learned attention pooling)
    "pooling": "attn",
    "multi_sample_dropout": 4,    # 1 disables

    # Layer-wise LR decay: layer i gets lr * decay^(depth-i); heads get lr * head_lr_mult.
    "use_llrd": True, "llrd_decay": 0.90, "head_lr_mult": 8.0,   # 0.85 starved the lower layers

    # ---- EDA augmentation (train split only, label-safety-guarded) ----
    # OFF for this category-target notebook. `aug_tiers` below is calibrated
    # for STRATEGY-level scarcity (some strategies had 2-20 train examples);
    # at category granularity the rarest label still has ~120 real train
    # examples, plenty for a fine-tuned encoder on its own. Measured on the
    # first run with this left on: the tiers still fired for the 3 least
    # common categories (120/122/168 real examples), inflating train from
    # 7,562 -> 64,082 rows (88% synthetic) -- ~5x slower per epoch, which
    # burned the whole session time budget after only 4 of 7 encoders, and
    # those exact 3 categories came out as the 3 weakest-scoring in that run
    # (plausible overfitting to repeated near-duplicate synthetic text rather
    # than a real difficulty). Left the tiers/knobs below defined, just inert,
    # so this is a one-flag decision, not a deleted feature.
    "use_augmentation": False,
    "aug_alpha": 0.12,            # fraction of eligible words perturbed
    "aug_tiers": [[30, 8], [80, 5], [150, 3], [400, 1]],
    # legacy two-tier knobs, still read if `aug_tiers` is removed
    "aug_rare_threshold": 120, "aug_copies_rare": 2,
    "aug_midrare_threshold": 300, "aug_copies_midrare": 1,
    "aug_current_turn_only": True,  # never perturb the context, only the labelled turn

    # ---- WHAT IS BEING OPTIMISED ---------------------------------------
    # "macro" -> every one of the 41 strategies counts equally, so the rare tail
    #            is the whole game. This retargets EVERY selection step in the
    #            notebook: the per-epoch checkpoint metric, the decision-rule
    #            search, the ensemble hillclimb, the finalist comparison and the
    #            headline-model choice in S10.
    # "micro"  -> reproduces the v4 behaviour exactly (per-prediction accuracy,
    #            dominated by the 16 frequent strategies).
    # Both metrics are always REPORTED; this only decides what is optimised.
    #
    # Read this before quoting a macro number: `authority_endorsement` and
    # `door_in_the_face` have ZERO test positives, so all-41 macro-F1 cannot
    # exceed 39/41 = 0.951 no matter what the model does, and 16 of the 41 have
    # 10 or fewer test positives. `macro_f1_eval` below is macro-F1 over the
    # labels with enough test support to be scoreable, and the label count is
    # printed with it every time so it cannot be mistaken for the full 41.
    "target_metric": "macro",
    "macro_min_test_support": 10,          # labels below this are unscoreable
    "report_support_tiers": [1, 10, 50, 250],
    # Section 3c: measure how reproducible the annotation is, so every F1 below
    # can be read against the ceiling rather than against 1.0. ~1-2 min.
    "estimate_noise_ceiling": True,
    # Report BOTH the micro-optimal and the macro-optimal operating point for
    # every model. They differ only in the decision rule, so this costs one
    # extra threshold search and means neither metric is ever quoted from the
    # other's cut.
    "report_both_operating_points": True,

    # ---- decision rule -------------------------------------------------
    # The previous rule maximised each label's OWN F1. That is locally optimal
    # and globally wrong: a label with 3 val positives maximises its own F1 at
    # a very low threshold (recall 1.0, precision 0.01) and the ~270 false
    # positives it then emits land in the SHARED micro-F1 denominator. Measured
    # on this corpus it cost -0.038 micro-F1 and inflated predicted cardinality
    # to 3.3 against a true 1.74. Section 7 now fits several candidate rules and
    # picks between them on grouped, complexity-penalised cross-fitted val.
    "threshold_floor": 0.10, "threshold_ceil": 0.90,
    "decision_rule_folds": 5,      # grouped by dialogue, like the real val->test shift
    "decision_rule_tol": 0.004,
    # Multipliers tried by the `prevalence` rule: it cuts each label so the model
    # predicts about alpha * p_j * n positives, where p_j is the label's TRAIN
    # frequency. One fitted parameter covering all 41 thresholds -- the only rule
    # here that gives a 2-positive label a sane cut without fitting to those 2.
    "prevalence_alphas": [0.5, 0.75, 1.0, 1.5, 2.0, 3.0],
    # "At least one label per turn": rows where nothing clears its threshold get
    # their arg-max strategy anyway. Only a few percent of turns are genuinely
    # empty-labelled, so an all-zero prediction row is usually a thresholding
    # artefact that costs a recall point and buys no precision. Searched on
    # cross-fitted val as one more decode switch, never assumed.
    "force_top1_options": [False, True],

    # Average the probability predictions of the best `snapshot_k` epochs of a
    # run instead of using the single best epoch. Free variance reduction -- the
    # probabilities are already computed each epoch for model selection.
    "snapshot_k": 3,

    # ---- ensemble / stacking ("the zoo", section 8c) --------------------
    # Section 8c no longer commits to ONE combination rule. It builds a zoo of
    # candidates, scores every one of them under the SAME grouped cross-fitted
    # val protocol used for single models, and keeps the winner. Nothing here is
    # assumed to help: a candidate that loses on val is never selected, so the
    # only cost of an extra candidate is a few seconds of numpy.
    "use_ensemble": True,
    # "zoo"    -> the full candidate search described below (recommended)
    # "greedy" -> only greedy forward selection with replacement (the old default)
    # "mean"   -> only the equal-weight probability average
    "ensemble_mode": "zoo",

    # Blend SPACE. "prob" averages sigmoid outputs; "logit" averages log-odds,
    # i.e. a weighted geometric mean of the odds, which is the right average when
    # members are independent evidence and is usually a touch sharper. Both are
    # tried and val picks.
    "ensemble_spaces": ["prob", "logit"],
    # Per-member Platt calibration (2 params per member: scale + shift on the
    # logit) fitted on val before blending. Equalises members that are
    # differently over-confident -- roberta-large and TF-IDF-XGB are not on the
    # same sharpness scale, and an unweighted average of the two is dominated by
    # whichever is louder. Tried both ways.
    "ensemble_calibrate": [False, True],
    "ensemble_greedy_steps": 30,      # granularity of the greedy weights: 1/30
    # Bagged greedy selection (Caruana et al. 2004): re-run the greedy hillclimb
    # on `bag_rounds` random dialogue subsamples of val, each seeing a random
    # subset of the members, then average the weight vectors. Plain greedy on one
    # val split is a high-variance selector that likes whichever member happens
    # to fit that split; bagging is the standard cure.
    # 12 rather than 24: under a macro objective each hillclimb evaluation
    # re-fits 41 thresholds instead of 1, so a bag round costs ~10x more.
    "ensemble_bag_rounds": 12,
    "ensemble_bag_frac": 0.65,        # fraction of val DIALOGUES per bag
    "ensemble_bag_member_frac": 0.75, # fraction of members visible per bag
    # Generalised power mean  (sum_i w_i p_i^q)^(1/q).  q=1 arithmetic,
    # q->0 geometric, q=-1 harmonic. One extra scalar; q<1 is more conservative
    # about a single loud member, which is what a false-positive-sensitive
    # micro-F1 usually wants.
    "ensemble_power_grid": [-1.0, -0.5, 0.0, 0.5, 1.0, 2.0],
    # How many of the zoo's candidates get the expensive full decode search
    # (rules x gate x smoothing x force-top1) before the final pick. The rest
    # are ranked on the cheap global-threshold proxy.
    "ensemble_finalists": 5,
    # Coarser threshold grid INSIDE the hillclimb only. Under macro each
    # candidate evaluation re-fits 41 thresholds instead of 1, so the inner grid
    # is what keeps the bagged search to a couple of minutes; the final decode
    # still uses the full 0.01 grid.
    "ensemble_greedy_grid_step": 0.05,
    # Fit a SEPARATE weight vector per label-support tier. The best member for
    # `rapport_building` (2,034 train turns) and for `deadline_pressure` (2) is
    # not the same model -- a zero-shot cue member can only help the starved
    # tail and would be voted down by any weighting fitted on all 41 at once.
    # Only built when target_metric == "macro"; guarded by cross-fitted val like
    # every other candidate.
    "ensemble_tier_weights": True,
    "ensemble_tier_edges": [60, 250],   # train-support boundaries between tiers
    # A finalist must beat a SIMPLER candidate by this much on cross-fitted val
    # to be preferred (order of simplicity: single model < equal mean < weighted
    # blend < power mean < linear stack < XGB stack < stack+blend mixture).
    "ensemble_select_tol": 0.002,

    # ---- level-2 stacking (section 8c) ---------------------------------
    # A stacker sees, for every (turn, strategy) cell, what each member said
    # about that cell -- plus context a blend cannot use: the label's own
    # frequency, the row's total predicted mass, the label's rank within the row,
    # and what the REST OF THE DIALOGUE says about that same strategy. It is
    # trained on the val split only (members never saw val), with grouped
    # out-of-fold predictions so its val score is honest and its thresholds are
    # fitted on out-of-fold probabilities.
    "stack_lr": True,             # logistic stacker (cheap, low variance)
    "stack_xgb": True,            # XGBoost stacker (the flexible one)
    "stack_folds": 5,             # grouped by dialogue
    "stack_xgb_params": {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.05,
                         "subsample": 0.85, "colsample_bytree": 0.8,
                         "min_child_weight": 6.0, "reg_lambda": 2.0},
    # Mixture of the stacker with the best plain blend, mix weight chosen on val.
    # Almost always at least as good as either alone.
    "stack_blend_grid": [0.0, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0],
    # Minimum val rows / dialogues before a stacker is even attempted -- below
    # this it is pure variance (and SMOKE_TEST would hit empty folds).
    "stack_min_val_rows": 400, "stack_min_val_dialogues": 40,
    # Weight the level-2 loss to match a macro objective: in long format every
    # label contributes the same number of ROWS, but positives of a frequent
    # label outnumber those of a rare one 1000:1, so an unweighted logloss
    # learns the head and ignores the tail. Positive rows get weight
    # 1/prevalence (capped), which equalises each label's positive mass.
    "stack_macro_weight": True, "stack_pos_weight_cap": 60.0,

    # ---- shallow (non-transformer) ensemble members, section 8b --------
    # TF-IDF + SVD -> XGBoost, and TF-IDF -> binary-relevance logistic
    # regression. On their own these are ~0.40-0.46 micro-F1 against the
    # encoders' ~0.65, so they are NOT here to win: they are here because a
    # gradient-boosted bag-of-ngrams gets a different *kind* of turn right
    # (fixed lexical tells: "$", "%", "deadline", "tax deductible") and its
    # errors are far less correlated with a transformer's than another
    # transformer's are. That decorrelation is what an ensemble is paid for --
    # and the stacker can consult it per label instead of weighting it globally.
    # Few-shot prototype members (section 8b). "proto": True on an encoder entry
    # makes it also emit a centroid classifier over its own embeddings.
    "use_proto_members": True,
    "use_shallow_members": True,
    "shallow_svd_dim": 320,       # TF-IDF -> TruncatedSVD dims fed to XGBoost
    "shallow_xgb_params": {"n_estimators": 450, "max_depth": 6, "learning_rate": 0.06,
                           "subsample": 0.85, "colsample_bytree": 0.7,
                           "min_child_weight": 3.0, "reg_lambda": 1.5},
    "shallow_xgb_min_pos": 10,    # a label with fewer train positives keeps its prior
    "shallow_use_gpu": True,      # XGBoost device="cuda" when a GPU is visible

    # ---- zero-shot taxonomy-cue members (section 8b) -------------------
    # Two members that use NO training labels: each strategy gets a document
    # made of its taxonomy DEFINITION plus its example CUE sentences, and every
    # turn is scored by similarity to those 41 documents (lexical, and semantic
    # via a small sentence encoder). The raw similarity is calibrated into a
    # probability by a 2-feature logistic fit on the TRAIN split.
    #
    # This is the one lever aimed directly at the labels that make all-41
    # macro-F1 low. Thirteen strategies have under 30 training turns; every
    # fine-tuned encoder in v4 scored EXACTLY 0.000 on all of them, and they
    # carry 13/41 of the macro score between them. A similarity model needs no
    # training examples to notice that "the campaign closes tonight" is
    # deadline_pressure. Requires the real taxonomy json (the bundled fallback
    # has strategy names only, no definitions or cues) -- the members skip
    # themselves with a printed note otherwise.
    # MEASURED on this corpus before shipping (offline, real split, per-label
    # thresholds fitted on val): the LEXICAL cue member reaches only ~0.08 macro
    # over the 25 starved labels, because the taxonomy cues are paraphrases
    # ("the campaign closes tonight") and bag-of-ngrams overlap misses them. It
    # is kept because it costs 30 seconds and the zoo drops what does not help.
    # The SEMANTIC variant below is the one with real upside, and the few-shot
    # prototype members (which reach ~0.08 lexically and should do considerably
    # better in a fine-tuned encoder's space) are the stronger bet of the two
    # ideas. Do not quote the cue member as a result on its own.
    "use_cue_member": True,
    # Small, fast, built for cosine similarity; ~80 MB, about a minute for the
    # whole corpus. Set to "" to use the lexical cue member only.
    "cue_embed_model": "sentence-transformers/all-MiniLM-L6-v2",

    # ---- baselines -----------------------------------------------------
    # The old ClassifierChain(XGBoost) baseline cost 61 min for micro-F1 0.29
    # last run. It is superseded by the `tfidf-xgb` member in 8b -- the same
    # idea done properly (one booster per label on dense SVD features, ~3 min),
    # reported in the same results table. Left here only for reproducing the
    # published chain number.
    "run_xgb_chain": False,

    # ---- Phase 4 -------------------------------------------------------
    "phase4_label_source": "gold",   # "gold" (human-corrected) or "pred" (best model)
    "cooc_top_k": 12,
    "fdr_alpha": 0.05,

    # ---- artifacts -----------------------------------------------------
    "zip_checkpoints": False,     # .pt files are ~0.5 GB each; off by default
    # Nine members at 0.5-1.4 GB each will fill /kaggle/working, and nothing
    # downstream needs the weights (every member's val/test/full probabilities
    # are already in `results`). Set True if you hit the 20 GB output cap.
    "delete_ckpt_after_use": False,

    # ---- dev -----------------------------------------------------------
    # Two independent dev modes, both non-production and neither meaningful
    # as a result:
    #   FAST_DEV_RUN -- fastest possible check: 1 encoder, no XGBoost, no
    #                   ensemble, no augmentation. "Does the loop run at all."
    #   SMOKE_TEST   -- slower but exercises EVERY section (all configured
    #                   encoders, the ensemble, the XGBoost chain,
    #                   augmentation, the hierarchical gate, Phase 4) on a
    #                   tiny subsample. Run this once after any edit, before
    #                   trusting a full Kaggle submission -- it is the only
    #                   mode that actually touches every code path this
    #                   notebook has. SMOKE_TEST wins if both are True.
    "FAST_DEV_RUN": False,        # True -> ~800 turns, 1 epoch, 1 encoder, no XGB/ensemble/aug
    "SMOKE_TEST": False,          # True -> tiny subsample, but EVERY section still runs
    "smoke_n_train": 160, "smoke_n_val": 60, "smoke_n_test": 60,
    "fastdev_n_train": 800, "fastdev_n_val": 300, "fastdev_n_test": 300,
}

if CONFIG["SMOKE_TEST"]:
    # Three of the nine encoders, chosen to cover both architectures and both
    # context widths -- enough to exercise every code path (the ensemble needs
    # >= 2 members, the ablation needs >= 2 widths) without downloading
    # roberta-large for a wiring check.
    _smoke_encoders = [e for e in CONFIG["encoders"]
                       if e["name"] in ("roberta-ctx0", "todbert-ctx0", "roberta-ctx1")]
    CONFIG.update(FAST_DEV_RUN=True, num_epochs=1, batch_size=4, eval_batch_size=8,
                  max_length=128, num_workers=0, time_budget_min_per_encoder=10,
                  time_budget_min_total=60,
                  encoders=(_smoke_encoders or CONFIG["encoders"][:3]),
                  run_xgb_chain=True, use_ensemble=True, use_augmentation=True,
                  aug_copies_rare=2, aug_copies_midrare=1,
                  # every zoo path still runs, just tiny
                  use_shallow_members=True, shallow_svd_dim=32,
                  shallow_xgb_params={"n_estimators": 40, "max_depth": 3,
                                      "learning_rate": 0.2},
                  shallow_xgb_min_pos=2,
                  ensemble_greedy_steps=6, ensemble_bag_rounds=3,
                  ensemble_power_grid=[0.0, 1.0], ensemble_finalists=2,
                  stack_folds=2, stack_min_val_rows=20, stack_min_val_dialogues=2,
                  stack_xgb_params={"n_estimators": 40, "max_depth": 3,
                                    "learning_rate": 0.2},
                  stack_blend_grid=[0.0, 0.5, 1.0],
                  # the macro paths, at toy scale
                  aug_tiers=[[30, 2], [150, 1]],
                  prevalence_alphas=[1.0, 2.0],
                  ensemble_greedy_grid_step=0.1,
                  macro_min_test_support=2, report_support_tiers=[1, 2],
                  ensemble_tier_edges=[3, 12])
    print(">>> SMOKE_TEST: every section will run (%d encoders, the shallow XGBoost "
          "members,\n    the ensemble zoo, both stackers, the XGBoost chain, "
          "augmentation, the\n    hierarchical gate, Phase 4) on a tiny subsample.\n"
          "    A few minutes; results are NOT meaningful -- this is a pass/fail "
          "wiring check." % len(CONFIG["encoders"]))
elif CONFIG["FAST_DEV_RUN"]:
    CONFIG.update(num_epochs=1, batch_size=16, encoders=CONFIG["encoders"][:1],
                  run_xgb_chain=False, use_ensemble=False, num_workers=0,
                  use_augmentation=False, use_shallow_members=False)
    print(">>> FAST_DEV_RUN: fastest wiring check only (1 encoder, no XGBoost / ensemble / "
          "augmentation)\n    -- results are NOT meaningful. For full section coverage use "
          "SMOKE_TEST instead.")

os.makedirs(CONFIG["out_dir"], exist_ok=True)
CKPT_DIR = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"])
os.makedirs(CKPT_DIR, exist_ok=True)

USE_AMP = bool(CONFIG["use_amp"]) and DEVICE == "cuda"
print("\nAMP fp16:", USE_AMP, "| DataParallel:", N_GPU > 1)
print(json.dumps({k: v for k, v in CONFIG.items() if k not in ("encoders", "files")}, indent=2))
print("Encoders:", [m["hf_id"] for m in CONFIG["encoders"]])
json.dump({k: v for k, v in CONFIG.items() if k != "files"},
          open(os.path.join(CONFIG["out_dir"], "config.json"), "w"), indent=2)


## 2. Load the labeled dataset

`multilabel_merged.jsonl` (turn -> strategy set, 4 merged annotator segments) is
the only required file. If `persuader_turns.csv` is present it is joined on
`turn_id` for the <=5-turn `context` and the dialogue-level donation outcome.

The model input for a turn is its context followed by the current persuader
utterance; `tokenizer.truncation_side = "left"` guarantees the current turn is
never truncated away (the `v4 - data_aug+left_trunc` fix).


In [ ]:
# =============================================================
# 2. DATA DISCOVERY & LOADING
# =============================================================
def find_files_ci(roots, must_contain_all, suffix):
    """Recursive, case-insensitive filename search -- same convention as the
    Intention/Classifier notebooks, so the Kaggle dataset layout does not matter."""
    must_contain_all = [t.lower() for t in must_contain_all]; suffix = suffix.lower()
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dn, filenames in os.walk(root):
            for fn in filenames:
                low = fn.lower()
                if low.endswith(suffix) and all(tok in low for tok in must_contain_all):
                    found.append(os.path.join(dirpath, fn))
    out, seen = [], set()
    for f in sorted(found, key=lambda p: (len(p), p)):
        if f not in seen:
            seen.add(f); out.append(f)
    return out

def locate(key, required=False):
    spec = CONFIG["files"][key]
    hits = find_files_ci(CONFIG["search_roots"], spec["contains"], spec["suffix"])
    print(f"  {key:9s}: {hits[0] if hits else '(not found)'}")
    if required and not hits:
        raise FileNotFoundError(
            f"No file matching {spec} under {CONFIG['search_roots']}. "
            f"Add Input -> Datasets -> attach '{CONFIG['dataset_hint']}' "
            "(or any dataset containing multilabel_merged.jsonl).")
    return hits[0] if hits else None

print("input discovery:")
PATH = {k: locate(k, required=(k == "labels")) for k in CONFIG["files"]}

# ---- taxonomy: 41 strategies (canonical order) + strategy->category ----
# The bundled fallback has been verified identical to taxonomy_multilabel.json,
# so the notebook is fully functional with only multilabel_merged.jsonl attached.
FALLBACK_TAX = {
 "Rational Appeal": ["logical_appeal","evidence_and_statistics","cost_benefit_framing","feasibility_and_ease"],
 "Emotional Appeal": ["emotion_appeal","guilt_induction","empathy_and_perspective_taking","hope_and_positive_impact"],
 "Credibility Appeal": ["organizational_credibility","transparency_and_accountability","source_citation","personal_credibility"],
 "Social Influence": ["social_proof","self_modeling","in_group_appeal","authority_endorsement"],
 "Reciprocity and Exchange": ["reciprocity","donor_benefit","gratitude_and_appreciation"],
 "Commitment and Consistency": ["foot_in_the_door","door_in_the_face","value_consistency_appeal","incremental_ask"],
 "Framing and Presentation": ["anchoring","minimization_framing","loss_versus_gain_framing","comparison_framing"],
 "Urgency and Scarcity": ["urgency_appeal","scarcity_appeal","deadline_pressure"],
 "Threat and Pressure": ["negative_consequence_warning","persistent_repetition","obligation_pressure"],
 "Information Provision": ["donation_procedure_information","organization_information","impact_information","task_clarification"],
 "Relational and Interactive": ["personal_story","personal_related_inquiry","source_related_inquiry","rapport_building"],
}
TAX, TAX_RAW = None, None
if PATH["taxonomy"]:
    try:
        _t = json.load(open(PATH["taxonomy"], encoding="utf-8"))["categories"]
        TAX = {c: list(v["strategies"].keys()) for c, v in _t.items()}
        TAX_RAW = _t                     # keeps the per-strategy definition + cues
    except Exception as exc:
        print("  taxonomy json unreadable (%s) - using bundled fallback" % exc)
if TAX is None:
    TAX = FALLBACK_TAX
    print("  taxonomy: bundled fallback (verified identical to taxonomy_multilabel.json)")

CATS = list(TAX)
S2C  = {s: c for c in CATS for s in TAX[c]}   # fine strategy -> its parent category
# ---- PIVOT: this notebook's prediction target is the 11 CATEGORIES, not the
# 41 fine-grained strategies. `STRATS`/`S_IDX` are the generic names every
# downstream piece of this pipeline (loss, decode-rule search, ensemble zoo,
# per-label reporting) was already written against -- reassigning them here
# to hold the 11 category names, rather than renaming everything downstream,
# is what makes that machinery work on categories with no other code changes.
# `S2C` above still maps FINE strategy names (as they appear in the raw
# labels) to their category, and stays that way for Phase 4's own use.
STRATS = CATS
S_IDX  = {c: i for i, c in enumerate(CATS)}
C_IDX  = S_IDX
assert len(STRATS) == 11, len(STRATS)

# Per-category "document" = its own taxonomy definition + its child
# strategies' definitions. Section 8b turns these into two zero-shot members;
# they are the only signal in this notebook that does not come from the
# 10,600 annotated turns.
CUE_DOC = {}
if TAX_RAW:
    for _c, _cv in TAX_RAW.items():
        _cdef = str(_cv.get("definition", ""))
        _parts = [_c.replace("_", " "), _cdef]
        for _st, _sv in (_cv.get("strategies") or {}).items():
            _parts += [_st.replace("_", " "), str(_sv.get("definition", ""))]
            _parts += [str(x) for x in (_sv.get("cues") or [])]
        CUE_DOC[_c] = " ".join(pp for pp in _parts if pp)
    _ncue = sum(1 for c in STRATS if c in CUE_DOC)
    print("  taxonomy cue documents: %d/%d categories (mean %d words)"
          % (_ncue, len(STRATS),
             np.mean([len(CUE_DOC[c].split()) for c in CUE_DOC]) if CUE_DOC else 0))
else:
    print("  [note] no raw taxonomy json -> cue members in S8b will be skipped")

# ---- load labels ------------------------------------------------------
# Raw labels are kept as the FINE strategy name strings straight from the
# source file -- no discarding here (a category is never short of examples,
# so there is nothing to filter for support reasons) -- and mapped to
# categories via `strat_vec` below and via `S2C` wherever Phase 4 needs them.
mrg = pd.read_json(PATH["labels"], lines=True)
mrg["strategies"] = mrg["strategies"].apply(
    lambda v: list(v) if isinstance(v, (list, tuple, np.ndarray)) else [])
if "annotator" not in mrg.columns:
    mrg["annotator"] = "unknown"
_unknown = sorted({s for ss in mrg["strategies"] for s in ss} - set(S2C))
assert not _unknown, f"labels contain strategies outside the taxonomy: {_unknown}"

# ---- optional join: context + donation outcome -----------------------
cm = CONFIG["column_map"]
turns, HAVE_TURNS, HAVE_OUTCOME = None, False, False
if PATH["turns"]:
    turns_raw = pd.read_csv(PATH["turns"], dtype=str).fillna("")
    missing = [v for v in cm.values() if v not in turns_raw.columns]
    if missing:
        print(f"  [WARNING] persuader_turns.csv is missing {missing}; "
              f"columns present = {list(turns_raw.columns)}. Ignoring the file.")
    else:
        turns = turns_raw.rename(columns={v: k for k, v in cm.items()})[list(cm.keys())].copy()
        HAVE_TURNS = True
        HAVE_OUTCOME = bool(turns["donated"].str.strip().ne("").any())

df = mrg.copy()
if HAVE_TURNS:
    df = df.merge(turns[["turn_id", "context", "donated", "modifier"]], on="turn_id", how="left")
    _matched = float(df["context"].notna().mean())
    print(f"\n  joined persuader_turns.csv: {100*_matched:.1f}% of turns matched on turn_id")
    if _matched < 0.5:
        print("  [WARNING] poor turn_id overlap - check the two files are the same corpus")
    df["context"] = df["context"].fillna("")
else:
    df["context"] = ""; df["donated"] = ""; df["modifier"] = ""
    print("\n  [NOTE] persuader_turns.csv not found.")
    print("         -> turns are classified WITHOUT dialogue context")
    print("         -> Phase 4 (sections 13-17) will be skipped")
if HAVE_TURNS and not HAVE_OUTCOME:
    # The join key (turn_id) matched fine (see the % above) but the outcome
    # column itself is empty for every row -- almost always means the
    # attached persuader_turns.csv is a pre-normalization export (Phase 0.1
    # not yet run) rather than a code bug here. Show the raw values so that
    # distinction is obvious instead of a bare "skipped".
    _raw_col = cm["donated"]
    _nonblank = int(turns_raw[_raw_col].astype(str).str.strip().ne("").sum())
    print(f"  [NOTE] '{_raw_col}' column found but EMPTY for all {len(turns_raw)} rows "
          f"({_nonblank} non-blank) -> Phase 4 will be skipped.")
    print(f"         sample raw values: {turns_raw[_raw_col].astype(str).unique()[:5].tolist()}")
    print("         This looks like a data issue (an un-normalized export of "
          "persuader_turns.csv), not a notebook bug -- re-attach the version with "
          "binary_label_norm/modifier_norm populated if you want Phase 4.")
elif not HAVE_OUTCOME:
    print("  [NOTE] no donation outcome available -> Phase 4 will be skipped")

def trim_context(ctx, k):
    if not isinstance(ctx, str) or not ctx.strip():
        return ""
    return "\n".join([l for l in ctx.split("\n") if l.strip()][-k:])

df["text"] = df["text"].fillna("").astype(str)
# Context is kept as a LIST of lines so any width can be rebuilt on demand --
# section 8 trains encoders at several widths and lets validation choose.
df["ctx_lines"] = df["context"].apply(
    lambda c: [l for l in str(c).split("\n") if l.strip()] if isinstance(c, str) else [])

def build_model_text(cur_text, ctx_lines, k):
    """Last-k context lines + the utterance being labelled. The labelled turn
    always comes LAST so truncation_side='left' keeps it.
    No annotator-ID tag: removed (was `[A:<annotator>] + ...`). A live,
    turn-by-turn deployment never has a human-labeler ID for a new
    conversation, so conditioning on one is a feature the deployed model
    could never actually receive. The `annotator` column and the
    label-density-by-segment table below are kept as descriptive EDA of that
    known rater confound -- they just no longer feed the model."""
    if k > 0 and len(ctx_lines):
        ctx = "\n".join(ctx_lines[-k:])
        return ctx + "\n[Persuader] " + cur_text
    return "[Persuader] " + cur_text

# Always include the default width: the TF-IDF baselines and `model_text` use it,
# and FAST_DEV_RUN may trim the encoder list down to a single non-default width.
CTX_WIDTHS = sorted({CONFIG["context_turns"]} |
                    {e.get("context_turns", CONFIG["context_turns"])
                     for e in CONFIG["encoders"]})
def add_text_cols(frame, text_col="text"):
    for k in CTX_WIDTHS:
        frame[f"mt{k}"] = [build_model_text(t, c, k) for t, c in
                           zip(frame[text_col], frame["ctx_lines"])]
    return frame
df = add_text_cols(df)
df["model_text"] = df[f"mt{CONFIG['context_turns']}"]      # default view, for baselines

print("context widths to be built:", CTX_WIDTHS)

# ---- multi-hot label matrices ---------------------------------------
def strat_vec(ss):
    """A turn's category vector: category c is present iff any of the turn's
    (fine-grained, raw-labelled) strategies belongs to c, via the static S2C
    map. Named `strat_vec` (not `cat_vec`) purely so it slots into the same
    call site every other revision of this notebook has used."""
    v = np.zeros(len(STRATS), np.float32)
    for s in (ss or []):
        c = S2C.get(s)
        if c is not None:
            v[S_IDX[c]] = 1.0
    return v

Y_STRAT = np.vstack(df["strategies"].apply(strat_vec))
df["n_strat"] = Y_STRAT.sum(1).astype(int)

print(f"\nLoaded {len(df)} turns across {df.dialogue_id.nunique()} dialogues.")
print("mean categories/turn: %.3f  |  empty-set turns: %.1f%%  |  multi-label turns: %.1f%%"
      % (Y_STRAT.sum(1).mean(), 100*(Y_STRAT.sum(1) == 0).mean(), 100*(Y_STRAT.sum(1) > 1).mean()))
print("turns with a non-empty context: %.1f%%"
      % (100 * df["ctx_lines"].map(len).gt(0).mean()))
print("context lines available per turn: median %d, p95 %d"
      % (df["ctx_lines"].map(len).median(), df["ctx_lines"].map(len).quantile(0.95)))
print("\nlabel density by annotator segment (the known rater effect):")
print(df.groupby("annotator").agg(turns=("turn_id", "size"),
                                  mean_labels=("n_strat", "mean"),
                                  empty_pct=("n_strat", lambda s: 100*(s == 0).mean())).round(2))
df[["turn_id", "annotator", "text", "n_strat", "donated", "modifier"]].head(4)


## 2b. Label distribution & prior baseline


In [ ]:
# =============================================================
# 2b. STRATEGY / CATEGORY DISTRIBUTION & PRIOR BASELINE
# =============================================================
strat_freq = pd.Series(Y_STRAT.sum(0), index=STRATS).sort_values()
lab_hist   = pd.Series(Counter(df.n_strat)).sort_index()

print("labels-per-turn:"); print(lab_hist.to_string())
print(f"\nrarest categories : {strat_freq.head(6).astype(int).to_dict()}")
print(f"commonest         : {strat_freq.tail(6).astype(int).to_dict()}")

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
strat_freq.plot.barh(ax=ax[0], color="#4C72B0"); ax[0].set_title("category support (all turns)")
ax[0].tick_params(labelsize=8)
ax[1].bar(lab_hist.index.astype(str), lab_hist.values, color="#C44E52")
ax[1].set_title(f"labels per turn (mean {Y_STRAT.sum(1).mean():.2f})")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["out_dir"], "label_distribution.png"), dpi=140); plt.show()

# `macro_f1_eval` is macro-F1 over the labels that HAVE enough test support to be
# scoreable (see below). With only 11 categories over 10,600 turns, every one of
# them should clear a reasonable support floor -- unlike the 41-strategy version
# of this pipeline, where 2 strategies had zero test examples and capped the
# all-label macro at 0.951 regardless of model quality.
METRIC_KEYS = ["micro_f1", "macro_f1", "macro_f1_eval", "weighted_f1", "samples_f1",
               "hamming_loss", "jaccard_samples", "subset_accuracy", "card_pred"]

def label_f1_vector(y_true, y_pred):
    """Per-label F1, length L. The building block of every macro number here.

    Closed form rather than `f1_score(..., average=None)`: the decode search
    calls this tens of thousands of times under a macro objective, where
    sklearn's per-call validation and label bookkeeping dominate the runtime.
    Identical output (F1 = 0 when a label has no positives and no predictions,
    i.e. sklearn's zero_division=0)."""
    yb = np.asarray(y_true).astype(bool)
    pb = np.asarray(y_pred).astype(bool)
    tp = (pb & yb).sum(0)
    fp = (pb & ~yb).sum(0)
    fn = ((~pb) & yb).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).astype(np.float64)

def f1_metric(y_true, y_pred, metric="micro"):
    """micro / macro / samples F1 without the sklearn dispatch overhead."""
    if metric == "macro":
        return float(label_f1_vector(y_true, y_pred).mean())
    if metric == "micro":
        yb = np.asarray(y_true).astype(bool); pb = np.asarray(y_pred).astype(bool)
        tp = int((pb & yb).sum()); fp = int((pb & ~yb).sum()); fn = int(((~pb) & yb).sum())
        den = 2 * tp + fp + fn
        return 0.0 if den == 0 else 2 * tp / den
    return float(f1_score(y_true, y_pred, average=metric, zero_division=0))

def macro_f1_tiers(y_true, y_pred, tiers=None):
    """macro-F1 restricted to labels whose TEST support clears each threshold.

    Why this exists: macro-F1 weights a rare label exactly as heavily as a
    common one. With only 11 categories this is a much smaller concern than
    it was at strategy granularity, but the tiers are kept so any category
    that does end up under-supported on a given split is still visible
    rather than silently blended into one number."""
    tiers = tiers or CONFIG.get("report_support_tiers", [1, 10, 50, 250])
    f1 = label_f1_vector(y_true, y_pred)
    sup = np.asarray(y_true).sum(0)
    out = {}
    for t in tiers:
        m = sup >= t
        out[int(t)] = {"n_labels": int(m.sum()),
                       "macro_f1": float(f1[m].mean()) if m.any() else 0.0}
    return out

def multilabel_metrics(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    _k = int(CONFIG.get("macro_min_test_support", 10))
    _sup = y_true.sum(0)
    _f1v = label_f1_vector(y_true, y_pred)
    _ev = _sup >= _k
    m = dict(
        micro_f1    = f1_score(y_true, y_pred, average="micro",    zero_division=0),
        macro_f1    = f1_score(y_true, y_pred, average="macro",    zero_division=0),
        macro_f1_eval = float(_f1v[_ev].mean()) if _ev.any() else 0.0,
        n_eval_labels = int(_ev.sum()),
        weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0),
        samples_f1  = f1_score(y_true, y_pred, average="samples",  zero_division=0),
        precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0),
        recall_micro    = recall_score(y_true, y_pred, average="micro", zero_division=0),
        hamming_loss    = hamming_loss(y_true, y_pred),
        jaccard_samples = jaccard_score(y_true, y_pred, average="samples", zero_division=0),
        subset_accuracy = accuracy_score(y_true, y_pred),
        card_true = float(y_true.sum(1).mean()), card_pred = float(y_pred.sum(1).mean()),
    )
    return {f"{prefix}{k}": (int(v) if k == "n_eval_labels" else round(float(v), 4))
            for k, v in m.items()}

# reference-only prior baseline over the whole dataset (the honest,
# test-split-matched version is in section 10)
k_prior = max(1, round(Y_STRAT.sum(1).mean()))
prior_top = np.argsort(Y_STRAT.sum(0))[::-1][:k_prior]
P = np.zeros_like(Y_STRAT); P[:, prior_top] = 1
print(f"\nprior baseline (predict the {k_prior} commonest: "
      f"{[STRATS[i] for i in prior_top]}) -- whole dataset, reference only:")
print(json.dumps({k: v for k, v in multilabel_metrics(Y_STRAT, P).items()
                  if k in METRIC_KEYS}, indent=2))


## 3. Dialogue-level, strategy-stratified 70/15/15 split

Splitting at the **dialogue** level is mandatory: turns from one conversation
share context text, so a turn-level split would leak. We aggregate each
dialogue's strategy vector (strategy present if *any* turn uses it) and feed that
to `MultilabelStratifiedKFold` so rare strategies are spread across the three
splits; without `iterative-stratification` we fall back to a grouped random split
and report the resulting coverage either way.

`door_in_the_face` has **2 turns in the entire corpus**, so it can still land
entirely outside val/test no matter how the split is drawn -- the coverage table
below names exactly which strategies that happened to, and every downstream
metric handles zero-support labels with `zero_division=0`.


In [ ]:
# =============================================================
# 3. DIALOGUE-LEVEL STRATIFIED SPLIT
# =============================================================
dlg = (pd.DataFrame(Y_STRAT, columns=STRATS)
       .assign(dialogue_id=df.dialogue_id.values)
       .groupby("dialogue_id").max().clip(upper=1))
dlg_ids, Ld = dlg.index.to_numpy(), dlg.to_numpy()
rng = np.random.default_rng(SEED)

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
    n_splits = max(3, round(1 / CONFIG["test_frac"]))
    folds = list(MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True,
                                           random_state=SEED).split(dlg_ids, Ld))
    test_idx, val_idx = folds[0][1], folds[1][1]
    train_idx = np.setdiff1d(np.arange(len(dlg_ids)), np.union1d(test_idx, val_idx))
    split_mode = f"MultilabelStratifiedKFold(n_splits={n_splits})"
except Exception as e:
    print("iterstrat unavailable (%s) - grouped random split" % e)
    perm = rng.permutation(len(dlg_ids))
    nt = int(len(perm) * CONFIG["test_frac"]); nv = int(len(perm) * CONFIG["val_frac"])
    test_idx, val_idx, train_idx = perm[:nt], perm[nt:nt+nv], perm[nt+nv:]
    split_mode = "grouped-random"

split_of = {}
for idx, nm in [(train_idx, "train"), (val_idx, "val"), (test_idx, "test")]:
    for i in idx: split_of[dlg_ids[i]] = nm
df["split"] = df.dialogue_id.map(split_of)
assert df["split"].notna().all(), "some dialogue was not assigned a split"
assert df.groupby("dialogue_id")["split"].nunique().max() == 1, "dialogue leaked across splits"

print("split mode:", split_mode)
print(df.groupby("split").agg(turns=("turn_id", "size"), dialogues=("dialogue_id", "nunique")))

cov = pd.DataFrame({sp: pd.DataFrame(Y_STRAT[df.split.values == sp], columns=STRATS)
                        .sum(axis=0).astype(int) for sp in ["train", "val", "test"]})
cov["total"] = cov.sum(axis=1)
ZERO_SUPPORT = list(cov[cov[["train", "val", "test"]].eq(0).any(axis=1)].index)
print("\nstrategies absent from at least one split:", ZERO_SUPPORT or "none")
print("  (these can only score F1 = 0; they are reported, never silently averaged away)")
display(cov.sort_values("total").head(10))

train_df = df[df.split == "train"].reset_index(drop=True).copy()
val_df   = df[df.split == "val"].reset_index(drop=True).copy()
test_df  = df[df.split == "test"].reset_index(drop=True).copy()


## 3b. EDA augmentation -- train split only, label-safety-guarded

Turns that carry a **rare** strategy (few training occurrences) get 2-4 synthetic
copies so the encoder sees more gradient signal for those classes. Only the two
lowest-risk EDA operations are used -- **synonym replacement** and **word swap**;
deletion and insertion are excluded because they more easily change *which*
strategy a turn expresses. Speaker markers, numbers and a curated list of
strategy-signalling words are never touched, and with
`aug_current_turn_only = True` the perturbation is confined to the utterance
being labelled -- the context is left verbatim, since corrupting it changes the
evidence without changing the label.

Applied strictly post-split, with a deterministic per-copy seed (`zlib.crc32`,
not Python's per-process-salted `hash()`), so reruns reproduce exactly.

> **Limitation (carried forward from the v4/v8 notebooks):** the rarest
> strategies (`door_in_the_face` = 2 turns, `authority_endorsement` = 5, ...) have
> only a handful of real source turns; every synthetic copy derives from those,
> so augmentation buys gradient signal, not new information. Treat rare-class F1
> gains as partial mitigation, not a solved problem.


In [ ]:
# =============================================================
# 3b. EDA AUGMENTATION (train split only)
# =============================================================
import random as _random

SPEAKER_MARKERS = {"[persuader]", "[persuadee]"}
# Words that carry strategy signal -- never replaced or swapped away.
PROTECTED_WORDS = {
    "donate","donation","donations","give","giving","pledge","match","matching","matched",
    "deadline","today","tonight","now","urgent","urgently","hurry","limited","expires","ends",
    "children","child","kids","families","poverty","hunger","suffering","dying","orphan",
    "guilt","guilty","shame","conscience","obligation","should","ought","responsibility",
    "percent","%","cents","cent","dollar","dollars","$","tax","deductible","receipt",
    "i","my","me","we","our","story","personally","myself",
    "statistics","study","research","evidence","data","proven","report",
    "not","no","never","don't","dont","won't","wont","can't","cant",
}
_SYNONYMS = {
    "good": ["nice","great","fine","decent"], "really": ["truly","genuinely","honestly"],
    "think": ["believe","feel","figure"], "want": ["would like","wish","hope"],
    "help": ["assist","support","aid"], "people": ["folks","individuals","persons"],
    "important": ["significant","vital","essential"], "cause": ["mission","campaign","effort"],
    "understand": ["see","get","realize"], "sure": ["certain","confident","positive"],
    "great": ["wonderful","fantastic","excellent"], "big": ["large","huge","major"],
    "small": ["tiny","little","modest"], "many": ["numerous","plenty of","lots of"],
    "need": ["require","could use"], "amazing": ["incredible","remarkable"],
    "happy": ["glad","pleased","delighted"], "hard": ["tough","difficult","rough"],
    "world": ["planet","globe"], "every": ["each","all"],
    "organization": ["charity","nonprofit","foundation"], "kids": ["children","youngsters"],
}
def _syn(w):
    return _random.choice(_SYNONYMS[w.lower()]) if w.lower() in _SYNONYMS else None

def _locked(tok):
    low = tok.strip(".,!?;:\"'()").lower()
    return (low in PROTECTED_WORDS or low in SPEAKER_MARKERS
            or tok.lower() in SPEAKER_MARKERS or any(ch.isdigit() for ch in tok))

def eda_synonym_replacement(words, n):
    w = words.copy()
    cand = [i for i, x in enumerate(w) if not _locked(x) and _syn(x)]
    _random.shuffle(cand)
    for idx in cand[:n]:
        w[idx] = _syn(w[idx])
    return w

def eda_random_swap(words, n):
    w = words.copy()
    sw = [i for i, x in enumerate(w) if not _locked(x)]
    for _ in range(n):
        if len(sw) < 2: break
        i, j = _random.sample(sw, 2); w[i], w[j] = w[j], w[i]
    return w

def eda_augment_one(text, alpha, seed=None):
    if seed is not None: _random.seed(seed)
    words = text.split(" ")
    if len(words) < 3: return text
    n_ops = max(1, int(alpha * len(words)))
    return " ".join((eda_synonym_replacement if _random.random() < 0.5
                     else eda_random_swap)(words, n_ops))

# Greedy `.*` -> the match lands on the LAST "[Persuader] " marker, i.e. the turn
# being labelled. Anything before it (context) is carried through verbatim and
# never perturbed.
_CUR_RE = re.compile(r"^(.*)(\[Persuader\] )(.*)$", re.S)

def augment_model_text(mt, alpha, seed):
    """Perturb only the utterance being labelled; leave the context verbatim."""
    if not CONFIG["aug_current_turn_only"]:
        return eda_augment_one(mt, alpha, seed)
    m = _CUR_RE.match(mt)
    if not m:
        return eda_augment_one(mt, alpha, seed)
    head, marker, cur = (m.group(1) or ""), m.group(2), m.group(3)
    return head + marker + eda_augment_one(cur, alpha, seed)

def _aug_seed(turn_id, k):
    """Deterministic across processes -- Python's hash() is salted per run."""
    return zlib.crc32(f"{turn_id}|{k}".encode("utf-8")) & 0x7FFFFFFF

train_df["_augmented"] = False
if CONFIG["use_augmentation"]:
    tr_counts = pd.DataFrame(Y_STRAT[df.split.values == "train"], columns=STRATS).sum(0)
    # TIERED copy counts. The previous two-tier scheme (<120 -> 2 copies,
    # 120-300 -> 1) gave `deadline_pressure` (2 train turns) the same treatment
    # as `urgency_appeal` (109). Under a macro-F1 objective every label counts
    # the same, so the copy count now scales with how starved the label is.
    # This buys GRADIENT SIGNAL, not information -- eight copies of two real
    # turns is still two real turns, and S19 note 9 still applies -- but the
    # 30-150 train band is where it can genuinely move the needle.
    TIERS = [(int(t), int(c)) for t, c in CONFIG.get("aug_tiers",
             [[CONFIG["aug_rare_threshold"], CONFIG["aug_copies_rare"]],
              [CONFIG["aug_midrare_threshold"], CONFIG["aug_copies_midrare"]]])]
    TIERS.sort()
    def _copies_for(ss):
        """Copies for a turn = the count of its most starved strategy."""
        best = 0
        for st in ss:
            c = tr_counts.get(st, 0)
            for lim, cp in TIERS:
                if c < lim:
                    best = max(best, cp); break
        return best
    print("augmentation tiers (train support < X -> Y extra copies):",
          {("<%d" % t): c for t, c in TIERS})
    for lim, cp in TIERS:
        _lo = max([0] + [t for t, _ in TIERS if t < lim])
        _in = sorted(tr_counts[(tr_counts >= _lo) & (tr_counts < lim)].index)
        print("  %4d-%4d train turns -> +%d copies  (%d strategies) %s"
              % (_lo, lim, cp, len(_in), _in if len(_in) <= 8 else _in[:8] + ["..."]))
    rare    = set(tr_counts[tr_counts < TIERS[-1][0]].index)
    midrare = set()

    _n_before = len(train_df)
    _tid  = train_df["turn_id"].to_numpy()
    _txt  = train_df["text"].to_numpy()          # perturb the UTTERANCE, not a rendered view
    _sets = train_df["strategies"].tolist()
    src_rows, new_texts = [], []
    for i, sset in enumerate(_sets):
        n = _copies_for(set(sset or []))
        for k in range(n):
            src_rows.append(i)
            new_texts.append(eda_augment_one(_txt[i], CONFIG["aug_alpha"],
                                             _aug_seed(_tid[i], k)))
    if src_rows:
        aug = train_df.iloc[src_rows].copy()
        aug["text"] = new_texts
        aug["_augmented"] = True
        train_df = (pd.concat([train_df, aug], ignore_index=True)
                    .sample(frac=1.0, random_state=SEED).reset_index(drop=True))
        # every context width is re-rendered from the perturbed utterance, so the
        # context stays verbatim and all views stay consistent with each other
        train_df = add_text_cols(train_df)
    print(f"\ntrain turns: {_n_before} -> {len(train_df)} "
          f"({int(train_df._augmented.sum())} synthetic, "
          f"{100*train_df._augmented.mean():.1f}% of the training set)")

    _Y_after = np.vstack(train_df["strategies"].apply(strat_vec))
    _cmp = pd.DataFrame({"before": tr_counts,
                         "after": pd.DataFrame(_Y_after, columns=STRATS).sum(0)})
    _cmp["x"] = (_cmp["after"] / _cmp["before"].clip(lower=1)).round(2)
    display(_cmp.loc[sorted(rare)].sort_values("before"))
    _ex = train_df[train_df._augmented].head(1)
    if len(_ex):
        _orig = df.loc[df.turn_id == _ex.turn_id.iloc[0], "text"].iloc[0]
        print("\n--- example augmented turn (context untouched) ---")
        print("  original :", str(_orig)[:170])
        print("  augmented:", str(_ex.text.iloc[0])[:170])
else:
    print("augmentation disabled")

for _f in (train_df, val_df, test_df):
    _f["model_text"] = _f[f"mt{CONFIG['context_turns']}"]

Y_STRAT_TR = np.vstack(train_df["strategies"].apply(strat_vec))
Y_STRAT_VA = np.vstack(val_df["strategies"].apply(strat_vec))
Y_STRAT_TE = np.vstack(test_df["strategies"].apply(strat_vec))

if CONFIG["FAST_DEV_RUN"]:
    if CONFIG["SMOKE_TEST"]:
        nt, nv, nte = CONFIG["smoke_n_train"], CONFIG["smoke_n_val"], CONFIG["smoke_n_test"]
        tag = "SMOKE_TEST"
    else:
        nt, nv, nte = CONFIG["fastdev_n_train"], CONFIG["fastdev_n_val"], CONFIG["fastdev_n_test"]
        tag = "FAST_DEV_RUN"
    train_df = train_df.head(nt).reset_index(drop=True); Y_STRAT_TR = Y_STRAT_TR[:nt]
    val_df   = val_df.head(nv).reset_index(drop=True);   Y_STRAT_VA = Y_STRAT_VA[:nv]
    test_df  = test_df.head(nte).reset_index(drop=True); Y_STRAT_TE = Y_STRAT_TE[:nte]
    print(f"[{tag}] subsampled:", len(train_df), len(val_df), len(test_df))

print("\nshapes  train %s  val %s  test %s" % (Y_STRAT_TR.shape, Y_STRAT_VA.shape, Y_STRAT_TE.shape))


### 3c. How high can any model score here? -- the annotation ceiling

Every F1 later in this notebook is meaningless without this number. There is no
doubly-annotated subset in this corpus (each of the 10,600 turns was labelled
once, by one of four annotators), so inter-annotator agreement cannot be
computed directly. Two proxies can:

1. **Exact duplicates.** A model that sees `(annotator, text)` must give
   identical inputs identical outputs. Wherever the corpus labels the same
   utterance two different ways, some of those turns are unwinnable for *any*
   such model. Scoring the best possible deterministic predictor -- the majority
   label vector within each group of identical inputs -- gives a hard upper
   bound.
2. **Near duplicates.** When two turns say essentially the same thing and get
   different label sets, that difference is annotation variance, not signal a
   model could learn. The F1 between the label sets of highly-similar turn
   pairs estimates how reproducible the labelling process itself is -- split by
   whether the two turns were labelled by the *same* annotator (self-consistency)
   or by *different* ones (cross-annotator agreement).

The second number is the one that matters. A model cannot be expected to agree
with an annotator more than that annotator agrees with themselves.


In [ ]:
# =============================================================
# 3c. ANNOTATION CEILING  (what any model could possibly score)
# =============================================================
CEILING = {}
if not CONFIG.get("estimate_noise_ceiling", True):
    print("noise-ceiling estimate disabled in CONFIG")
else:
    _t0 = time.time()
    _te = (df["split"].to_numpy() == "test")
    _norm = (df["text"].astype(str).str.strip().str.lower()
             .str.replace(r"[ \t]+", " ", regex=True))
    _ann = df["annotator"].astype(str)

    def _dup_ceiling(keys, tag):
        """Best deterministic predictor of `keys` = majority vote per group."""
        g = pd.Series(np.asarray(keys))
        grp = g.groupby(g).ngroup().to_numpy()
        ng = int(grp.max()) + 1
        sizes = np.bincount(grp, minlength=ng)
        sums = np.zeros((ng, len(STRATS)), np.float64)
        np.add.at(sums, grp, Y_STRAT)
        maj = ((sums >= sizes[:, None] / 2.0) & (sums > 0))
        P = maj[grp].astype(np.float32)
        yt, pt = Y_STRAT[_te], P[_te]
        m = multilabel_metrics(yt, pt)
        n_dup = int((sizes[grp][_te] > 1).sum())
        print("  %-26s max micro %.4f | max macro %.4f | max macro(>=%d) %.4f"
              "   [%d/%d test turns share their input]"
              % (tag, m["micro_f1"], m["macro_f1"],
                 CONFIG.get("macro_min_test_support", 10), m["macro_f1_eval"],
                 n_dup, int(_te.sum())))
        return {"micro_f1": m["micro_f1"], "macro_f1": m["macro_f1"],
                "macro_f1_eval": m["macro_f1_eval"], "n_dup_test": n_dup}

    print("1) EXACT-duplicate bound -- the best any deterministic model could do:")
    CEILING["exact_annotator_text"] = _dup_ceiling(_ann + "||" + _norm, "(annotator, text)")
    CEILING["exact_text"] = _dup_ceiling(_norm, "(text) only")

    # ---- 2) near-duplicate agreement -----------------------------------
    print("\n2) NEAR-duplicate agreement -- how reproducible the labelling is:")
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer as _TV
        from sklearn.preprocessing import normalize as _nz
        from scipy.sparse import hstack as _hs
        _w = _TV(max_features=60000, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
        _c = _TV(max_features=60000, ngram_range=(3, 5), analyzer="char_wb", min_df=2)
        _N = _nz(_hs([_w.fit_transform(df["text"].astype(str)),
                      _c.fit_transform(df["text"].astype(str))]).tocsr())
        _annv = _ann.to_numpy()
        _bkt = {"0.95+": [[], []], "0.90-0.95": [[], []], "0.80-0.90": [[], []]}
        _CH = 500
        for a in range(0, _N.shape[0], _CH):
            b = min(a + _CH, _N.shape[0])
            S = np.asarray((_N[a:b] @ _N.T).todense(), dtype=np.float32)
            for r in range(b - a):
                i = a + r
                S[r, i] = -1.0
                j = int(S[r].argmax()); sim = float(S[r, j])
                k = ("0.95+" if sim >= 0.95 else "0.90-0.95" if sim >= 0.90
                     else "0.80-0.90" if sim >= 0.80 else None)
                if k is None:
                    continue
                inter = float((Y_STRAT[i] * Y_STRAT[j]).sum())
                tot = float(Y_STRAT[i].sum() + Y_STRAT[j].sum())
                f = 1.0 if tot == 0 else 2 * inter / tot
                _bkt[k][0 if _annv[i] == _annv[j] else 1].append(f)
        print("   %-11s %8s %14s %14s" % ("similarity", "pairs", "SAME annotator",
                                          "DIFFERENT annotator"))
        for k in ("0.95+", "0.90-0.95", "0.80-0.90"):
            sa, da = _bkt[k]
            if not (sa or da):
                continue
            print("   %-11s %8d %14s %14s"
                  % (k, len(sa) + len(da),
                     ("%.3f (n=%d)" % (np.mean(sa), len(sa))) if sa else "-",
                     ("%.3f (n=%d)" % (np.mean(da), len(da))) if da else "-"))
        _sa, _da = _bkt["0.95+"]
        CEILING["near_dup_same_annotator"] = float(np.mean(_sa)) if _sa else None
        CEILING["near_dup_diff_annotator"] = float(np.mean(_da)) if _da else None
        del _N
        gc.collect()
    except Exception as _exc:
        print("   skipped (%s: %s)" % (type(_exc).__name__, _exc))

    _self = CEILING.get("near_dup_same_annotator")
    _cross = CEILING.get("near_dup_diff_annotator")
    if _self:
        # Pairwise agreement is NOT itself the ceiling. A predictor of the
        # CONSENSUS scores higher than two noisy draws of that consensus agree
        # with each other. Under the standard miss-only noise model -- each
        # annotation independently keeps each consensus label with probability
        # q, which makes the pairwise F1 exactly q -- a perfect consensus
        # predictor scores 2q/(1+q) against any single annotation.
        _ceil_self = 2 * _self / (1 + _self)
        CEILING["consensus_predictor_bound_self"] = float(_ceil_self)
        print("\n=> On near-identical text, an annotator agrees with THEMSELVES at "
              "F1 %.3f;" % _self)
        if _cross:
            _ceil_cross = 2 * _cross / (1 + _cross)
            CEILING["consensus_predictor_bound_cross"] = float(_ceil_cross)
            print("   two different annotators agree at F1 %.3f." % _cross)
        print("\n   Pairwise agreement is not the ceiling: a model that predicts the")
        print("   CONSENSUS beats two noisy draws of it. Under the miss-only noise")
        print("   model (pairwise F1 = q  =>  consensus predictor scores 2q/(1+q)")
        print("   against a single annotation), the practical ceiling is:")
        print("     ~%.2f  against this annotator's own consistent judgement" % _ceil_self)
        if _cross:
            print("     ~%.2f  against a consensus across annotators" % _ceil_cross)
        print("\n   So a micro-F1 in the high 0.80s is the most that could ever be")
        print("   reached on this data, and only by a model that is a PERFECT")
        print("   consensus predictor. Report scores as a fraction of ~%.2f rather"
              % _ceil_self)
        print("   than of 1.0 -- that is the honest way to show progress here.")
    CEILING["minutes"] = round((time.time() - _t0) / 60, 2)
    json.dump(CEILING, open(os.path.join(CONFIG["out_dir"], "annotation_ceiling.json"),
                            "w"), indent=2)
    print("\n(%.1f min)" % CEILING["minutes"])


## 4. Torch dataset + speaker-role IDs

Identical machinery to `v2 - speaker-aware-encoding` / `v4`: role IDs
(Persuader = 0, Persuadee = 1, special/pad = 2) are computed from the
`[Persuader]`/`[Persuadee]` markers already in the text via the tokenizer's
character-offset mapping, so the same code works for RoBERTa/TOD-BERT (BPE) and
DeBERTa-v3 (SentencePiece), and stays correct under **left** truncation because
offsets index the original string.

This *reads* the speaker markers, it does not predict them.


In [ ]:
# =============================================================
# 4. TORCH DATASET + SPEAKER-ROLE ID COMPUTATION
# =============================================================
_SPEAKER_ROLE_MAP = {"[Persuader]": 0, "[Persuadee]": 1}
_SPEAKER_PATTERN  = re.compile(r"\[Persuader\]|\[Persuadee\]")
ROLE_PERSUADER, ROLE_PERSUADEE, ROLE_SPECIAL = 0, 1, 2

class TurnDataset(Dataset):
    def __init__(self, frame, Ys, text_col="model_text"):
        self.text = frame[text_col].tolist()
        self.ys   = np.asarray(Ys, dtype=np.float32)
        assert len(self.text) == len(self.ys), \
            "frame / label-matrix length mismatch"
    def __len__(self): return len(self.text)
    def __getitem__(self, i):
        return {"text": self.text[i], "y_strat": self.ys[i]}

def compute_batch_role_ids(texts, offset_mapping):
    """Assign a speaker-role id to every token from character offsets."""
    out = []
    for text, offs in zip(texts, offset_mapping.tolist()):
        bounds = sorted(((m.start(), _SPEAKER_ROLE_MAP[m.group()])
                         for m in _SPEAKER_PATTERN.finditer(text)), key=lambda x: x[0])
        row = []
        for cs, ce in offs:
            if cs == 0 and ce == 0:
                row.append(ROLE_SPECIAL); continue
            role = ROLE_SPECIAL
            for bpos, brole in bounds:
                if bpos <= cs: role = brole
                else: break
            row.append(role)
        out.append(row)
    return torch.tensor(out, dtype=torch.long)

class Collator:
    """Picklable collate (a closure would break num_workers>0 on spawn).
    Warns once, not once per batch, if offsets are unavailable."""
    def __init__(self, tokenizer, max_length, use_speaker_roles):
        self.tok, self.max_length = tokenizer, max_length
        self.use_roles = use_speaker_roles
        self.warned = False
    def __call__(self, batch):
        texts = [b["text"] for b in batch]
        kw = dict(padding=True, truncation=True, max_length=self.max_length,
                  return_tensors="pt")
        enc = None
        if self.use_roles:
            try:
                enc = self.tok(texts, return_offsets_mapping=True, **kw)
                enc["role_ids"] = compute_batch_role_ids(texts, enc.pop("offset_mapping"))
            except Exception as exc:
                if not self.warned:
                    print(f"  [WARNING] offset_mapping unavailable ({exc}); "
                          f"speaker roles OFF for this encoder")
                    self.warned = True
                enc = None
        if enc is None:
            enc = self.tok(texts, **kw)
        enc["y_strat"] = torch.from_numpy(np.stack([b["y_strat"] for b in batch]))
        return enc

def load_tokenizer(hf_id):
    try:
        tok = AutoTokenizer.from_pretrained(hf_id, use_fast=True)
    except Exception as exc:
        print(f"  fast tokenizer failed ({exc}); falling back to slow "
              f"(speaker roles will be disabled for this encoder)")
        tok = AutoTokenizer.from_pretrained(hf_id, use_fast=False)
    tok.truncation_side = "left"   # keep the CURRENT turn, drop the oldest context
    return tok

# quick sanity check on the input construction
_t = load_tokenizer("roberta-base")
_demo = _t(df["model_text"].iloc[7], truncation=True, max_length=CONFIG["max_length"])
print("truncation_side:", _t.truncation_side, "| max_length:", CONFIG["max_length"])
print("token length p50/p95/max over the corpus:",
      np.percentile([len(x) for x in _t(df["model_text"].head(2000).tolist())["input_ids"]],
                    [50, 95, 100]).round(0))
print("\nkept TAIL of a context-bearing example (the labelled turn is last):")
print("  ...", _t.decode(_demo["input_ids"])[-220:].replace("\n", " | "))
del _t


## 5. Speaker-aware multi-label model

```
text -> encoder -> last_hidden [B,L,H] -(+ role embedding)-> pooling -> multi-sample dropout -> strategy_head (H -> 11)
```

Single head predicting the 11 categories (the constructor parameter and
internal attribute are still named `n_strat`/`strategy_head` -- purely so
this class is identical code to the strategy-level revision, just fed a
different label space). No auxiliary second head, no decode-time gate.

* **`token_type_ids` support is resolved once in `__init__`**, not by calling
  `inspect.signature()` inside `forward()` on every batch of every replica.
* **Pooling** is selectable: masked mean, mean+max concat, or a learned
  **attention pool** (default) -- it consistently beats plain mean pooling
  here because the persuasive evidence is usually a handful of tokens, not
  the whole utterance.
* **Multi-sample dropout** (Inoue 2019): average the head logits over `k`
  dropout masks. Costs nothing extra (the encoder runs once) and is worth a
  few tenths of an F1 point.


In [ ]:
# =============================================================
# 5. SPEAKER-AWARE MULTI-LABEL MODEL
# =============================================================
class AttentionPool(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))
    def forward(self, hs, mask):
        a = self.proj(hs).squeeze(-1)
        a = a.masked_fill(mask == 0, torch.finfo(a.dtype).min)
        a = torch.softmax(a, dim=-1).unsqueeze(-1)
        return (hs * a).sum(1)

class SpeakerAwareMultiLabel(nn.Module):
    NUM_ROLES = 3
    def __init__(self, hf_id, n_strat, dropout=0.15, use_speaker_roles=True,
                 pooling="attn", n_msd=4, grad_checkpointing=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(hf_id)
        # Force fp32 MASTER weights regardless of the dtype the Hub checkpoint
        # happens to be stored in. AMP here is autocast+GradScaler around an
        # fp32 model, not a half-precision model -- if `from_pretrained` loads
        # some checkpoints (observed: microsoft/deberta-v3-base on newer
        # transformers) in fp16 by default, GradScaler.unscale_() then raises
        # "Attempting to unscale FP16 gradients." on the very first step.
        # .float() is a no-op if the weights were already fp32.
        self.encoder = self.encoder.float()
        H = self.encoder.config.hidden_size
        # resolved ONCE -- not per forward pass, and not per DataParallel replica
        self.accepts_tti = ("token_type_ids" in
                            inspect.signature(self.encoder.forward).parameters)
        if grad_checkpointing and hasattr(self.encoder, "gradient_checkpointing_enable"):
            try:
                self.encoder.gradient_checkpointing_enable(
                    gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:                       # older transformers
                self.encoder.gradient_checkpointing_enable()
            if hasattr(self.encoder.config, "use_cache"):
                self.encoder.config.use_cache = False

        self.use_speaker_roles = use_speaker_roles
        self.role_embedding = nn.Embedding(self.NUM_ROLES, H) if use_speaker_roles else None
        if self.role_embedding is not None:
            nn.init.normal_(self.role_embedding.weight, std=0.02)

        self.pooling = pooling
        self.attn_pool = AttentionPool(H) if pooling == "attn" else None
        pool_dim = H * 2 if pooling == "meanmax" else H

        self.n_msd   = max(1, int(n_msd))
        self.dropout = nn.Dropout(dropout)
        self.strategy_head = nn.Linear(pool_dim, n_strat)
        # No auxiliary category head: tested twice (a direct ablation pilot and
        # a full re-run) and neither recovered any performance, so strategy is
        # the model's only prediction target.

    def _pool(self, hs, attn):
        m = attn.unsqueeze(-1).to(hs.dtype)
        mean = (hs * m).sum(1) / m.sum(1).clamp(min=1e-6)
        if self.pooling == "mean":
            return mean
        if self.pooling == "meanmax":
            mx = hs.masked_fill(m == 0, torch.finfo(hs.dtype).min).max(1).values
            return torch.cat([mean, mx], dim=-1)
        return self.attn_pool(hs, attn)

    def forward(self, input_ids, attention_mask, token_type_ids=None, role_ids=None, **_):
        kw = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None and self.accepts_tti:
            kw["token_type_ids"] = token_type_ids
        hs = self.encoder(**kw).last_hidden_state
        if self.role_embedding is not None and role_ids is not None:
            hs = hs + self.role_embedding(role_ids)
        pooled = self._pool(hs, attention_mask)
        s = 0.0
        for _i in range(self.n_msd):
            d = self.dropout(pooled)
            s = s + self.strategy_head(d)
        # The pooled representation is returned as a second output so section 8b
        # can build a few-shot PROTOTYPE classifier on it for free. DataParallel
        # gathers it along dim 0 like any other batch-first tensor.
        return s / self.n_msd, pooled

print("model class ready | pooling =", CONFIG["pooling"],
      "| multi-sample dropout =", CONFIG["multi_sample_dropout"])


## 6. Loss -- Asymmetric Loss (default) or weighted BCE

The `v8 - hierarchical+aug` notebook uses **class-balanced focal loss** for its
single-label heads. The multi-label analogue is **Asymmetric Loss** (Ben-Baruch
et al., ICCV 2021): a focal term with a *different* focusing parameter for
positives (`gamma_pos = 0`, keep all positive gradient) and negatives
(`gamma_neg = 4`, throw away easy negatives), plus a **probability shift**
(`clip = 0.05`) that hard-zeroes the gradient from negatives the model is already
confident about.

That matters here because the label matrix is 96 % zeros: with 41 sigmoid heads
and 10,600 turns, plain BCE spends almost all of its gradient budget confirming
negatives, and the rare-strategy heads never move. `pos_weight` (the previous
default, still available as `loss_type = "bce"`) fixes the *scale* of that
imbalance but not its *composition* -- it up-weights hard and easy negatives
alike.

Per-label inverse-frequency `pos_weight` is still computed and printed either
way, because it is the honest picture of how skewed the training split is.


In [ ]:
# =============================================================
# 6. LOSS FUNCTIONS + PER-LABEL pos_weight
# =============================================================
def pos_weight_vec(Ymat, cap):
    pos = Ymat.sum(0); neg = len(Ymat) - pos
    return torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, cap), dtype=torch.float32)

POS_W_STRAT = (pos_weight_vec(Y_STRAT_TR, CONFIG["pos_weight_cap"])
               if CONFIG["use_pos_weight"] else torch.ones(len(STRATS)))

_pos = Y_STRAT_TR.sum(0)
print("train positives/label: min %d (%s)  max %d (%s)  median %d"
      % (_pos.min(), STRATS[int(_pos.argmin())], _pos.max(), STRATS[int(_pos.argmax())],
         int(np.median(_pos))))
print("label matrix is %.1f%% zeros" % (100 * (1 - Y_STRAT_TR.mean())))
print("strategy pos_weight: min %.1f  max %.1f (cap %.0f)"
      % (POS_W_STRAT.min(), POS_W_STRAT.max(), CONFIG["pos_weight_cap"]))
print("highest-weighted strategies:",
      [STRATS[i] for i in POS_W_STRAT.argsort(descending=True)[:5].tolist()])

class AsymmetricLoss(nn.Module):
    """Ben-Baruch et al. 2021. gamma_neg > gamma_pos + a probability shift on the
    negative branch: easy negatives contribute (almost) nothing.

    `pos_weight` is an addition, off in v4 and on by default now. Plain ASL
    treats a positive of `rapport_building` (2,034 train turns) and a positive
    of `deadline_pressure` (2 train turns) as equally important, which is right
    for micro-F1 and wrong for macro-F1 -- where the rare label carries 1/41 of
    the score all by itself. Weighting the positive branch by
    min(neg/pos, cap) per label puts the gradient where the macro objective
    puts the credit.

    Note on scale: the weighted loss is ~1.4x larger in magnitude, which does
    NOT act as a learning-rate change here -- AdamW normalises by the gradient
    second moment, so a constant factor on the loss cancels to first order."""
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, eps=1e-8,
                 pos_weight=None):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
        self.register_buffer("pw", None if pos_weight is None
                             else torch.as_tensor(pos_weight, dtype=torch.float32))
    def forward(self, logits, targets):
        x = torch.sigmoid(logits.float())
        targets = targets.float()
        xs_pos = x
        xs_neg = (1.0 - x + self.clip).clamp(max=1.0) if self.clip > 0 else (1.0 - x)
        pos_term = targets * torch.log(xs_pos.clamp(min=self.eps))
        if self.pw is not None:
            pos_term = pos_term * self.pw.to(logits.device).view(1, -1)
        loss = pos_term + (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        if self.gn > 0 or self.gp > 0:
            with torch.no_grad():                       # standard ASL: weight is a constant
                pt = xs_pos * targets + xs_neg * (1.0 - targets)
                g  = self.gp * targets + self.gn * (1.0 - targets)
                w  = torch.pow(1.0 - pt, g)
            loss = loss * w
        return -loss.mean()

def make_loss_fn(pos_w):
    """Returns loss(logits, target) for whichever loss_type CONFIG selects."""
    if CONFIG["loss_type"] == "asl":
        _asl = AsymmetricLoss(CONFIG["asl_gamma_neg"], CONFIG["asl_gamma_pos"],
                              CONFIG["asl_clip"],
                              pos_weight=(pos_w if CONFIG.get("asl_use_pos_weight")
                                          else None))
        return lambda lg, y: _asl(lg, y)
    def _bce(lg, y):
        return nn.functional.binary_cross_entropy_with_logits(
            lg.float(), y.float(), pos_weight=pos_w.to(lg.device))
    return _bce

LOSS_STRAT = make_loss_fn(POS_W_STRAT)

# Train-split prevalence per strategy. Used by the `prevalence` decision rule to
# give every rare label a threshold derived from how OFTEN it occurs rather than
# from the two or three positives it happens to have in validation. Train is a
# different split from val/test, so nothing leaks.
# Computed from the REAL train rows, never the augmented copies: the rare tiers
# get 8 synthetic copies each, so Y_STRAT_TR would report `deadline_pressure` at
# 18/12772 instead of its true 2/7592 -- a 5x inflation, and worst exactly where
# the prevalence rule matters most. The deliberate over-prediction that macro-F1
# wants is supplied by the fitted `prevalence_alphas` instead, where it is
# visible and one parameter rather than an accident of the copy schedule.
_real_tr = (~train_df["_augmented"].to_numpy()) if "_augmented" in train_df else None
_Y_real = Y_STRAT_TR if _real_tr is None else Y_STRAT_TR[_real_tr]
PREV_TRAIN = np.clip(_Y_real.mean(0), 1.0 / max(len(_Y_real), 1), 1.0)
print("\nloss_type:", CONFIG["loss_type"],
      "| ASL pos_weight:", bool(CONFIG.get("asl_use_pos_weight")),
      "| strategy weight", CONFIG["strategy_loss_weight"])
print("target metric for EVERY selection step:", CONFIG.get("target_metric"))
print("train prevalence: rarest %.5f (%s), commonest %.4f (%s)"
      % (PREV_TRAIN.min(), STRATS[int(PREV_TRAIN.argmin())],
         PREV_TRAIN.max(), STRATS[int(PREV_TRAIN.argmax())]))


## 7. Train / eval -- AMP, LLRD, decision-rule search

Hand-written loop, one function reused across all three encoders. Everything in
it that is not in the donation-intent notebooks is here for a T4x2 reason:

* **`nn.DataParallel`** splits the batch across both T4s. `batch_size` in
  `CONFIG` is the *total* batch, so 32 means 16 per GPU.
* **fp16 autocast + `GradScaler`** -- roughly 2x throughput on Turing tensor
  cores and ~40 % less activation memory, which is what makes 10 epochs x 3
  encoders fit in a Kaggle session. `torch.nn.parallel.parallel_apply` propagates
  the autocast state into each device thread, so AMP and DataParallel compose.
  The scaler silently skips any step whose gradients overflowed, which is also
  the cheapest guard against DeBERTa-v3's fp16 NaNs; the explicit non-finite-loss
  skip from v4 is kept on top.
* **Layer-wise LR decay** (`lr * 0.85^depth`, heads at `8x`): lower layers hold
  general syntax that does not need to move; the randomly-initialised head does.
  Standard, and reliably worth a point of macro-F1 on small corpora.
* **Cosine schedule** with warmup, grad-norm clip at 1.0, `AdamW eps=1e-6`.
* **Early stopping** on the honest tuned-threshold val micro-F1 (below), with the
  best checkpoint reloaded before test.
* **OOM / crash containment**: a failing encoder is caught at the call site in
  section 8, retried once at half batch with gradient checkpointing, and only
  then skipped -- one bad encoder never kills the run.
* **Wall-clock budget** per encoder so a slow model cannot consume the session.

### Decision rule -- the part that was costing the most

The previous revision tuned **each label's own F1** independently. That is
locally optimal and globally wrong, and on this corpus it cost **-0.038
micro-F1**, measured. The mechanism: `cost_benefit_framing` has 3 test
positives, and its own F1 is maximised at threshold 0.18 -- recall 1.0,
precision 0.011, i.e. emitting ~273 predictions to catch 3. Three labels
behaving that way contributed roughly **700 false positives** against 2,598 true
positive slots, and predicted cardinality reached 3.3 against a true 1.74. Every
one of those thresholds maximised its own label's F1; together they wrecked the
shared micro-F1 denominator.

So the notebook now fits several candidate rules and picks between them:

| rule | fitted params | idea |
|---|---|---|
| `fixed@0.5` | 0 | no fitting at all -- the floor nothing may fall below |
| `global_micro` | 1 | one threshold for all labels, maximising **micro**-F1 |
| `cardinality` | 1 | one threshold making predicted cardinality match the truth |
| `coord_shrunk` | ~20 | coordinate ascent, pulled halfway back to the global threshold |
| `coord_micro` | 39 | coordinate ascent on micro-F1; may switch a label **off** |
| `coord_keepall` | 39 | same, but never switches a label off |
| `perlabel_ownF1` | 39 | the old rule -- kept **only** as a reported baseline |

Selection is on **grouped, complexity-penalised, cross-fitted validation**:

* **grouped** -- the cross-fit folds are split by `dialogue_id`, not by turn.
  With random turn-level folds, turns of one dialogue land on both sides, the
  held-out part is not really unseen, and a 39-parameter rule scores far better
  than it generalises. Grouping reproduces the real val->test shift.
* **complexity-penalised** -- each rule's score is docked `k / 2n` for the `k`
  thresholds it fits, so a 39-parameter rule must beat a 1-parameter rule by a
  real margin rather than by fitting noise on ~1.5k rows.
* the plain `0.5` cut is always a candidate **and** always reported, so the
  reported number can never be worse than not tuning at all.

### No hierarchical gate, no whole-dialogue smoothing, no decoupled training

Not present: a category-head-driven decode gate (removed with the head), and
whole-dialogue probability smoothing (blending toward the mean over *other*
turns of the same dialogue, including turns after the one being classified --
non-causal for a live, turn-by-turn deployment).

Decoupled long-tail training (Kang et al. 2020: freeze the encoder, reset the
head, re-fit under class-balanced sampling) was piloted and tested twice --
it under-performed its own parent on every support tier while over-predicting
(higher cardinality), so it isn't part of this pipeline.


In [ ]:
# =============================================================
# 7. TRAIN / EVAL  (manual loop, shared across encoders)
# =============================================================
# ---- AMP compatibility shim (torch 1.x / 2.0-2.2 / 2.3+) --------------
def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def autocast_ctx(enabled):
    try:
        return torch.amp.autocast("cuda", dtype=torch.float16, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def sigmoid_np(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

THR_GRID = np.round(np.arange(CONFIG["threshold_floor"],
                              CONFIG["threshold_ceil"] + 1e-9, 0.01), 4)
NEVER = 1.01                      # a threshold that switches a label off entirely

# ---------------------------------------------------------------------
# DECISION RULES
# ---------------------------------------------------------------------
# The rule this notebook used to ship maximised each label's OWN F1. On this
# corpus that cost -0.038 micro-F1, measured. Why: cost_benefit_framing has 3
# test positives, and its own F1 is maximised at threshold 0.18 (recall 1.0,
# precision 0.011) -- i.e. emitting ~273 predictions to catch 3. Three labels
# behaving like that contributed ~700 false positives against 2,598 true
# positive slots. Each was locally optimal; together they wrecked the shared
# micro-F1 denominator, and predicted cardinality hit 3.3 against a true 1.74.
#
# Every rule below optimises the metric actually being reported, and section 7
# picks between them on grouped, complexity-penalised cross-fitted validation.

def _micro_f1_thr(prob, y, thr):
    pred = prob >= thr; yb = y.astype(bool)
    tp = int((pred & yb).sum()); fp = int((pred & ~yb).sum()); fn = int(((~pred) & yb).sum())
    d = 2 * tp + fp + fn
    return 0.0 if d == 0 else 2 * tp / d

def _label_curves(prob, y, grid):
    """tp/fp/fn for every (label, candidate threshold) -> three [L, G] arrays."""
    L, G = prob.shape[1], len(grid)
    tp = np.zeros((L, G), np.int64); fp = np.zeros((L, G), np.int64); fn = np.zeros((L, G), np.int64)
    for j in range(L):
        pj = prob[:, j]; yj = y[:, j].astype(bool)
        pred = pj[None, :] >= grid[:, None]
        tp[j] = (pred & yj[None, :]).sum(1)
        fp[j] = (pred & ~yj[None, :]).sum(1)
        fn[j] = ((~pred) & yj[None, :]).sum(1)
    return tp, fp, fn

def rule_fixed05(prob, y):
    return np.full(prob.shape[1], 0.5)

def _best_global_micro(prob, y, lo=None, hi=None):
    """EXACT best single global threshold for micro-F1, in O(N log N).

    micro-F1 at cut t is 2*TP(t) / (P(t) + n_pos), where P(t) counts the cells at
    or above t and TP(t) how many of those are positive. Sorting the flattened
    probability matrix once gives both curves at EVERY realisable cut, so there
    is no 0.01 grid for the optimum to fall between. It is also ~30x faster than
    scanning the grid, which is what makes the bagged greedy search in S8c
    affordable (it evaluates this tens of thousands of times)."""
    pr = np.asarray(prob, dtype=np.float64).ravel()
    yy = np.asarray(y).ravel().astype(bool)
    n_pos = int(yy.sum())
    if n_pos == 0 or pr.size == 0:
        return NEVER, 0.0                     # nothing to find: predict nothing
    order = np.argsort(-pr, kind="stable")
    ps, ys = pr[order], yy[order]
    f1 = 2.0 * np.cumsum(ys) / (np.arange(1, ps.size + 1) + n_pos)
    ties = np.r_[ps[1:] != ps[:-1], True]     # only the last cell of a tie-run
    keep = ties.copy()
    if lo is not None: keep &= (ps >= lo)
    if hi is not None: keep &= (ps <= hi)
    if not keep.any():
        keep = ties                           # range too tight -- ignore it
    idx = np.flatnonzero(keep)
    k = idx[int(np.argmax(f1[idx]))]
    return float(ps[k]), float(f1[k])

def rule_global(prob, y, grid=THR_GRID):
    """One threshold for every label, chosen to maximise micro-F1 exactly."""
    t, _ = _best_global_micro(prob, y, CONFIG["threshold_floor"], CONFIG["threshold_ceil"])
    return np.full(prob.shape[1], min(t, 1.0))

def rule_cardinality(prob, y, grid=THR_GRID):
    """Global threshold that matches predicted label cardinality to the truth."""
    target = y.sum(1).mean()
    best_t, best_gap = 0.5, 1e9
    for t in grid:
        gap = abs((prob >= t).sum(1).mean() - target)
        if gap < best_gap: best_gap, best_t = gap, float(t)
    return np.full(prob.shape[1], best_t)

def best_perlabel_macro(prob, y, grid=THR_GRID):
    """Per-label F1-optimal thresholds AND the macro-F1 they achieve, in one
    pass over the tp/fp/fn curves.

    This is the exact maximiser of macro-F1 for a fixed probability matrix:
    macro-F1 is a plain average of per-label F1s with NO shared denominator, so
    each label's threshold can be optimised independently without affecting any
    other. That is the whole reason `perlabel_ownF1` is a disaster for micro-F1
    (where the denominator IS shared) and the right answer for macro."""
    L = prob.shape[1]
    thr = np.full(L, 0.5)
    best = np.zeros(L)
    tp, fp, fn = _label_curves(prob, y, grid)
    for j in range(L):
        if y[:, j].sum() == 0:
            thr[j] = NEVER               # unscoreable: do not spend predictions on it
            continue
        d = 2 * tp[j] + fp[j] + fn[j]
        f1 = np.where(d > 0, 2 * tp[j] / np.maximum(d, 1), 0.0)
        k = int(f1.argmax())
        thr[j], best[j] = float(grid[k]), float(f1[k])
    return thr, float(best.mean())

def rule_perlabel_ownf1(prob, y, grid=THR_GRID):
    """Maximise each label's OWN F1. Catastrophic for micro-F1 (measured
    -0.038 on this corpus: rare labels emit hundreds of false positives into
    the shared denominator) and EXACTLY optimal for macro-F1, which has no
    shared denominator. Which is why the rule search is now metric-aware."""
    thr, _ = best_perlabel_macro(prob, y, grid)
    return thr

def rule_perlabel_shrunk(prob, y, grid=THR_GRID, lam=0.5):
    """Per-label own-F1 thresholds pulled halfway to the global one.

    For a label with 3 validation positives the own-F1 optimum is fitted on
    3 points and is mostly noise; the global cut is fitted on ~62k cells and is
    mostly bias. Halfway between beats both often enough to be worth a
    candidate slot, and it costs one parameter more than the global rule."""
    base = rule_global(prob, y, grid)[0]
    own, _ = best_perlabel_macro(prob, y, grid)
    live = own <= 1.0
    out = np.where(live, base + lam * (np.minimum(own, 1.0) - base), NEVER)
    return out

def rule_prevalence(prob, y, grid=None):
    """Per-label cut from the label PREVALENCE, not from its positives.

    Threshold for label j is set so the model predicts about alpha * p_j * n
    positives, where p_j is that strategy's frequency in TRAIN (a different
    split, so nothing leaks) and alpha is ONE globally fitted number.

    This is the rule built for the rare tail. `deadline_pressure` has 2 val
    positives: its own-F1 optimum is a coin flip and the global micro cut never
    fires for it at all, but its prevalence is perfectly estimable from train,
    and a cut that simply predicts the right NUMBER of positives gives it a real
    chance of a non-zero F1. Forty-one adaptive thresholds out of one fitted
    parameter."""
    L = prob.shape[1]
    n = prob.shape[0]
    alphas = CONFIG.get("prevalence_alphas", [0.5, 0.75, 1.0, 1.5, 2.0, 3.0])
    best_thr, best_sc = None, -1.0
    for a in alphas:
        thr = np.empty(L)
        for j in range(L):
            k = int(round(a * PREV_TRAIN[j] * n))
            if k <= 0:
                thr[j] = NEVER; continue
            k = min(k, n)
            # threshold = the k-th largest probability for this label
            thr[j] = float(np.partition(prob[:, j], n - k)[n - k])
        sc = float(label_f1_vector(y, apply_pred(prob, thr)).mean())
        if sc > best_sc:
            best_sc, best_thr = sc, thr
    return best_thr

def rule_coord_micro(prob, y, grid=THR_GRID, passes=3, allow_off=True):
    """Coordinate ascent on MICRO-F1, with 'never predict' as a candidate.

    Starts from the best global threshold, then re-optimises one label at a
    time against the real global TP/FP/FN pool. A label whose predictions cost
    more false positives than they win true positives gets switched off."""
    L = prob.shape[1]
    g = np.concatenate([grid, [NEVER]]) if allow_off else np.asarray(grid, float)
    tp, fp, fn = _label_curves(prob, y, g)
    thr0 = rule_global(prob, y, grid)
    idx = np.array([int(np.argmin(np.abs(g - t))) for t in thr0])
    TP = int(tp[np.arange(L), idx].sum()); FP = int(fp[np.arange(L), idx].sum())
    FN = int(fn[np.arange(L), idx].sum())
    order = np.argsort(-y.sum(0))
    for _ in range(passes):
        changed = False
        for j in order:
            c_tp = TP - tp[j, idx[j]] + tp[j]
            c_fp = FP - fp[j, idx[j]] + fp[j]
            c_fn = FN - fn[j, idx[j]] + fn[j]
            d = 2 * c_tp + c_fp + c_fn
            f1 = np.where(d > 0, 2 * c_tp / np.maximum(d, 1), 0.0)
            k = int(f1.argmax())
            if k != idx[j]: changed = True
            idx[j] = k; TP, FP, FN = int(c_tp[k]), int(c_fp[k]), int(c_fn[k])
        if not changed: break
    return g[idx].astype(float)

def rule_coord_keepall(prob, y, grid=THR_GRID):
    return rule_coord_micro(prob, y, grid, allow_off=False)

def rule_coord_shrunk(prob, y, grid=THR_GRID, lam=0.5):
    """Coordinate-ascent thresholds pulled halfway back to the single global
    one -- most of the per-label signal at a fraction of the variance."""
    base = rule_global(prob, y, grid)[0]
    co = rule_coord_micro(prob, y, grid, allow_off=True)
    return np.where(co > 1.0, NEVER, base + lam * (np.minimum(co, 1.0) - base))

# Ordered simplest -> most flexible; near-ties break toward the simpler rule.
DECISION_RULES = {
    "fixed@0.5":       rule_fixed05,
    "global_micro":    rule_global,
    "cardinality":     rule_cardinality,
    "prevalence":      rule_prevalence,
    "coord_shrunk":    rule_coord_shrunk,
    "perlabel_shrunk": rule_perlabel_shrunk,
    "coord_micro":     rule_coord_micro,
    "coord_keepall":   rule_coord_keepall,
    "perlabel_ownF1":  rule_perlabel_ownf1,
}
# Thresholds each rule actually FITS to the data it is given -- used for an
# AIC-flavoured optimism correction so a 41-parameter rule has to beat a
# 1-parameter rule by a real margin rather than by fitting noise.
# Parameter counts scale with len(STRATS) (39, after discarding the two
# zero-test-support strategies), not a hardcoded 41 -- coord_shrunk/
# perlabel_shrunk use a half-count shrinkage heuristic (ceil(n/2)), the full
# per-label rules charge one parameter per label.
_N_STRAT = len(STRATS)
_RULE_NPARAM = {"fixed@0.5": 0, "global_micro": 1, "cardinality": 1,
                "prevalence": 1, "coord_shrunk": (_N_STRAT + 1) // 2,
                "perlabel_shrunk": (_N_STRAT + 1) // 2,
                "coord_micro": _N_STRAT, "coord_keepall": _N_STRAT,
                "perlabel_ownF1": _N_STRAT}
# Rules whose parameters are PER LABEL, i.e. fitted only against that label own
# F1 with no interaction between labels.
_PERLABEL_RULES = {"perlabel_ownF1", "perlabel_shrunk", "prevalence"}

def rule_nparam(name, metric="micro"):
    """Effective free-parameter count, which depends on the metric.

    Under micro-F1, per-label thresholds are genuine extra flexibility AND
    misaligned with the objective: labels share one denominator, so tuning a
    rare label against its own F1 spends false positives out of everyone else
    budget. They are penalised accordingly.

    Under macro-F1 there is no shared denominator -- the objective is literally
    the mean of the per-label F1s -- so a per-label threshold is the aligned
    parameterisation rather than a flexibility bonus, and charging it 41
    parameters would rule out the correct answer a priori. Cross-fitting still
    measures whether it generalises; this only stops the tie-breaker from
    vetoing it."""
    k = _RULE_NPARAM.get(name, 0)
    if metric == "macro" and name in _PERLABEL_RULES:
        return 1
    return k

def apply_pred(prob, thr, force_top1=False):
    """Thresholds -> 0/1 prediction matrix.

    With `force_top1`, a row where NOTHING clears its threshold is given its
    arg-max strategy anyway (skipping any label the rule switched off entirely).
    An all-zero row is almost always a thresholding artefact rather than a
    genuine "no strategy here" call, and it can only lose recall."""
    prob = np.asarray(prob)
    pred = (prob >= thr).astype(int)
    if force_top1:
        empty = np.flatnonzero(pred.sum(1) == 0)
        if empty.size:
            live = np.asarray(thr) <= 1.0
            if live.any():
                masked = np.where(live[None, :], prob[empty], -np.inf)
                pred[empty, np.argmax(masked, axis=1)] = 1
    return pred

def grouped_folds(groups, folds, seed=SEED, n=None):
    """Fold row-indices grouped by dialogue (shared by the decode search and
    the level-2 stackers, so both honour the same val->test shift)."""
    if groups is None:
        idx = np.arange(n); np.random.default_rng(seed).shuffle(idx)
        return list(np.array_split(idx, folds))
    uniq = np.unique(groups)
    np.random.default_rng(seed).shuffle(uniq)
    return [np.where(np.isin(groups, ch))[0] for ch in np.array_split(uniq, folds)]

def crossfit_rule(prob, y, rule, groups=None, folds=None, seed=SEED, metric="micro",
                  force_top1=False):
    """Fit the rule on part of val, score the held-out part, average over folds.

    Folds are GROUPED BY DIALOGUE. That matters: with random turn-level folds,
    turns of one dialogue land on both sides, the held-out part is not really
    unseen, and a flexible rule looks better than it generalises. Grouping
    reproduces the real val->test shift (entirely unseen dialogues)."""
    folds = folds or CONFIG["decision_rule_folds"]
    parts = grouped_folds(groups, folds, seed, n=len(y))
    sc = []
    for k in range(len(parts)):
        te = parts[k]
        tr = (np.concatenate([parts[j] for j in range(len(parts)) if j != k])
              if len(parts) > 1 else te)
        if len(te) == 0 or len(tr) == 0: continue
        thr = rule(prob[tr], y[tr])
        sc.append(f1_metric(y[te], apply_pred(prob[te], thr, force_top1), metric))
    return float(np.mean(sc)) if sc else 0.0

def select_decode(vs, vy, groups, metric=None):
    """Search over {force-top1} x {rules}, every combination scored by grouped
    cross-fitted val F1 in CONFIG["target_metric"] (macro by default).

    Returns (force_top1, rule_name, thresholds, table, cf_score).
    Used for single encoders, for the shallow members AND for every ensemble
    candidate, so nothing in this notebook is decoded on privileged terms.

    NOTE: this used to also search {hierarchical gate on/off} and {whole-
    dialogue smoothing alpha}. Both were removed: the gate depended on the
    model's category head, which is gone (team-lead call: strategy is the
    only ask); whole-dialogue smoothing blended each turn's probability
    toward the mean over the OTHER turns of the same dialogue -- including
    turns AFTER the one being classified, which a live, turn-by-turn
    deployment can never see. Both were confirmed removals, not adaptations."""
    metric = metric or CONFIG.get("target_metric", "micro")
    force_opts = CONFIG.get("force_top1_options", [False]) or [False]
    best_pick, table = None, {}
    for force in force_opts:
        rname, rthr, _ = select_decision_rule(vs, vy, groups=groups,
                                              metric=metric,
                                              force_top1=force)
        sc = crossfit_rule(vs, vy, DECISION_RULES[rname], groups=groups,
                           metric=metric, force_top1=force)
        table[f"top1={force}"] = {"rule": rname, "cf_micro": round(sc, 4)}
        if best_pick is None or sc > best_pick[0]:
            best_pick = (sc, force, rname, rthr)
    sc, force, rname, rthr = best_pick
    return force, rname, rthr, table, sc

def apply_decode(ps, groups):
    """Thin pass-through, kept for call-site symmetry with `select_decode` and
    to keep every caller's shape identical. Used to apply the hierarchical
    gate and whole-dialogue smoothing; both were removed (category head gone;
    whole-dialogue smoothing is non-causal for a live deployment), so this no
    longer transforms `ps` at all. `groups` is accepted and unused so call
    sites do not need special-casing."""
    return ps

def select_decision_rule(prob_val, y_val, groups=None, metric=None, force_top1=False):
    """Pick a decision rule on complexity-penalised grouped cross-fitted val,
    then refit it on the whole val split. Returns (name, thresholds, table)."""
    metric = metric or CONFIG.get("target_metric", "micro")
    raw = {}
    for name, fn in DECISION_RULES.items():
        try:
            raw[name] = crossfit_rule(prob_val, y_val, fn, groups=groups, metric=metric,
                                      force_top1=force_top1)
        except Exception:
            raw[name] = float("nan")
    valid = {k: v for k, v in raw.items() if v == v}
    if not valid:
        return "fixed@0.5", rule_fixed05(prob_val, y_val), {}
    n = max(len(y_val), 1)
    adj = {k: v - rule_nparam(k, metric) / (2.0 * n) for k, v in valid.items()}
    top = max(adj.values()); tol = CONFIG["decision_rule_tol"]
    best = next(r for r in DECISION_RULES if r in adj and adj[r] >= top - tol)
    table = {k: {"cf": round(valid[k], 4), "adj": round(adj[k], 4)} for k in valid}
    return best, DECISION_RULES[best](prob_val, y_val), table

# NOTE: `dialogue_neighbor_mean`, `dialogue_smooth` and `hier_gate` were
# removed entirely (Change Sets A and C). `dialogue_neighbor_mean`/
# `dialogue_smooth` computed a whole-dialogue (past+future) leave-one-out
# mean and blended it into a turn's decoded probability -- non-causal for a
# live, turn-by-turn deployment, which never has future turns of the same
# conversation. `hier_gate` multiplied a strategy's probability by its
# parent category's probability from the (now removed) category head. All
# three were confirmed by the user as removals, not adaptations. See
# Section 19 notes for the measured effect sizes and the open note on a
# causal ("turns before this one only") replacement for a future iteration.

RAW_KEYS = ("val_s", "test_s", "full_s")

def package_result(name, hf_id, raw, extra=None, verbose=True):
    """Decode + score ONE member from its raw probability matrices.

    Shared by the transformer encoders (S8), the shallow XGBoost / logistic
    members (S8b) and every ensemble candidate (S8c). Every model in this
    notebook therefore gets: the same grouped cross-fitted decode search, the
    same operating point chosen on VALIDATION (never on test), and the same
    metric dictionary -- which is the only thing that makes S10's comparison
    table mean anything.

    `raw` needs the three RAW_KEYS (raw strategy probabilities) plus
    `val_y` / `test_y`."""
    vgrp = val_df["dialogue_id"].to_numpy()
    tgrp = test_df["dialogue_id"].to_numpy()
    fgrp = df["dialogue_id"].to_numpy()
    vs, ts, fs = (np.asarray(raw[k], dtype=np.float32) for k in RAW_KEYS)
    vy, ty = raw["val_y"], raw["test_y"]

    tgt = CONFIG.get("target_metric", "micro")
    # BOTH operating points, from the same probabilities. micro-F1 and macro-F1
    # are maximised by different thresholds -- micro by a high global cut, macro
    # by per-label cuts that fire for the rare tail -- but they are decisions
    # taken on identical model outputs, so there is no reason to pick one and
    # report the other's number from the wrong cut.
    metrics_set = [m for m in ("micro", "macro")
                   if m == tgt or CONFIG.get("report_both_operating_points", True)]
    DEC = {}
    for _mt in metrics_set:
        _f, _r, _thr, _tab, _cf = select_decode(vs, vy, vgrp, metric=_mt)
        _tpm = apply_decode(ts, tgrp)
        DEC[_mt] = {"force": _f, "rule": _r,
                    "thr": np.asarray(_thr), "table": _tab, "cf": _cf,
                    "metrics": multilabel_metrics(ty, apply_pred(_tpm, _thr, _f))}
    _D = DEC[tgt]
    force, rule_name, THR, tables, cf = (
        _D["force"], _D["rule"], _D["thr"], _D["table"], _D["cf"])
    vp  = apply_decode(vs, vgrp)
    tp  = apply_decode(ts, tgrp)
    fp_ = apply_decode(fs, fgrp)
    n_off = int((np.asarray(THR) > 1.0).sum())

    res = {
        "name": name, "hf_id": hf_id,
        "thresholds": np.asarray(THR).tolist(),
        "rule": rule_name, "rule_tables": tables,
        "force_top1": bool(force), "val_cf": float(cf),
        # RAW (un-decoded) probabilities. S8c blends THESE and then runs ONE
        # decode search on the blend -- averaging matrices that each went
        # through a different decode already would mix incompatible scales.
        "raw": {k: v for k, v in zip(RAW_KEYS, (vs, ts, fs))},
        "val_prob": vp, "val_y": vy, "test_prob": tp, "test_y": ty, "full_prob": fp_,
        "metrics_0.5":   multilabel_metrics(ty, apply_pred(tp, 0.5, False)),
        "metrics_tuned": DEC[tgt]["metrics"],
        # each objective at ITS OWN optimum, so neither number is quoted from
        # the other's operating point
        "metrics_at_micro": DEC.get("micro", {}).get("metrics"),
        "metrics_at_macro": DEC.get("macro", {}).get("metrics"),
        "decode_micro": ({k: v for k, v in DEC["micro"].items() if k != "metrics"}
                         if "micro" in DEC else None),
        "decode_macro": ({k: v for k, v in DEC["macro"].items() if k != "metrics"}
                         if "macro" in DEC else None),
        "val_cf_micro": DEC.get("micro", {}).get("cf"),
        "val_cf_macro": DEC.get("macro", {}).get("cf"),
        # the rule this notebook used to ship, kept so the regression stays visible
        "metrics_oldrule": multilabel_metrics(
            ty, apply_pred(tp, rule_perlabel_ownf1(vp, vy), False)),
        # defaults so that the shallow members and the ensemble are valid rows
        # in every downstream table without special-casing
        "history": [], "epochs_run": 0, "minutes": 0.0, "use_roles": None,
        "ckpt": None, "tokenizer_id": None, "kind": "member",
    }
    res["raw"]["val_y"], res["raw"]["test_y"] = vy, ty
    # The REPORTED operating point is the val-selected one, full stop. It is not
    # re-picked on test: `fixed@0.5` is one of the candidate rules, so if 0.5
    # really is best the search returns it, and reporting max(0.5, tuned) on
    # test would just be a two-way peek at the test split.
    res["metrics_best"] = res["metrics_tuned"]
    res.update(extra or {})
    res["macro_by_tier"] = macro_f1_tiers(ty, apply_pred(tp, THR, force))
    if verbose:
        for _mt in metrics_set:
            _d = DEC[_mt]
            print("  decode@%-5s: rule=%-15s top1=%-5s "
                  "(cf-val %s %.4f)"
                  % (_mt, _d["rule"], _d["force"], _mt, _d["cf"]))
        print(f"    thresholds min {np.minimum(THR,1).min():.2f}"
              f" / median {np.median(np.minimum(THR,1)):.2f}"
              f" / max {np.minimum(THR,1).max():.2f}"
              + (f" | {n_off} label(s) switched off as net-negative" if n_off else ""))
        if n_off:
            print("    off:", [STRATS[j] for j in np.where(np.asarray(THR) > 1.0)[0]])
        _m = res["metrics_tuned"]
        for _mt in metrics_set:
            _mm = DEC[_mt]["metrics"]
            print("  TEST @%-5s-opt  micro %.4f | macro %.4f | macro(>=%d pos) %.4f "
                  "over %d | jacc %.4f | card %.2f%s"
                  % (_mt, _mm["micro_f1"], _mm["macro_f1"],
                     CONFIG.get("macro_min_test_support", 10), _mm["macro_f1_eval"],
                     _mm["n_eval_labels"], _mm["jaccard_samples"], _mm["card_pred"],
                     "   <-- headline" if _mt == tgt else ""))
        print("        macro-F1 by test support (at the %s optimum): " % tgt
              + "  ".join("%s+:%.3f(n=%d)" % (k, v["macro_f1"], v["n_labels"])
                          for k, v in sorted(res["macro_by_tier"].items())))
    return res

@torch.no_grad()
def collect_probs(model, loader, device, use_amp, want_embed=False):
    """Returns (strategy_probs, y_strat[, pooled_embeddings])."""
    model.eval()
    S, YS, E = [], [], []
    for batch in loader:
        ys = batch.pop("y_strat")
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with autocast_ctx(use_amp):
            s_logits, pooled = model(**batch)
        S.append(s_logits.float().cpu().numpy())
        YS.append(ys.numpy())
        if want_embed:
            E.append(pooled.float().cpu().numpy())
    out = (sigmoid_np(np.concatenate(S)), np.concatenate(YS))
    return out + ((np.concatenate(E).astype(np.float32),) if want_embed else ())

def build_optimizer(core, base_lr, llrd_decay=None, head_lr_mult=None):
    """AdamW with layer-wise LR decay and no weight decay on bias / LayerNorm.

    `llrd_decay` / `head_lr_mult` may be overridden per encoder: a 24-layer
    roberta-large or a 128k-vocab DeBERTa-v3 needs a flatter chain than 0.90,
    which would leave its bottom layer on base_lr * 0.9**23 = 9% of base."""
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias",
                "layer_norm.weight", "layer_norm.bias")
    wd = CONFIG["weight_decay"]
    llrd_decay   = CONFIG["llrd_decay"]   if llrd_decay   is None else llrd_decay
    head_lr_mult = CONFIG["head_lr_mult"] if head_lr_mult is None else head_lr_mult
    enc = core.encoder
    n_layers = int(getattr(enc.config, "num_hidden_layers", 12))
    layer_re = re.compile(r"\.layer\.(\d+)\.")
    buckets, enc_ids = defaultdict(list), set()
    for n, p in enc.named_parameters():
        if not p.requires_grad:
            continue
        enc_ids.add(id(p))
        if CONFIG["use_llrd"]:
            m = layer_re.search("." + n)
            depth = int(m.group(1)) if m else (-1 if "embedding" in n else n_layers - 1)
            lr = base_lr * (llrd_decay ** (n_layers - 1 - depth))
        else:
            lr = base_lr
        buckets[(round(lr, 12), 0.0 if any(nd in n for nd in no_decay) else wd)].append(p)
    head_lr = base_lr * (head_lr_mult if CONFIG["use_llrd"] else 1.0)
    for n, p in core.named_parameters():
        if id(p) in enc_ids or not p.requires_grad:
            continue
        buckets[(round(head_lr, 12), 0.0 if any(nd in n for nd in no_decay) else wd)].append(p)
    groups = [{"params": ps, "lr": lr, "weight_decay": w} for (lr, w), ps in buckets.items()]
    return torch.optim.AdamW(groups, lr=base_lr, eps=CONFIG["adam_eps"])

def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:                     # torch < 1.13
        return torch.load(path, map_location="cpu")

def train_one_encoder(enc_cfg, overrides=None):
    """Train + evaluate one encoder. `overrides` lets section 8 retry a run at a
    smaller batch with gradient checkpointing after an OOM."""
    ov = overrides or {}
    name, hf_id = enc_cfg["name"], enc_cfg["hf_id"]
    bs      = ov.get("batch_size", enc_cfg.get("batch_size", CONFIG["batch_size"]))
    ebs     = ov.get("eval_batch_size", enc_cfg.get("eval_batch_size", CONFIG["eval_batch_size"]))
    lr      = enc_cfg.get("lr", CONFIG["lr"])
    wr      = enc_cfg.get("warmup_ratio", CONFIG["warmup_ratio"])
    epochs  = CONFIG["num_epochs"]
    gckpt   = ov.get("grad_checkpointing", CONFIG["grad_checkpointing"])
    nw      = ov.get("num_workers", CONFIG["num_workers"])
    use_amp = ov.get("use_amp", USE_AMP)
    ctxw    = enc_cfg.get("context_turns", CONFIG["context_turns"])
    maxlen  = enc_cfg.get("max_length", CONFIG["max_length"])
    tcol    = f"mt{ctxw}"
    eseed   = enc_cfg.get("seed", SEED)
    torch.manual_seed(eseed); np.random.seed(eseed); random.seed(eseed)

    bar = "=" * 74
    print(f"\n{bar}\nTraining {name}  ({hf_id})")
    print(f"  batch={bs} (total, {max(1,N_GPU)} GPU) | eval_batch={ebs} | lr={lr} | epochs={epochs}")
    print(f"  context={ctxw} turns | max_length={maxlen} | seed={eseed}")
    print(f"  amp={use_amp} | grad_ckpt={gckpt} | roles={CONFIG['use_speaker_roles']} "
          f"| pooling={CONFIG['pooling']} | loss={CONFIG['loss_type']} "
          f"| llrd={enc_cfg.get('llrd_decay', CONFIG['llrd_decay'])}\n{bar}")

    tokenizer = load_tokenizer(hf_id)
    use_roles = CONFIG["use_speaker_roles"] and getattr(tokenizer, "is_fast", False)
    if CONFIG["use_speaker_roles"] and not use_roles:
        print("  [note] slow tokenizer -> no char offsets -> speaker roles disabled here")
    collate = Collator(tokenizer, maxlen, use_roles)

    def dl(frame, Ys, shuffle, batch, workers=0):
        # Workers ONLY on the training loader: keeping a single worker-backed
        # loader is what stops forked workers from inheriting a sibling loader's
        # iterator and spamming "can only test a child process" on exit.
        return DataLoader(TurnDataset(frame, Ys, text_col=tcol), batch_size=batch,
                          shuffle=shuffle, collate_fn=collate, num_workers=workers,
                          persistent_workers=(workers > 0),
                          pin_memory=(DEVICE == "cuda"), drop_last=False)

    train_loader = dl(train_df, Y_STRAT_TR, True,  bs, workers=nw)
    val_loader   = dl(val_df,   Y_STRAT_VA, False, ebs)
    test_loader  = dl(test_df,  Y_STRAT_TE, False, ebs)
    full_loader  = dl(df,       Y_STRAT,    False, ebs)

    core = SpeakerAwareMultiLabel(
        hf_id, len(STRATS), dropout=CONFIG["dropout"],
        use_speaker_roles=use_roles, pooling=CONFIG["pooling"],
        n_msd=CONFIG["multi_sample_dropout"], grad_checkpointing=gckpt).to(DEVICE)
    model = nn.DataParallel(core) if N_GPU > 1 else core

    opt = build_optimizer(core, lr, enc_cfg.get("llrd_decay"),
                          enc_cfg.get("head_lr_mult"))
    print("  optimizer: %d param groups, lr %.2e .. %.2e"
          % (len(opt.param_groups), min(g["lr"] for g in opt.param_groups),
             max(g["lr"] for g in opt.param_groups)))
    total_steps = max(1, len(train_loader) * epochs)
    sched  = get_cosine_schedule_with_warmup(opt, int(total_steps * wr), total_steps)
    scaler = make_scaler(use_amp)

    # Per-epoch selection uses ONE cheap rule (the full search runs once at the
    # end). It has to be a rule ALIGNED WITH THE TARGET METRIC, or the
    # checkpoint chosen is the one that was best at something else: under
    # macro-F1 a single global threshold never fires for the rare tail, so
    # every epoch scores near-identically on it and selection becomes a
    # coin flip.
    tgt_metric = CONFIG.get("target_metric", "micro")
    epoch_rule = "perlabel_shrunk" if tgt_metric == "macro" else "global_micro"
    val_groups = val_df["dialogue_id"].to_numpy()

    ckpt = os.path.join(CKPT_DIR, f"{name}_best.pt")
    snap_k = max(1, int(CONFIG.get("snapshot_k", 1)))
    snaps = []          # (select_score, epoch, path) for the best `snap_k` epochs
    best, no_improve, history, skipped = -1.0, 0, [], 0
    t_start, budget_s = time.time(), CONFIG["time_budget_min_per_encoder"] * 60

    for epoch in range(1, epochs + 1):
        model.train(); run, nb = 0.0, 0
        pbar = tqdm(train_loader, desc=f"[{name}] epoch {epoch}/{epochs}", leave=False)
        for batch in pbar:
            ys = batch.pop("y_strat").to(DEVICE, non_blocking=True)
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with autocast_ctx(use_amp):
                s_logits, _pooled = model(**batch)
            loss = CONFIG["strategy_loss_weight"] * LOSS_STRAT(s_logits, ys)
            if not torch.isfinite(loss):
                skipped += 1; continue          # v4's NaN guard, on top of the scaler's
            scaler.scale(loss).backward()
            scaler.unscale_(opt)                # unscale before clipping
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            run += float(loss.item()); nb += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        pbar.close(); del pbar          # release the train iterator before eval

        vs, vy = collect_probs(model, val_loader, DEVICE, use_amp)
        m05 = multilabel_metrics(vy, (vs >= 0.5).astype(int))
        # Select the checkpoint on the number we will actually REPORT: the
        # grouped cross-fitted val micro-F1 of the best decision rule.
        sel = crossfit_rule(vs, vy, DECISION_RULES[epoch_rule],
                            groups=val_groups, metric=tgt_metric)
        history.append({"epoch": epoch, "train_loss": run / max(nb, 1),
                        "val_micro_f1_05": m05["micro_f1"], "val_macro_f1_05": m05["macro_f1"],
                        "val_jaccard_05": m05["jaccard_samples"],
                        "select": round(sel, 4),
                        "minutes": round((time.time() - t_start) / 60, 1)})
        print(f"  epoch {epoch:2d}: loss={run/max(nb,1):.4f}  "
              f"val_microF1@0.5={m05['micro_f1']:.4f}  val_macroF1@0.5={m05['macro_f1']:.4f}  "
              f"select({tgt_metric},{epoch_rule},cf)={sel:.4f}")

        # Keep the top-k epochs, not just the single best: averaging their
        # PREDICTIONS is free variance reduction (the probabilities are already
        # computed each epoch for model selection).
        if snap_k > 1:
            sp = os.path.join(CKPT_DIR, f"{name}_snap{epoch}.pt")
            if len(snaps) < snap_k or sel > min(x[0] for x in snaps):
                torch.save(core.state_dict(), sp)
                snaps.append((sel, epoch, sp))
                snaps.sort(key=lambda x: -x[0])
                for _s, _e, _p in snaps[snap_k:]:
                    if os.path.exists(_p):
                        os.remove(_p)
                snaps[:] = snaps[:snap_k]

        if sel > best + 1e-5:
            best, no_improve = sel, 0
            torch.save(core.state_dict(), ckpt)
            print(f"    -> new best (select={best:.4f}), checkpoint saved")
        else:
            no_improve += 1
            if no_improve >= CONFIG["early_stopping_patience"]:
                print(f"    -> no improvement for {no_improve} epochs, early stop"); break
        if time.time() - t_start > budget_s:
            print(f"    -> wall-clock budget ({CONFIG['time_budget_min_per_encoder']} min) "
                  f"reached, stopping this encoder"); break

    if skipped:
        print(f"  [note] skipped {skipped} non-finite-loss batches")
    if not os.path.exists(ckpt):
        raise RuntimeError(f"{name}: no checkpoint written (training produced no valid epoch)")
    core.load_state_dict(_torch_load(ckpt))
    core.to(DEVICE)
    model = nn.DataParallel(core) if N_GPU > 1 else core

    def _all_probs():
        a = collect_probs(model, val_loader, DEVICE, use_amp)
        b = collect_probs(model, test_loader, DEVICE, use_amp)
        c = collect_probs(model, full_loader, DEVICE, use_amp)
        return a, b, c

    (vs, vy), (ts, ty), (fs, _) = _all_probs()
    used_snaps = [(best, "best")]
    if snap_k > 1 and len(snaps) > 1:
        # average the probability matrices over the top-k epochs
        accum = [np.array(x, dtype=np.float64) for x in (vs, ts, fs)]
        n_ok = 1
        for _sc, _ep, _pth in snaps:
            if not os.path.exists(_pth):
                continue
            try:
                core.load_state_dict(_torch_load(_pth)); core.to(DEVICE)
                model = nn.DataParallel(core) if N_GPU > 1 else core
                (a1, _), (b1, _), (c1, _) = _all_probs()
            except Exception as exc:
                print(f"    [note] snapshot epoch {_ep} unusable ({type(exc).__name__})"); continue
            for k_, arr in enumerate((a1, b1, c1)):
                accum[k_] += arr
            n_ok += 1; used_snaps.append((_sc, f"ep{_ep}"))
        vs, ts, fs = [a / n_ok for a in accum]
        print(f"  snapshot-averaged {n_ok} checkpoints "
              f"({', '.join(t for _, t in used_snaps)})")
    for _sc, _ep, _pth in snaps:                      # tidy up the extra files
        if os.path.exists(_pth):
            try: os.remove(_pth)
            except OSError: pass
    # ---- few-shot PROTOTYPE scores from this encoder's own embeddings ----
    # Free: `full_loader` already covers every turn, so one extra pass over it
    # with want_embed=True is the only cost, and the centroids are a matmul.
    if CONFIG.get("use_proto_members") and enc_cfg.get("proto", False):
        try:
            t_pr = time.time()
            _, _, EMB = collect_probs(model, full_loader, DEVICE, use_amp,
                                      want_embed=True)
            EMB = EMB / np.maximum(np.linalg.norm(EMB, axis=1, keepdims=True), 1e-9)
            _spl = df["split"].to_numpy()
            _itr = np.flatnonzero(_spl == "train")
            Ytr_full = Y_STRAT[_itr]                    # un-augmented train rows
            Etr = EMB[_itr]
            H = Etr.shape[1]
            CENT = np.zeros((len(STRATS), H), np.float32)
            NPOS = np.zeros(len(STRATS), np.int64)
            for j in range(len(STRATS)):
                pj = np.flatnonzero(Ytr_full[:, j] > 0)
                NPOS[j] = len(pj)
                if len(pj):
                    CENT[j] = Etr[pj].sum(0)
            SUMS = CENT.copy()
            CENT = CENT / np.maximum(np.linalg.norm(CENT, axis=1, keepdims=True), 1e-9)
            SC_ALL = EMB @ CENT.T                        # (n_corpus, L) cosine
            # LEAVE-ONE-OUT for the train rows the calibration is fitted on: a
            # turn contributes to its own label's centroid, and with 2 positives
            # that is half the centroid, which would make the train scores wildly
            # optimistic and the calibration useless off-train.
            SC_TR = np.array(SC_ALL[_itr], copy=True)
            for j in range(len(STRATS)):
                pj = np.flatnonzero(Ytr_full[:, j] > 0)
                if len(pj) < 2:
                    continue
                loo = SUMS[j][None, :] - Etr[pj]
                loo = loo / np.maximum(np.linalg.norm(loo, axis=1, keepdims=True), 1e-9)
                SC_TR[pj, j] = np.einsum("ij,ij->i", Etr[pj], loo)
            PROTO_RAW[name] = {
                "train": SC_TR.astype(np.float32),
                "train_y": Ytr_full.astype(np.float32),
                "val":  SC_ALL[_spl == "val"].astype(np.float32),
                "test": SC_ALL[_spl == "test"].astype(np.float32),
                "full": SC_ALL.astype(np.float32),
                "hf_id": hf_id}
            print("  prototype scores stored (%d-d embeddings, %d labels with >=1 "
                  "train positive, %.1f min)"
                  % (H, int((NPOS > 0).sum()), (time.time() - t_pr) / 60))
            del EMB, Etr, SC_ALL, SC_TR, SUMS, CENT
            gc.collect()
        except Exception as exc:
            print("  [note] prototype scores unavailable (%s: %s)"
                  % (type(exc).__name__, exc))

    res = package_result(
        name, hf_id,
        {"val_s": vs, "test_s": ts, "full_s": fs, "val_y": vy, "test_y": ty},
        extra={"history": history, "epochs_run": len(history),
               "minutes": round((time.time() - t_start) / 60, 1),
               "use_roles": bool(use_roles), "context_turns": ctxw,
               "max_length": maxlen, "seed": eseed, "kind": "encoder",
               "ckpt": ckpt, "tokenizer_id": hf_id})
    print(f"  ({res['minutes']:.1f} min, {len(history)} epochs)")
    if CONFIG.get("delete_ckpt_after_use"):
        # The full-corpus probabilities are already in `res`, so nothing
        # downstream needs the weights. Nine members at ~0.5-1.4 GB each will
        # otherwise fill /kaggle/working.
        try:
            os.remove(ckpt); res["ckpt"] = None
        except OSError:
            pass

    del model, core, opt, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return res



## 8. Run all encoders

Each encoder is isolated: an OOM triggers one automatic retry at half batch with
gradient checkpointing, any other failure (a model id that will not download, a
missing SentencePiece) is caught, reported and skipped. The run continues with
whatever trained.

Nine encoders are configured, ordered by expected value per minute, and
`time_budget_min_total` (default 420 min) stops the loop when the session budget
is spent -- so what gets dropped is dropped from the cheap end of the list, and
the ensemble, the stacker and Phase 4 always get their (few) minutes. Rough
costs from the last run, at `num_epochs = 16` with early stopping: ~20 min for a
`ctx0` base encoder, ~25 min at `ctx1`, ~35 min at `ctx5`, ~30 min for
DeBERTa-v3, ~55 min for roberta-large.

The last line ranks the members by **cross-fitted validation** micro-F1, which
is the number that decides everything downstream; test scores are printed beside
it purely for the record.


In [ ]:
# =============================================================
# 8. RUN ALL ENCODERS
# =============================================================
def _is_oom(exc):
    # torch.cuda.OutOfMemoryError only exists on newer torch, so match on the
    # message too -- a missing attribute must not turn an OOM into a hard crash.
    return "out of memory" in str(exc).lower() or type(exc).__name__ == "OutOfMemoryError"

def _is_amp_dtype_issue(exc):
    # GradScaler.unscale_() raises this ValueError when it finds a half-
    # precision gradient, which happens if `from_pretrained` ever loads a
    # checkpoint's master weights in fp16 (the SpeakerAwareMultiLabel fix
    # forces .float() to prevent this, but this is a second line of defence
    # for any checkpoint/transformers-version combo we have not seen yet).
    msg = str(exc).lower()
    return "unscale" in msg and "fp16" in msg

results, failed, skipped_encoders, PROTO_RAW = {}, {}, [], {}
_loop_t0 = time.time()
_total_budget_s = CONFIG.get("time_budget_min_total", 10 ** 6) * 60
for _i_enc, enc_cfg in enumerate(CONFIG["encoders"]):
    nm = enc_cfg["name"]
    if not enc_cfg.get("enabled", True):
        skipped_encoders.append((nm, "disabled in CONFIG")); continue
    _spent = time.time() - _loop_t0
    if results and _spent > _total_budget_s:
        # Ordered list -> whatever is left is the least valuable. Stopping here
        # protects the ensemble / stacker / Phase 4, which are cheap but must run.
        for _c in CONFIG["encoders"][_i_enc:]:
            skipped_encoders.append((_c["name"], "session budget spent"))
        print(f"\n[BUDGET] {_spent/60:.0f} min of {CONFIG['time_budget_min_total']} min "
              f"spent on encoders -- skipping the remaining "
              f"{len(CONFIG['encoders']) - _i_enc}: "
              f"{[c['name'] for c in CONFIG['encoders'][_i_enc:]]}\n")
        break
    try:
        results[nm] = train_one_encoder(enc_cfg)
    except Exception as exc:
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        if _is_oom(exc):
            half = max(4, enc_cfg.get("batch_size", CONFIG["batch_size"]) // 2)
            print(f"\n[OOM] {nm}: retrying once at batch={half} with gradient checkpointing\n")
            try:
                results[nm] = train_one_encoder(
                    enc_cfg, overrides={"batch_size": half,
                                        "eval_batch_size": max(8, CONFIG["eval_batch_size"] // 4),
                                        "grad_checkpointing": True})
                continue
            except Exception as exc2:
                exc = exc2
        elif _is_amp_dtype_issue(exc):
            print(f"\n[AMP] {nm}: fp16/GradScaler mismatch -- retrying once with AMP off\n")
            try:
                results[nm] = train_one_encoder(enc_cfg, overrides={"use_amp": False})
                continue
            except Exception as exc2:
                exc = exc2
        failed[nm] = f"{type(exc).__name__}: {exc}"
        print(f"\n[SKIPPED] {nm}: {failed[nm]}\n")
        gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\ntrained:", list(results) or "(none)")
print("encoder wall clock: %.0f min" % ((time.time() - _loop_t0) / 60))
if failed:
    print("failed  :", json.dumps(failed, indent=2))
if skipped_encoders:
    print("skipped :", json.dumps(dict(skipped_encoders), indent=2))
assert results, "no encoder trained successfully - check internet access and the model ids"
ENCODER_NAMES = list(results)

_rank = sorted(results.items(), key=lambda kv: -kv[1]["val_cf"])
print("\nranked by CROSS-FITTED VAL micro-F1 (the selection metric; test is only reported):")
for _n, _r in _rank:
    print(f"  {_n:<18} cf-val {_r['val_cf']:.4f}   test {_r['metrics_tuned']['micro_f1']:.4f}"
          f"   ({_r['minutes']:.0f} min)")


### 8b. Shallow (non-transformer) ensemble members -- TF-IDF -> XGBoost

Two members that are *not* transformers:

* **`tfidf-xgb`** -- word (1-2 gram) + char (3-5 gram) TF-IDF, reduced to 320
  dense components by truncated SVD, plus ten label-free turn features
  (length, digit / `$` / `%` / `?` / `!` counts, position in the dialogue,
  dialogue length), then **one XGBoost booster per strategy**. An earlier
  revision also included an annotator-id code here (+0.031 micro-F1,
  measured) -- removed for production validity: a live, turn-by-turn
  deployment never has a human-labeler ID for a new conversation, so
  conditioning on one is not a feature the deployed model could ever
  actually receive.
* **`tfidf-lr`** -- binary-relevance logistic regression straight on the sparse
  TF-IDF.

Neither is competitive alone: expect ~0.40-0.46 micro-F1 against the encoders'
~0.65. **That is not the point.** Five transformers fine-tuned on the same
10.6 k turns make *highly correlated* errors -- section 8c prints the error
correlation matrix, and RoBERTa-vs-TOD-BERT typically sits above 0.8. A
gradient-boosted bag-of-ngrams fails differently: it is very good at fixed
lexical tells (`$`, `%`, "tax deductible", "deadline", "matching gift") and
hopeless at paraphrase, so its errors land on different turns. An ensemble's
gain is a function of member *decorrelation*, not member strength, which is why
a 0.44 member with r = 0.5 to the pack can be worth more than a sixth 0.65
member with r = 0.9.

They also give the level-2 stacker in 8c something a blend cannot use: a per
*label* view of when to trust lexical evidence over semantic evidence
(`source_citation` and `evidence_and_statistics` are near-keyword classes;
`empathy_and_perspective_taking` is not).

These members use the **context-free (`mt0`) text**, because the section-8e
ablation found dialogue context actively hurts a bag-of-words model on this
corpus -- 29 of 33 evaluable strategies got worse when it was included.

The XGBoost run here also replaces the old `ClassifierChain(XGBoost)` baseline,
which cost 61 minutes for micro-F1 0.29. One booster per label on dense SVD
features costs ~2-4 minutes and scores far better, so it is reported as the
strong classical baseline in section 10 as well as serving as a member.


In [ ]:
# =============================================================
# 8b. SHALLOW ENSEMBLE MEMBERS  (TF-IDF -> XGBoost / logistic regression)
# =============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack as sp_hstack

# ---- row meta, shared with the level-2 stacker in 8c ------------------
# Built from `df` and keyed by turn_id on purpose: train_df carries augmented
# COPIES of turns (same turn_id), so a positional cumcount() on train_df would
# report the wrong position in the dialogue for every copy.
_pos_all  = df.groupby("dialogue_id").cumcount().to_numpy()
TURN_POS  = dict(zip(df["turn_id"].tolist(), _pos_all.tolist()))

def _f64(x):
    return np.asarray(pd.Series(x).to_numpy(), dtype=np.float64)

def meta_feats(frame):
    """Eight turn-level features that carry no strategy-label information.

    NOTE: this used to also emit an annotator-ID code (`ANN_CODE`) as an 11th
    feature -- a second, independent leak of the same rater-ID variable
    dropped from the model's TEXT input in Section 2 (Change Set B). Removed
    here too: a live, turn-by-turn deployment never has a human-labeler ID
    for a new conversation, and this fed both the tfidf-xgb shallow member's
    dense feature matrix and the level-2 stacker's design matrix (Section
    8c's `stack_matrix`)."""
    t   = frame["text"].astype(str)
    tid = frame["turn_id"]
    # `pos` (how many persuader turns have happened so far, cumcount) is causal --
    # legitimately known mid-conversation. `DLG_LEN`/`dl` (the dialogue's EVENTUAL
    # total turn count) and the `pos/dl` ratio derived from it were REMOVED here:
    # both encode the dialogue's future -- at live turn 3 of an eventual 10-turn
    # conversation, nothing knows yet that it will run to 10. This fed both the
    # tfidf-xgb shallow member and (worse) the level-2 stacker, so a turn-by-turn
    # deployment could not reproduce this feature honestly. Caught by reasoning
    # through what a live, single-turn-at-a-time deployment can and cannot see.
    pos = _f64(tid.map(TURN_POS).fillna(0))
    return np.column_stack([
        _f64(t.str.split().str.len().fillna(0)),
        _f64(t.str.len()),
        _f64(t.str.count(r"[0-9]")),
        _f64(t.str.count(r"[$]")),
        _f64(t.str.count("%")),
        _f64(t.str.count(r"[?]")),
        _f64(t.str.count("!")),
        pos,
    ]).astype(np.float32)

# ---- XGBoost availability + device, resolved once ---------------------
try:
    from xgboost import XGBClassifier
    import xgboost as _xgb_mod
    XGB_OK = True
    print("xgboost", _xgb_mod.__version__)
except Exception as _e:
    XGB_OK = False
    print("xgboost unavailable (%s) -> the XGBoost member and stacker are skipped"
          % type(_e).__name__)

XGB_DEVICE = None
if XGB_OK and CONFIG["shallow_use_gpu"] and DEVICE == "cuda":
    try:                                    # xgboost >= 2.0 API
        XGBClassifier(n_estimators=2, tree_method="hist", device="cuda",
                      verbosity=0).fit(np.zeros((10, 2), np.float32),
                                       np.array([0, 1] * 5))
        XGB_DEVICE = "cuda"
    except Exception as _e:
        print("  xgboost GPU path unavailable (%s) -> CPU hist" % type(_e).__name__)
print("xgboost device:", XGB_DEVICE or "cpu")

def xgb_clf(params, **over):
    prm = dict(params); prm.update(over)
    prm.update(objective="binary:logistic", eval_metric="logloss",
               tree_method="hist", n_jobs=-1, verbosity=0, random_state=SEED)
    if XGB_DEVICE:
        prm["device"] = XGB_DEVICE
    return XGBClassifier(**prm)

# NOTE: `cat_from_strat` (category probability = max over a category's own
# strategies, fed to the now-deleted hierarchical gate via each shallow
# member's `val_c`/`test_c`/`full_c` slots) was removed along with the
# category head -- strategy is the only ask.
SHALLOW_NAMES = []
if not CONFIG["use_shallow_members"]:
    print("\nshallow members disabled in CONFIG (use_shallow_members=False)")
else:
    _t0 = time.time()
    tcol = "mt0" if "mt0" in df.columns else "model_text"
    print("\nfeature text column: %s (context-free view -- context hurts a "
          "bag-of-ngrams here)" % tcol)
    _nfeat = 3000 if CONFIG["FAST_DEV_RUN"] else 40000
    wv  = TfidfVectorizer(max_features=_nfeat, ngram_range=(1, 2), min_df=2,
                          sublinear_tf=True)
    cvz = TfidfVectorizer(max_features=_nfeat, ngram_range=(3, 5),
                          analyzer="char_wb", min_df=3)
    _Wtr = wv.fit_transform(train_df[tcol]); _Ctr = cvz.fit_transform(train_df[tcol])
    SPARSE = {"train": sp_hstack([_Wtr, _Ctr]).tocsr()}
    for _nm, _fr in (("val", val_df), ("test", test_df), ("full", df)):
        SPARSE[_nm] = sp_hstack([wv.transform(_fr[tcol]),
                                 cvz.transform(_fr[tcol])]).tocsr()
    print("  TF-IDF: %s (word %d + char %d)"
          % (SPARSE["train"].shape, _Wtr.shape[1], _Ctr.shape[1]))

    _k = int(min(CONFIG["shallow_svd_dim"], min(SPARSE["train"].shape) - 1))
    _svd = TruncatedSVD(n_components=max(2, _k), random_state=SEED)
    DENSE = {"train": np.hstack([_svd.fit_transform(SPARSE["train"]),
                                 meta_feats(train_df)]).astype(np.float32)}
    for _nm, _fr in (("val", val_df), ("test", test_df), ("full", df)):
        DENSE[_nm] = np.hstack([_svd.transform(SPARSE[_nm]),
                                meta_feats(_fr)]).astype(np.float32)
    print("  SVD %d comps (%.1f%% of TF-IDF variance) + 10 meta -> %d features"
          % (_svd.n_components, 100 * _svd.explained_variance_ratio_.sum(),
             DENSE["train"].shape[1]))

    _SH_SPLITS = (("val", val_df), ("test", test_df), ("full", df))

    def fit_per_label(fit_predict, tag):
        """Fit one binary model per strategy; a label with too few train
        positives keeps its train prior instead of a fitted model (nothing to
        learn, and a booster on 3 positives is a random-number generator)."""
        out = {nm: np.zeros((len(fr), len(STRATS)), np.float32) for nm, fr in _SH_SPLITS}
        n_fit, n_prior = 0, []
        for j, sname in enumerate(STRATS):
            yj = Y_STRAT_TR[:, j]
            npos = int(yj.sum())
            if npos < CONFIG["shallow_xgb_min_pos"] or npos == len(yj):
                prior = max(npos, 0.5) / max(len(yj), 1)
                for nm, _fr in _SH_SPLITS:
                    out[nm][:, j] = prior
                n_prior.append(sname); continue
            preds = fit_predict(yj)
            for nm, _fr in _SH_SPLITS:
                out[nm][:, j] = preds[nm]
            n_fit += 1
            if n_fit % 10 == 0:
                print("    [%s] %d/%d labels fitted (%.1f min)"
                      % (tag, n_fit, len(STRATS), (time.time() - _t0) / 60))
        if n_prior:
            print("    [%s] %d label(s) left at their train prior: %s"
                  % (tag, len(n_prior), n_prior))
        return out

    # ---------------- member 1: XGBoost on SVD + meta ------------------
    if XGB_OK:
        def _xgb_one(yj):
            clf = xgb_clf(CONFIG["shallow_xgb_params"])
            clf.fit(DENSE["train"], yj)
            return {nm: clf.predict_proba(DENSE[nm])[:, 1] for nm, _fr in _SH_SPLITS}
        _t1 = time.time()
        _P = fit_per_label(_xgb_one, "xgb")
        print("  tfidf-xgb fitted in %.1f min" % ((time.time() - _t1) / 60))
        results["tfidf-xgb"] = package_result(
            "tfidf-xgb", "tfidf+svd%d -> XGBoost x%d labels" % (_svd.n_components, len(STRATS)),
            {"val_s": _P["val"], "test_s": _P["test"], "full_s": _P["full"],
             "val_y": Y_STRAT_VA, "test_y": Y_STRAT_TE},
            extra={"kind": "shallow", "minutes": round((time.time() - _t1) / 60, 1)})
        SHALLOW_NAMES.append("tfidf-xgb")

    # ---------------- member 2: BR logistic on sparse TF-IDF -----------
    def _lr_one(yj):
        clf = LogisticRegression(max_iter=400, C=4.0, class_weight="balanced")
        clf.fit(SPARSE["train"], yj)
        return {nm: clf.predict_proba(SPARSE[nm])[:, 1] for nm, _fr in _SH_SPLITS}
    _t1 = time.time()
    _P = fit_per_label(_lr_one, "lr")
    print("  tfidf-lr fitted in %.1f min" % ((time.time() - _t1) / 60))
    results["tfidf-lr"] = package_result(
        "tfidf-lr", "tfidf(word+char) -> binary-relevance logistic",
        {"val_s": _P["val"], "test_s": _P["test"], "full_s": _P["full"],
         "val_y": Y_STRAT_VA, "test_y": Y_STRAT_TE},
        extra={"kind": "shallow", "minutes": round((time.time() - _t1) / 60, 1)})
    SHALLOW_NAMES.append("tfidf-lr")

    # ---------- shared calibration for every similarity-style member ----
    # These use NO training labels. Each strategy gets a document built from its
    # taxonomy definition plus its example cue sentences, and every turn is
    # scored by similarity to those 41 documents.
    #
    # Aimed squarely at the macro-F1 tail. `deadline_pressure` has 2 training
    # turns, so every fine-tuned encoder predicts it never and scores F1 =
    # 0.000; but its taxonomy entry describes exactly what it looks like, and a
    # similarity model needs no training examples to use that. Thirteen of the
    # 41 strategies have under 30 training turns and together carry 13/41 of the
    # macro score, so moving them off zero is worth more than any further gain
    # on `rapport_building`.
    #
    # The raw similarity is turned into a probability per label by a 2-feature
    # logistic fit on the TRAIN split (own similarity, and own minus the row
    # mean -- a competition term). Train is a different split from val/test, so
    # this calibration leaks nothing into the reported numbers. A label with
    # fewer than 2 train positives cannot fit even that, and falls back to a
    # monotone rescaling anchored at the label's prevalence.
    def calibrate_sim(S, tag, y_train=None):
        """Turn a raw similarity into a per-label probability.

        S: {"train","val","test","full"} -> (n, 41) similarity matrices.
        y_train must line up with S["train"] -- which is NOT always Y_STRAT_TR:
        the encoder prototypes score only the real train rows, while the TF-IDF
        and cue members score the augmented frame."""
        Ytr_use = Y_STRAT_TR if y_train is None else np.asarray(y_train)
        assert S["train"].shape[0] == Ytr_use.shape[0], (
            "%s: train scores %d rows vs labels %d rows"
            % (tag, S["train"].shape[0], Ytr_use.shape[0]))
        out = {nm: np.zeros((S[nm].shape[0], len(STRATS)), np.float32)
               for nm in ("val", "test", "full")}
        n_fit = 0
        for j in range(len(STRATS)):
            yj = Ytr_use[:, j]
            def feats(M):
                own = M[:, j:j + 1].astype(np.float64)
                return np.hstack([own, own - M.mean(1, keepdims=True)])
            ok = False
            if 2 <= yj.sum() < len(yj):
                try:
                    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
                    lr.fit(feats(S["train"]), yj)
                    for nm in ("val", "test", "full"):
                        out[nm][:, j] = lr.predict_proba(feats(S[nm]))[:, 1]
                    ok = True; n_fit += 1
                except Exception:
                    ok = False
            if not ok:
                for nm in ("val", "test", "full"):
                    c = S[nm][:, j].astype(np.float64)
                    rg = max(float(c.max() - c.min()), 1e-9)
                    out[nm][:, j] = np.clip(2.0 * PREV_TRAIN[j] * (c - c.min()) / rg, 0, 1)
        print("    [%s] logistic calibration fitted for %d/%d labels "
              "(rest use the prevalence-anchored fallback)" % (tag, n_fit, len(STRATS)))
        return out

    # ---------- members: TAXONOMY CUE similarity (zero-shot) -----------
    def register_cue_member(name, hf, S, tag):
        P = calibrate_sim(S, tag)
        results[name] = package_result(
            name, hf,
            {"val_s": P["val"], "test_s": P["test"], "full_s": P["full"],
             "val_y": Y_STRAT_VA, "test_y": Y_STRAT_TE},
            extra={"kind": "cue"})
        SHALLOW_NAMES.append(name)

    # ---------- members: FEW-SHOT PROTOTYPES ---------------------------
    # Cosine to the centroid of the (few) training turns carrying each strategy.
    # This is the right estimator in the few-shot regime: with 5 positives a
    # sigmoid head sees a 1:1500 imbalance and learns to always say no, while
    # "closest to the mean of these five" needs no imbalance handling at all.
    # Two flavours: the trained encoders' own embedding spaces (semantic, from
    # PROTO_RAW) and a TF-IDF space (lexical). Calibrated per label on TRAIN,
    # using leave-one-out centroids there so a turn is never scored against a
    # centroid it is part of.
    def register_proto(name, hf, S, tag, y_train=None):
        P = calibrate_sim(S, tag, y_train)
        results[name] = package_result(
            name, hf,
            {"val_s": P["val"], "test_s": P["test"], "full_s": P["full"],
             "val_y": Y_STRAT_VA, "test_y": Y_STRAT_TE},
            extra={"kind": "proto"})
        SHALLOW_NAMES.append(name)

    if not CONFIG.get("use_proto_members"):
        print("\nprototype members disabled in CONFIG")
    else:
        # (a) TF-IDF prototype -- measured offline at 0.106 macro over the 25
        #     starved labels, against 0.000 for every v4 transformer.
        try:
            from sklearn.preprocessing import normalize as _l2p
            _t1 = time.time()
            # REAL train rows only. Including the 8 augmented copies of a
            # 2-example strategy would make that turn's own copies most of its
            # centroid, and the calibration fitted on those scores would not
            # transfer to a val turn that is in nobody's centroid.
            _msk = (~train_df["_augmented"].to_numpy()
                    if "_augmented" in train_df else np.ones(len(train_df), bool))
            _Ntr = _l2p(SPARSE["train"][_msk])
            _Yr  = Y_STRAT_TR[_msk]
            _SUMS = np.vstack([
                np.asarray(_Ntr[np.flatnonzero(_Yr[:, j] > 0)].sum(0)).ravel()
                if _Yr[:, j].sum() else np.zeros(_Ntr.shape[1])
                for j in range(len(STRATS))]).astype(np.float32)
            _CENT = _l2p(_SUMS)
            _S = {nm: np.asarray(_l2p(SPARSE[nm]) @ _CENT.T, dtype=np.float32)
                  for nm in ("val", "test", "full")}
            # leave-one-out for the train rows the calibration is fitted on
            _Etr = np.asarray(_Ntr.todense(), dtype=np.float32)
            _Str = _Etr @ _CENT.T
            for j in range(len(STRATS)):
                pj = np.flatnonzero(_Yr[:, j] > 0)
                if len(pj) < 2:
                    continue
                loo = _SUMS[j][None, :] - _Etr[pj]
                loo = loo / np.maximum(np.linalg.norm(loo, axis=1, keepdims=True), 1e-9)
                _Str[pj, j] = np.einsum("ij,ij->i", _Etr[pj], loo)
            _S["train"] = _Str.astype(np.float32)
            print("\n  tfidf prototype cosine: mean %.3f, max %.3f (centroids from "
                  "%d real train turns, leave-one-out on train)"
                  % (_S["test"].mean(), _S["test"].max(), int(_msk.sum())))
            register_proto("tfidf-proto", "tfidf centroid per category", _S,
                           "tfidf-proto", y_train=_Yr)
            print("  (%.1f min)" % ((time.time() - _t1) / 60))
            del _S, _CENT, _Ntr, _SUMS, _Etr, _Str
            gc.collect()
        except Exception as exc:
            print("  tfidf-proto skipped: %s: %s" % (type(exc).__name__, exc))

        # (b) one prototype member per encoder that stored embeddings
        for _en, _pr in sorted(PROTO_RAW.items()):
            try:
                _S = {"train": _pr["train"], "val": _pr["val"],
                      "test": _pr["test"], "full": _pr["full"]}
                register_proto("%s-proto" % _en,
                               "%s embeddings -> centroid per category" % _pr["hf_id"],
                               _S, "%s-proto" % _en, y_train=_pr["train_y"])
            except Exception as exc:
                print("  %s-proto skipped: %s: %s" % (_en, type(exc).__name__, exc))
        if not PROTO_RAW:
            print("  (no encoder stored embeddings -- set \"proto\": True on an "
                  "encoder entry)")

    _cue_ok = bool(CONFIG.get("use_cue_member")) and len(CUE_DOC) >= len(STRATS) // 2
    if not _cue_ok:
        print("\ncue members skipped (use_cue_member=%s, %d cue docs)"
              % (CONFIG.get("use_cue_member"), len(CUE_DOC)))
    else:
        _docs = [CUE_DOC.get(st, st.replace("_", " ")) for st in STRATS]

        # ---- cue-tfidf: lexical overlap with the cue documents -----------
        try:
            from sklearn.preprocessing import normalize as _l2
            _t1 = time.time()
            _C = _l2(sp_hstack([wv.transform(_docs), cvz.transform(_docs)]).tocsr())
            _S = {nm: np.asarray((_l2(SPARSE[nm]) @ _C.T).todense(), dtype=np.float32)
                  for nm in ("train", "val", "test", "full")}
            print("\n  cue-tfidf similarity: %s, mean cos %.3f, max %.3f"
                  % (_S["test"].shape, _S["test"].mean(), _S["test"].max()))
            register_cue_member("cue-tfidf", "taxonomy cues -> tfidf cosine", _S, "cue-tfidf")
            print("  (%.1f min)" % ((time.time() - _t1) / 60))
            del _S
        except Exception as exc:
            print("  cue-tfidf skipped: %s: %s" % (type(exc).__name__, exc))

        # ---- cue-emb: SEMANTIC similarity with a sentence encoder --------
        # The cues are paraphrases, not quotations ("the campaign closes
        # tonight" vs "only a few hours left"), so lexical overlap misses most
        # of them. A small sentence-embedding model costs ~1 min for the whole
        # corpus and matches meaning instead of tokens. Nothing is fine-tuned:
        # this member is pure zero-shot.
        _emb_id = CONFIG.get("cue_embed_model")
        if not _emb_id:
            print("  cue-emb skipped (cue_embed_model is empty)")
        else:
            try:
                _t1 = time.time()
                _etok = AutoTokenizer.from_pretrained(_emb_id)
                _enc  = AutoModel.from_pretrained(_emb_id).to(DEVICE).eval().float()

                @torch.no_grad()
                def _embed(texts, bs=128, maxlen=128):
                    acc = []
                    for i in range(0, len(texts), bs):
                        b = _etok([str(x) for x in texts[i:i + bs]], padding=True,
                                  truncation=True, max_length=maxlen,
                                  return_tensors="pt")
                        b = {k: v.to(DEVICE) for k, v in b.items()}
                        h = _enc(**b).last_hidden_state
                        m = b["attention_mask"].unsqueeze(-1).float()
                        v = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
                        acc.append(nn.functional.normalize(v, dim=-1).cpu().numpy())
                    return np.vstack(acc).astype(np.float32)

                _E = _embed(_docs)
                _S = {}
                for nm, fr in (("train", train_df), ("val", val_df),
                               ("test", test_df), ("full", df)):
                    _S[nm] = (_embed(fr["text"].tolist()) @ _E.T).astype(np.float32)
                print("\n  cue-emb (%s): mean cos %.3f, max %.3f"
                      % (_emb_id, _S["test"].mean(), _S["test"].max()))
                register_cue_member("cue-emb", "taxonomy cues -> %s cosine" % _emb_id,
                                    _S, "cue-emb")
                print("  (%.1f min)" % ((time.time() - _t1) / 60))
                del _S, _E, _enc, _etok
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception as exc:
                print("  cue-emb skipped: %s: %s" % (type(exc).__name__, exc))

    del _P
    gc.collect()
    print("\nnon-transformer members ready: %s  (%.1f min total)"
          % (SHALLOW_NAMES, (time.time() - _t0) / 60))


### 8c. Ensemble / stacking zoo -- every combiner scored the same way

The previous revision committed to one combiner: greedy forward selection with
replacement on the probability average. It worked (0.6784 against a best single
model of 0.6532), but "greedy on the arithmetic mean of raw sigmoids" is one
point in a space of combiners, and which point wins is an empirical question
about *this* corpus. So the notebook now builds a **zoo** and scores every
member of it under exactly the protocol single models are scored under --
grouped-by-dialogue cross-fitted validation micro-F1 -- and keeps the winner.

**The axes**

| axis | options | why it can matter |
|---|---|---|
| space | arithmetic mean of probabilities / mean of **log-odds** | the log-odds mean is a weighted *geometric* mean of the odds, the right average when members are independent evidence rather than noisy copies |
| calibration | raw / per-member **Platt** scale+shift on the logit | `tfidf-lr` (trained `class_weight="balanced"`) and `roberta-large` are not on the same confidence scale; an unweighted mean of the two is dominated by whichever is louder |
| weights | equal / greedy-with-replacement / **bagged** greedy | greedy on one val split is a high-variance selector; bagging it over 24 dialogue subsamples with random member subsets is the standard cure (Caruana et al. 2004) |
| power | generalised mean exponent q in {-1, -0.5, 0, 0.5, 1, 2} | q < 1 is more conservative about a single loud member, which is what a false-positive-sensitive micro-F1 usually wants |
| level 2 | linear stack / **XGBoost** stack / stack-blend mixture | a stacker can use information no blend can: see below |

**The stacker, and why it is different from the one that failed.** Note 6 of
section 19 records a stacking attempt that lost **0.10** micro-F1: 41 logistic
models over the 41 predicted probabilities, each fitted on ~1.5 k validation
rows. That failed for a specific, fixable reason -- 41 separate models times 41
free parameters on 1.5 k rows is a variance disaster. This one is built the
other way round:

* **One model, long format.** Each training example is a *(turn, strategy)*
  cell, so the val split provides `n_val x 41` ~ 65 k examples, and the
  parameters are **shared across labels**. The label's identity enters as a
  feature (its train frequency, its base log-odds, and for the tree its id), not
  as a separate model. Same information, a fortieth of the variance.
* **Features a blend structurally cannot use:** the spread of member opinions
  on that cell (mean/std/min/max of the member logits), where the label ranks
  *within its turn*, and how much total probability mass the turn carries (a
  cardinality signal). Two earlier features were removed from here: what the
  members said about the *parent category* (depended on the model's
  now-removed category head), and a leave-one-out mean of the same label's
  probability over the **rest of the whole dialogue** (non-causal -- it used
  turns *after* the one being classified, which a live, turn-by-turn
  deployment never has; this was a separate leak from the decode-time
  `dialogue_smooth` alpha, which was also removed, not merely disabled).
* **Grouped out-of-fold, always.** The stacker is trained only on the val split
  (which no member ever saw), with 5 folds grouped by dialogue. Its val score is
  computed from out-of-fold predictions, its thresholds are fitted on those same
  out-of-fold predictions, and test/full get the average of the five fold
  models. Nothing about test enters at any point.

**Guard rails.** Every candidate -- including "just use the single best member"
and "just take the equal-weight mean" -- competes on the same cross-fitted val
number, and near-ties break toward the *simpler* candidate
(single < mean < weighted < power mean < linear stack < XGB stack < mixture).
If the zoo is all noise, it selects the mean and costs nothing but a few minutes
of numpy. The candidate table is written to `ensemble_candidates.csv` with both
the val score that made the decision and the test score that did not.


In [ ]:
# =============================================================
# 8c. ENSEMBLE / STACKING ZOO
# =============================================================
ENSEMBLE_NAME = "ensemble"
MEMBERS = [n for n in results if n != ENSEMBLE_NAME and "raw" in results[n]]
ens_table, STACK_RAN = [], False

def _logit(p, c=1e-6):
    p = np.clip(np.asarray(p, dtype=np.float64), c, 1.0 - c)
    return np.log(p / (1.0 - p))

def _sig(x):
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(x, dtype=np.float64), -30.0, 30.0)))

if not (CONFIG["use_ensemble"] and len(MEMBERS) > 1):
    print("ensemble skipped (needs >= 2 members with stored probabilities); "
          "members = %s" % MEMBERS)
else:
    vy = results[MEMBERS[0]]["raw"]["val_y"]
    ty = results[MEMBERS[0]]["raw"]["test_y"]
    vgrp = val_df["dialogue_id"].to_numpy()
    tgrp = test_df["dialogue_id"].to_numpy()
    fgrp = df["dialogue_id"].to_numpy()
    LO, HI = CONFIG["threshold_floor"], CONFIG["threshold_ceil"]
    L = len(STRATS)
    TGT = CONFIG.get("target_metric", "micro")
    # Coarse threshold grid for the INNER hillclimb only (the final decode still
    # uses the full 0.01 grid). Under macro the hillclimb has to re-fit 41
    # thresholds per candidate evaluation instead of one, so the grid is what
    # keeps 20-odd bagged searches inside a couple of minutes.
    _GG = np.round(np.arange(LO, HI + 1e-9,
                             CONFIG.get("ensemble_greedy_grid_step", 0.05)), 4)
    # Label tiers by TRAIN support. macro-F1 gives a 2-example strategy the same
    # weight as a 2,000-example one, and the member that is best on each is NOT
    # the same model -- a zero-shot cue member can only ever help the starved
    # tail, and would be voted down by any weighting fitted on all 41 at once.
    _SUP = Y_STRAT_TR.sum(0)
    TIER_EDGES = list(CONFIG.get("ensemble_tier_edges", [60, 250]))
    TIER_OF = np.digitize(_SUP, TIER_EDGES)
    TIER_COLS = [np.flatnonzero(TIER_OF == t) for t in range(len(TIER_EDGES) + 1)]
    print("members (%d): %s" % (len(MEMBERS), MEMBERS))
    print("target metric: %s | label support tiers (train): %s"
          % (TGT, {("<%d" % TIER_EDGES[0] if t == 0 else
                    (">=%d" % TIER_EDGES[-1] if t == len(TIER_EDGES) else
                     "%d-%d" % (TIER_EDGES[t - 1], TIER_EDGES[t]))): len(c)
                   for t, c in enumerate(TIER_COLS)}))

    # ---------- 0. are the members actually decorrelated? --------------
    _err = {}
    for m in MEMBERS:
        _err[m] = (apply_pred(results[m]["val_prob"], np.array(results[m]["thresholds"]),
                              results[m].get("force_top1", False)) != vy).ravel().astype(np.float32)
    _E = pd.DataFrame(_err).corr()
    _E.to_csv(os.path.join(CONFIG["out_dir"], "member_error_correlation.csv"))
    print("\nerror correlation between members (val, each at its own operating point):")
    display(_E.round(2))
    _pairs = sorted([(a, b, float(_E.loc[a, b]))
                     for i, a in enumerate(MEMBERS) for b in MEMBERS[i + 1:]],
                    key=lambda x: x[2])
    print("least correlated pairs (what the ensemble is actually paid for):")
    for a, b, c in _pairs[:5]:
        print("   %-18s %-18s r = %.3f" % (a, b, c))
    print("most correlated pairs (near-duplicates -- weighting them twice buys little):")
    for a, b, c in _pairs[-3:]:
        print("   %-18s %-18s r = %.3f" % (a, b, c))
    del _err
    gc.collect()

    # ---------- 1. member views: raw, or Platt-calibrated on val -------
    def platt(pv, y):
        """Two-parameter (scale, shift) logistic recalibration of the logit,
        fitted on val by BCE. It is fitted on the same val split the
        cross-fitted search then scores on: with ~65k cells and 2 parameters the
        optimism is of order 1e-4, but it is not exactly zero, which is one more
        reason the number this notebook reports is the TEST one."""
        z = _logit(pv).ravel(); t = np.asarray(y, dtype=np.float64).ravel()
        def nll(ab):
            u = np.clip(ab[0] * z + ab[1], -30.0, 30.0)
            return float(np.mean(np.logaddexp(0.0, u) - t * u))
        try:
            from scipy.optimize import minimize
            r = minimize(nll, np.array([1.0, 0.0]), method="L-BFGS-B",
                         bounds=[(0.05, 20.0), (-12.0, 12.0)])
            a, b = float(r.x[0]), float(r.x[1])
            return (a, b) if np.isfinite(a) and np.isfinite(b) else (1.0, 0.0)
        except Exception as exc:
            print("  [note] Platt fit failed (%s) -> identity" % type(exc).__name__)
            return 1.0, 0.0

    _VIEWS = {}
    def views(cal):
        # NOTE: this used to also Platt-calibrate a `_c` (category) view per
        # member, for the hierarchical gate. Removed along with the category
        # head -- every RAW_KEYS entry is now a strategy ("_s") key, so the
        # per-key branch collapses to one case.
        cal = bool(cal)
        if cal in _VIEWS:
            return _VIEWS[cal]
        out = {}
        for m in MEMBERS:
            R = results[m]["raw"]
            if not cal:
                out[m] = {k: np.asarray(R[k], dtype=np.float32) for k in RAW_KEYS}
            else:
                a, b = platt(R["val_s"], vy)
                out[m] = {k: _sig(a * _logit(R[k]) + b).astype(np.float32)
                          for k in RAW_KEYS}
        _VIEWS[cal] = out
        return out

    if True in [bool(c) for c in CONFIG["ensemble_calibrate"]]:
        views(True)
        print("\nPlatt calibration (scale, shift) per member, fitted on val:")
        for m in MEMBERS:
            a, b = platt(results[m]["raw"]["val_s"], vy)
            print("   %-18s scale %.3f  shift %+.3f%s"
                  % (m, a, b, "   <- squashed hard" if a < 0.6 else ""))

    # ---------- 2. combination operators -------------------------------
    def _fwd(P, space, q):
        if space == "logit": return _logit(P)
        if q == 1.0:         return np.asarray(P, dtype=np.float64)
        if q == 0.0:         return np.log(np.clip(P, 1e-6, 1.0))
        return np.power(np.clip(P, 1e-6, 1.0), q)

    def _inv(A, space, q):
        if space == "logit": return _sig(A)
        if q == 1.0:         return A
        if q == 0.0:         return np.exp(A)
        return np.power(np.clip(A, 1e-12, None), 1.0 / q)

    def combine(vw, W, space="prob", q=1.0):
        """Weighted mean of the member matrices in `space`. The weights sum to
        1, so every operator here returns a probability in [0, 1] and the decode
        machinery downstream needs no special case."""
        out = {}
        for k in RAW_KEYS:
            acc = None
            for m, w in W.items():
                a = w * _fwd(vw[m][k], space, q)
                acc = a if acc is None else acc + a
            out[k] = _inv(acc, space, q).astype(np.float32)
        return out

    OBJECTIVES = ([m for m in ("micro", "macro")]
                  if CONFIG.get("report_both_operating_points", True) else [TGT])

    def hill_score(prob, yv, metric):
        """In-bag score for the hillclimb, in `metric`.

        micro: the exact best single global threshold, O(N log N).
        macro: the exact best PER-LABEL thresholds on the coarse grid -- which
        is the right thing, because macro-F1 is the mean of per-label F1s and a
        single global cut simply never fires for the rare tail, so under a
        global cut every candidate blend would score identically on the labels
        that macro cares most about."""
        if metric == "macro":
            _thr, sc = best_perlabel_macro(prob, yv, _GG)
            return sc
        _t, sc = _best_global_micro(prob, yv, LO, HI)
        return sc

    def greedy(vw, space="prob", q=1.0, rows=None, pool=None, steps=None, cols=None,
               metric=None):
        """Caruana forward selection WITH REPLACEMENT: picking a member twice
        weights it, never picking it drops it.

        `cols` restricts the fit to a subset of LABEL columns, which is how the
        per-support-tier weights below are fitted."""
        metric = metric or TGT
        pool  = list(pool or MEMBERS)
        steps = int(steps or CONFIG["ensemble_greedy_steps"])
        idx   = slice(None) if rows is None else np.asarray(rows)
        yv    = vy if rows is None else vy[idx]
        if cols is not None:
            cols = np.asarray(cols)
            yv = yv[:, cols]
        F = {m: (vw[m]["val_s"][idx] if cols is None else vw[m]["val_s"][idx][:, cols])
             for m in pool}
        F = {m: _fwd(v, space, q) for m, v in F.items()}
        acc, chosen, best = None, [], -1.0
        for _step in range(steps):
            bs, bm, bacc = -1.0, None, None
            for m in pool:
                cand = F[m] if acc is None else acc + F[m]
                f1 = hill_score(_inv(cand / (len(chosen) + 1.0), space, q), yv, metric)
                if f1 > bs:
                    bs, bm, bacc = f1, m, cand
            if bm is None or (chosen and bs <= best + 1e-6):
                break
            acc, best = bacc, bs
            chosen.append(bm)
        if not chosen:
            return {m: 1.0 / len(pool) for m in pool}, 0.0
        return {m: chosen.count(m) / len(chosen) for m in set(chosen)}, best

    def bagged(vw, space="prob", q=1.0, metric=None):
        """Average the greedy weight vector over `ensemble_bag_rounds` random
        dialogue subsamples of val, each seeing a random subset of members."""
        rng  = np.random.default_rng(SEED)
        uniq = np.unique(vgrp)
        n_dlg = max(3, int(round(CONFIG["ensemble_bag_frac"] * len(uniq))))
        n_mem = max(2, int(round(CONFIG["ensemble_bag_member_frac"] * len(MEMBERS))))
        tot, hits = {m: 0.0 for m in MEMBERS}, 0
        for _b in range(int(CONFIG["ensemble_bag_rounds"])):
            dl   = rng.choice(uniq, size=min(n_dlg, len(uniq)), replace=False)
            rows = np.flatnonzero(np.isin(vgrp, dl))
            pool = list(rng.choice(np.array(MEMBERS, dtype=object),
                                   size=min(n_mem, len(MEMBERS)), replace=False))
            if len(rows) < 20 or len(pool) < 2:
                continue
            W, _ = greedy(vw, space, q, rows=rows, pool=pool, metric=metric)
            for m, w in W.items():
                tot[m] += w
            hits += 1
        z = sum(tot.values())
        if hits == 0 or z <= 0:
            return None
        return {m: tot[m] / z for m in MEMBERS if tot[m] > 1e-9}

    FAST_RULE = {"macro": rule_perlabel_shrunk, "micro": rule_global}
    _FAST_RULE = FAST_RULE[TGT]

    def cf_fast(mats, metric=None):
        """Cheap ranking metric: grouped cross-fitted val F1 in `metric`, using
        ONE aligned rule instead of the full rule search."""
        metric = metric or TGT
        return crossfit_rule(mats["val_s"], vy, FAST_RULE[metric],
                             groups=vgrp, metric=metric)

    def combine_tiered(vw, Ws, Wglob, space="prob", q=1.0):
        """One weight vector per label-support tier, composed column-wise.

        NOTE: this used to skip the (now-removed) category matrices, which
        kept the global weighting because they fed only the hierarchical
        gate. With the category head gone every RAW_KEYS entry is a
        strategy key, so that branch is gone too."""
        base = combine(vw, Wglob, space, q)
        out = {}
        for k in RAW_KEYS:
            M = np.array(base[k], dtype=np.float32, copy=True)
            for t, cols in enumerate(TIER_COLS):
                if len(cols) == 0:
                    continue
                Wt = Ws[t] or Wglob
                acc = None
                for m, w in Wt.items():
                    a = w * _fwd(vw[m][k][:, cols], space, q)
                    acc = a if acc is None else acc + a
                M[:, cols] = _inv(acc, space, q).astype(np.float32)
            out[k] = M
        return out

    # ---------- 3. level-2 stackers ------------------------------------
    BASE_LOGIT = _logit(np.clip(Y_STRAT_TR.mean(0), 1e-4, 1 - 1e-4))

    def stack_matrix(vw, mats, frame, split, groups, with_label_id):
        """Long-format design matrix: ONE ROW PER (turn, strategy) cell.

        NOTE: this used to also include (a) a category-probability feature
        (`cat`, from each member's now-removed `_c` view) and (b)
        `dialogue_neighbor_mean(b, groups)` -- the mean probability of a
        label over the OTHER turns of the same dialogue, INCLUDING turns
        AFTER the one being classified. Both were removed: (a) along with
        the category head (Change Set A); (b) because it is a whole-dialogue
        leave-one-out feature that peeks at future turns, which a live,
        turn-by-turn deployment can never see (Change Set C). (b) was an
        independent leak from `dialogue_smooth`'s decode-time use of the
        same helper -- removing that alone would NOT have fixed this."""
        ks   = split + "_s"
        n    = mats[ks].shape[0]
        per  = np.stack([_logit(vw[m][ks]) for m in MEMBERS], axis=0)       # (M,n,L)
        b    = np.asarray(mats[ks], dtype=np.float64)
        rank = (-b).argsort(1).argsort(1).astype(np.float64)                # 0 = strongest
        mt   = meta_feats(frame)
        cols  = [x.ravel() for x in per]
        cols += [per.mean(0).ravel(), per.std(0).ravel(),
                 per.min(0).ravel(), per.max(0).ravel(),
                 _logit(b).ravel(),
                 np.tile(BASE_LOGIT, n),
                 np.repeat(b.sum(1), L), np.repeat(b.max(1), L),
                 rank.ravel()]
        if with_label_id:
            cols.append(np.tile(np.arange(L, dtype=np.float64), n))
        cols += [np.repeat(mt[:, c].astype(np.float64), L) for c in range(mt.shape[1])]
        return np.column_stack(cols).astype(np.float32)

    def run_stack(kind, vw, mats):
        """Grouped out-of-fold level-2 fit on the VAL split only."""
        nv, ndlg = mats["val_s"].shape[0], len(np.unique(vgrp))
        if nv < CONFIG["stack_min_val_rows"] or ndlg < CONFIG["stack_min_val_dialogues"]:
            return None, "val too small (%d rows / %d dialogues)" % (nv, ndlg)
        if kind == "xgb" and not XGB_OK:
            return None, "xgboost unavailable"
        with_id = (kind == "xgb")          # a numeric label id is only useful to a tree
        Xv = stack_matrix(vw, mats, val_df,  "val",  vgrp, with_id)
        Xt = stack_matrix(vw, mats, test_df, "test", tgrp, with_id)
        Xf = stack_matrix(vw, mats, df,      "full", fgrp, with_id)
        n_feat = Xv.shape[1]
        yl = np.asarray(vy).ravel()
        # MACRO alignment for the level-2 loss. In long format every label
        # already contributes the same number of ROWS, but a positive of
        # `rapport_building` is 1,000x more common than a positive of
        # `deadline_pressure`, so an unweighted logloss learns the frequent
        # labels and ignores the tail -- which is exactly backwards for a metric
        # that scores all 41 equally. Positives are therefore weighted by
        # 1/prevalence (capped), so each label's positive class carries
        # comparable total mass.
        if TGT == "macro" and CONFIG.get("stack_macro_weight", True):
            _pl = np.tile(PREV_TRAIN, mats["val_s"].shape[0])
            wl = np.where(yl > 0, np.minimum(1.0 / np.maximum(_pl, 1e-4),
                                             float(CONFIG.get("stack_pos_weight_cap", 60.0))), 1.0)
            wl = wl / wl.mean()
        else:
            wl = None
        folds = grouped_folds(vgrp, CONFIG["stack_folds"])
        oof = np.zeros(nv * L); te = np.zeros(Xt.shape[0]); fu = np.zeros(Xf.shape[0])
        def _ex(rows):
            return (np.asarray(rows)[:, None] * L + np.arange(L)[None, :]).ravel()
        used = 0
        for k in range(len(folds)):
            tr_rows = np.concatenate([folds[j] for j in range(len(folds)) if j != k])
            te_rows = folds[k]
            if len(te_rows) == 0 or len(tr_rows) == 0:
                continue
            itr, ite = _ex(tr_rows), _ex(te_rows)
            if yl[itr].sum() < 5 or yl[itr].sum() == len(itr):
                continue
            if kind == "xgb":
                mdl = xgb_clf(CONFIG["stack_xgb_params"])
                mdl.fit(Xv[itr], yl[itr],
                        sample_weight=(None if wl is None else wl[itr]))
                oof[ite] = mdl.predict_proba(Xv[ite])[:, 1]
                te += mdl.predict_proba(Xt)[:, 1]
                fu += mdl.predict_proba(Xf)[:, 1]
            else:
                mu = Xv[itr].mean(0); sd = Xv[itr].std(0) + 1e-6
                mdl = LogisticRegression(max_iter=2000, C=1.0)
                mdl.fit((Xv[itr] - mu) / sd, yl[itr],
                        sample_weight=(None if wl is None else wl[itr]))
                oof[ite] = mdl.predict_proba((Xv[ite] - mu) / sd)[:, 1]
                te += mdl.predict_proba((Xt - mu) / sd)[:, 1]
                fu += mdl.predict_proba((Xf - mu) / sd)[:, 1]
            used += 1
        del Xv, Xt, Xf
        gc.collect()
        if used == 0:
            return None, "no usable fold"
        return ({"val_s":  oof.reshape(nv, L).astype(np.float32),
                 "test_s": (te / used).reshape(-1, L).astype(np.float32),
                 "full_s": (fu / used).reshape(-1, L).astype(np.float32)},
                "%d folds x %d features, %d rows" % (used, n_feat, nv * L))

    # ---------- 4. build the zoo ---------------------------------------
    CAND = []
    def add(name, simp, mats, meta):
        if mats is None:
            return None
        c = {"name": name, "simplicity": simp, "mats": mats, "meta": meta}
        for _mt in OBJECTIVES:
            c["cf_" + _mt] = cf_fast(mats, _mt)
        c["cf_fast"] = c["cf_" + TGT]
        CAND.append(c)
        return c

    best_single = max(MEMBERS, key=lambda m: results[m]["val_cf"])
    add("single:" + best_single, 0,
        {k: np.asarray(results[best_single]["raw"][k], np.float32) for k in RAW_KEYS},
        {"note": "best single member by cross-fitted val"})

    mode = CONFIG["ensemble_mode"]
    spaces = ["prob"] if mode == "mean" else list(CONFIG["ensemble_spaces"])
    cals   = [False]  if mode == "mean" else [bool(c) for c in CONFIG["ensemble_calibrate"]]
    W_EQ   = {m: 1.0 / len(MEMBERS) for m in MEMBERS}
    best_w = {}
    print("\nbuilding candidates ...")
    for cal in cals:
        vw  = views(cal)
        tag = "+cal" if cal else ""
        for space in spaces:
            add("mean.%s%s" % (space, tag), 1, combine(vw, W_EQ, space), {"weights": W_EQ})
            if mode == "mean":
                continue
            # Weights fitted separately for each objective: micro wants the
            # strongest members, macro wants whoever can see the rare tail.
            for _mt in OBJECTIVES:
                _sfx = "" if len(OBJECTIVES) == 1 else "@" + _mt
                Wg, _sc = greedy(vw, space, metric=_mt)
                add("greedy.%s%s%s" % (space, tag, _sfx), 2,
                    combine(vw, Wg, space), {"weights": Wg, "fit_metric": _mt})
                Wb = bagged(vw, space, metric=_mt)
                if Wb:
                    add("bagged.%s%s%s" % (space, tag, _sfx), 2,
                        combine(vw, Wb, space), {"weights": Wb, "fit_metric": _mt})
                if _mt == TGT:
                    best_w[(cal, space)] = Wb or Wg
                # per-support-tier weights: fit the blend separately for the
                # rare, mid and common labels. Only meaningful for macro.
                if _mt == "macro" and CONFIG.get("ensemble_tier_weights", True):
                    Ws, ok = [], True
                    for t, cols in enumerate(TIER_COLS):
                        if len(cols) < 2:
                            Ws.append(None); continue
                        try:
                            Wt, _ = greedy(vw, space, cols=cols, metric="macro")
                            Ws.append(Wt)
                        except Exception:
                            Ws.append(None); ok = False
                    if ok and any(Ws):
                        add("tiered.%s%s" % (space, tag), 3,
                            combine_tiered(vw, Ws, Wb or Wg, space),
                            {"weights": Wb or Wg, "tier_weights": Ws,
                             "fit_metric": "macro"})
    if mode == "zoo":
        for cal in cals:
            W = best_w.get((cal, "prob"))
            if not W:
                continue
            vw = views(cal)
            for q in CONFIG["ensemble_power_grid"]:
                if q == 1.0:
                    continue
                add("powmean(q=%+.1f).prob%s" % (q, "+cal" if cal else ""), 3,
                    combine(vw, W, "prob", q), {"weights": W, "q": q})

    CAND.sort(key=lambda c: (-c["cf_fast"], c["simplicity"]))
    print("%d plain combinations; best so far %s (cf-val %.4f)"
          % (len(CAND), CAND[0]["name"], CAND[0]["cf_fast"]))
    ref = CAND[0]

    if mode == "zoo":
        for kind, on in (("lr", CONFIG["stack_lr"]), ("xgb", CONFIG["stack_xgb"])):
            if not on:
                continue
            _t0 = time.time()
            try:
                mats, note = run_stack(kind, views(False), ref["mats"])
            except Exception as exc:
                mats, note = None, "%s: %s" % (type(exc).__name__, exc)
            if mats is None:
                print("  stack-%-3s skipped (%s)" % (kind, note)); continue
            c = add("stack.%s" % kind, 4 if kind == "lr" else 5, mats, {"note": note})
            STACK_RAN = True
            print("  stack-%-3s %s -> cf-val %.4f  (%.1f min)"
                  % (kind, note, c["cf_fast"], (time.time() - _t0) / 60))
            for w in CONFIG["stack_blend_grid"]:
                if w in (0.0, 1.0):
                    continue
                mix = {k: (w * np.asarray(mats[k], np.float64)
                           + (1.0 - w) * np.asarray(ref["mats"][k], np.float64)
                           ).astype(np.float32) for k in RAW_KEYS}
                add("mix(%.2f*stack.%s + %.2f*%s)" % (w, kind, 1 - w, ref["name"]),
                    6, mix, {"w": w, "stack": kind, "blend": ref["name"]})

    # ---------- 5. finalists get the FULL decode search, per objective --
    n_fin = max(1, int(CONFIG["ensemble_finalists"]))
    fin_ix = []
    for _mt in OBJECTIVES:                       # top-N for EACH objective
        order = sorted(range(len(CAND)),
                       key=lambda i: (-CAND[i]["cf_" + _mt], CAND[i]["simplicity"]))
        for i in order[:n_fin]:
            if i not in fin_ix:
                fin_ix.append(i)
    for i, c in enumerate(CAND):                 # always give the simplest a shot
        if c["simplicity"] <= 1 and i not in fin_ix:
            fin_ix.append(i)
    print("\nfull decode search (rules x force-top1) on %d of %d "
          "candidates, for %s:" % (len(fin_ix), len(CAND), " and ".join(OBJECTIVES)))
    for i in fin_ix:
        c = CAND[i]
        c["decode"], c["cf_full"] = {}, {}
        for _mt in OBJECTIVES:
            f, r, thr, tab, cf = select_decode(c["mats"]["val_s"], vy, vgrp,
                                               metric=_mt)
            c["decode"][_mt], c["cf_full"][_mt] = (f, r, thr), cf
        print("  %-42s " % c["name"][:42]
              + " | ".join("cf-%s %.4f (%s)" % (_mt, c["cf_full"][_mt],
                                                c["decode"][_mt][1])
                           for _mt in OBJECTIVES))

    _fin = [CAND[i] for i in fin_ix]
    WINNERS = {}
    for _mt in OBJECTIVES:
        _top = max(c["cf_full"][_mt] for c in _fin)
        WINNERS[_mt] = min(
            (c for c in _fin if c["cf_full"][_mt] >= _top - CONFIG["ensemble_select_tol"]),
            key=lambda c: (c["simplicity"], -c["cf_full"][_mt]))
        print("  best for %-5s: %s (cf-val %.4f)"
              % (_mt, WINNERS[_mt]["name"], WINNERS[_mt]["cf_full"][_mt]))
    win = WINNERS[TGT]

    def _register(nm, cand):
        raw = {k: cand["mats"][k] for k in RAW_KEYS}
        raw["val_y"], raw["test_y"] = vy, ty
        return package_result(
            nm, cand["name"], raw,
            extra={"kind": "ensemble", "candidate": cand["name"],
                   "members": cand["meta"].get("weights"),
                   "candidates_tried": len(CAND),
                   "cf_full": cand["cf_full"][TGT], "minutes": 0.0})

    print("\nSELECTED (on cross-fitted val, never on test): %s" % win["name"])
    results[ENSEMBLE_NAME] = _register(ENSEMBLE_NAME, win)
    # If the other objective prefers a DIFFERENT blend, keep that one too, so
    # both headline numbers come from the blend that is actually best for them.
    for _mt in OBJECTIVES:
        if _mt != TGT and WINNERS[_mt]["name"] != win["name"]:
            _nm = "%s@%s" % (ENSEMBLE_NAME, _mt)
            results[_nm] = _register(_nm, WINNERS[_mt])
            print("  (also kept %s = %s, which wins on %s)"
                  % (_nm, WINNERS[_mt]["name"], _mt))

    _ens  = results[ENSEMBLE_NAME]
    _key  = "macro_f1" if TGT == "macro" else "micro_f1"
    _bs   = max(results[m]["metrics_tuned"][_key] for m in MEMBERS)
    for _mt in OBJECTIVES:
        _k2 = "macro_f1" if _mt == "macro" else "micro_f1"
        _w  = WINNERS[_mt]
        _wr = results.get("%s@%s" % (ENSEMBLE_NAME, _mt), _ens)
        _mb = max(results[m]["metrics_at_" + _mt][_k2] for m in MEMBERS
                  if results[m].get("metrics_at_" + _mt))
        _cb2 = max(results[m]["val_cf_" + _mt] for m in MEMBERS
                   if results[m].get("val_cf_" + _mt) is not None)
        print("  %-5s  cf-val %.4f (best member %.4f, %+.4f)   ->   TEST %s %.4f "
              "(best member %.4f, %+.4f)"
              % (_mt, _w["cf_full"][_mt], _cb2, _w["cf_full"][_mt] - _cb2, _k2,
                 _wr["metrics_at_" + _mt][_k2], _mb,
                 _wr["metrics_at_" + _mt][_k2] - _mb))
    print("  headline (%s): micro %.4f | macro %.4f | macro(>=%d pos) %.4f over %d"
          % (TGT, _ens["metrics_tuned"]["micro_f1"], _ens["metrics_tuned"]["macro_f1"],
             CONFIG.get("macro_min_test_support", 10),
             _ens["metrics_tuned"]["macro_f1_eval"],
             _ens["metrics_tuned"]["n_eval_labels"]))
    if _ens["metrics_tuned"][_key] < _bs:
        print("  [note] the val winner does not lead on test. That is a real val->test")
        print("         gap on ~1.5k rows, not a bug -- selection never sees test, and")
        print("         re-picking after looking would stop test from being a test set.")
    if isinstance(win["meta"].get("weights"), dict):
        print("  weights:", {k: round(v, 3) for k, v in
                             sorted(win["meta"]["weights"].items(), key=lambda kv: -kv[1])})
        _dropped = [m for m in MEMBERS if m not in win["meta"]["weights"]]
        if _dropped:
            print("  dropped entirely:", _dropped)

    # ---------- 6. the candidate table --------------------------------
    for c in CAND:
        row = {"candidate": c["name"], "simplicity": c["simplicity"]}
        for _mt in OBJECTIVES:
            _thr = FAST_RULE[_mt](c["mats"]["val_s"], vy)
            _mm = multilabel_metrics(ty, apply_pred(c["mats"]["test_s"], _thr))
            row["cf_val_" + _mt] = round(c["cf_" + _mt], 4)
            row["cf_full_" + _mt] = (round(c["cf_full"][_mt], 4)
                                     if isinstance(c.get("cf_full"), dict)
                                     and _mt in c["cf_full"] else None)
            row["test_" + _mt] = _mm["micro_f1" if _mt == "micro" else "macro_f1"]
        row["selected"] = c["name"] == win["name"]
        ens_table.append(row)
    ens_df = pd.DataFrame(ens_table).sort_values("cf_val_" + TGT, ascending=False)
    ens_df.to_csv(os.path.join(CONFIG["out_dir"], "ensemble_candidates.csv"), index=False)
    display(ens_df.reset_index(drop=True))
    print("The choice used cf_val_* ONLY. `test_micro_globalthr` is printed for the")
    print("record (and to show how small the val->test drift is); reading it and then")
    print("changing the pick is exactly how a test split stops being one.")

    for c in CAND:                      # release ~3 MB x n_candidates
        c.pop("mats", None)
    _VIEWS.clear()
    gc.collect()


In [ ]:
# =============================================================
# 8d. OPERATING-POINT COMPARISON  (0.5 vs the val-selected rule, per model)
# =============================================================
print("Every row is scored on TEST. The operating point marked '<-- reported' was")
print("chosen on cross-fitted VALIDATION, before test was touched.\n")
print(f"{'model':<22} {'cut':>26}  {'micro_f1':>9} {'macro_f1':>9} {'macro>=k':>10} "
      f"{'jaccard':>8} {'card_pred':>9}")
print("-" * 100)
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["metrics_tuned"]["micro_f1"]):
    _rows = [("@0.5", "metrics_0.5")]
    for _mt in ("micro", "macro"):
        if r.get("metrics_at_" + _mt):
            _d = r.get("decode_" + _mt) or {}
            _rows.append(("@%s-opt (%s%s)" % (_mt, _d.get("rule", "?"),
                                              "+top1" if _d.get("force") else ""),
                          "metrics_at_" + _mt))
    _rows.append(("@old perlabelF1", "metrics_oldrule"))
    for tag, key in _rows:
        if key not in r or r[key] is None: continue
        m = r[key]
        mark = "  <-- headline" if m is r.get("metrics_best") else ""
        print(f"{name:<22} {tag:>26}  "
              f"{m['micro_f1']:>9.4f} {m['macro_f1']:>9.4f} {m['macro_f1_eval']:>10.4f} "
              f"{m['jaccard_samples']:>8.4f} {m['card_pred']:>9.3f}{mark}")
    print()
print("true test label cardinality: %.3f" % Y_STRAT_TE.sum(1).mean())
print("\n'@old perlabelF1' is the rule this notebook shipped before: it maximises each")
print("label's own F1, which over-predicts rare labels into the shared micro-F1 pool.")


### 8e. Context-width ablation

The notebook inherits "show the model up to 5 prior turns" from Petrova et al.'s
*prompt* design. That is a reasonable prior for an LLM asked to reason about a
turn, but it was never tested for a fine-tuned classifier -- and there is a
specific reason to doubt it here: **the context window contains previous
persuader turns, which themselves used strategies.** Their strategy-signalling
language is exactly what the classifier keys on, so it can bleed into the
current turn's prediction.

A controlled ablation on this corpus (bag-of-words surrogate, same
dialogue-level split, trained and tested at each width) showed the effect is
large and monotonic -- 0 turns 0.575, 1 turn 0.537, 2 turns 0.466, 3 turns
0.451, 5 turns 0.419 micro-F1 -- with **29 of 33 evaluable strategies improving
when context was removed**. `gratitude_and_appreciation` went 0.54 -> 0.86 (a
previous turn's "thank you" was being attributed to the current one), and
several rare strategies went from never-predicted to F1 0.3-0.6.

A transformer is not a bag of words: attention, position, the `[Persuader]`
markers and the speaker-role embeddings all give it ways to discount context
that TF-IDF does not have, so the effect should be smaller here. This table is
the honest test on the real encoders -- read it before quoting a context width
as a design choice, and note that the ensemble search in 8c is free to drop
whichever width loses.


In [ ]:
# =============================================================
# 8e. CONTEXT-WIDTH ABLATION
# =============================================================
abl = []
for name, r in results.items():
    if name == ENSEMBLE_NAME or "context_turns" not in r:
        continue
    abl.append({"model": name, "encoder": r["hf_id"], "context_turns": r["context_turns"],
                "max_length": r["max_length"], "seed": r.get("seed"),
                "micro_f1": r["metrics_best"]["micro_f1"],
                "macro_f1": r["metrics_best"]["macro_f1"],
                "jaccard": r["metrics_best"]["jaccard_samples"],
                "card_pred": r["metrics_best"]["card_pred"],
                "minutes": r["minutes"], "epochs": r["epochs_run"]})
if abl:
    abl_df = pd.DataFrame(abl).sort_values(["encoder", "context_turns"])
    abl_df.to_csv(os.path.join(CONFIG["out_dir"], "context_ablation.csv"), index=False)
    display(abl_df.round(4))

    by_ctx = abl_df.groupby("context_turns")["micro_f1"].agg(["mean", "max", "count"])
    print("\nmicro-F1 by context width (mean over encoders trained at that width):")
    print(by_ctx.round(4).to_string())
    if len(by_ctx) > 1:
        best_w = int(by_ctx["mean"].idxmax()); worst_w = int(by_ctx["mean"].idxmin())
        d = float(by_ctx["mean"].max() - by_ctx["mean"].min())
        print(f"\n-> best width {best_w} turns, worst {worst_w} turns, spread {d:.4f} micro-F1")
        if best_w < worst_w:
            print("   Context HURT on the real encoders too: the previous persuader turns'")
            print("   strategy language bleeds into the current turn's prediction. Set")
            print("   CONFIG['context_turns'] to the winning width for Phase 4 labelling.")
        else:
            print("   Context HELPED here -- the transformer discounts it well enough that the")
            print("   extra history pays for itself, unlike in the bag-of-words ablation.")
        print("\nwall-clock by width (shorter context is also much cheaper):")
        print(abl_df.groupby("context_turns")["minutes"].mean().round(1).to_string())
else:
    print("only one configuration trained - nothing to ablate")


## 9. Baselines on the matched test split (Phase 3.4)

Three, all scored with the exact same `multilabel_metrics()` on the exact same
test split, so section 10 is apples-to-apples:

* **prior** -- always predict the *k* globally-commonest strategies (*k* = the
  train split's mean label cardinality). This is the number a 41-way multi-label
  model has to beat to have learned anything at all.
* **TF-IDF (word 1-2-gram + char_wb 3-5-gram) -> Binary Relevance logistic
  regression** -- the classic strong non-neural multi-label baseline, from
  `kaggle_phase3_4_pipeline.ipynb` §3.
* **TF-IDF -> `ClassifierChain(XGBoost)`** -- chains the 41 labels so each
  classifier sees the previous labels' predictions, which is the cheapest way to
  model label *co-occurrence* rather than treating the 41 heads as independent.

Strategies with **no positive training example** are handled explicitly (a
constant-zero predictor) instead of crashing XGBoost.


In [ ]:
# =============================================================
# 9. BASELINES ON THE MATCHED TEST SPLIT
# =============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from scipy.sparse import hstack

baseline_metrics = {}
corpus_tr = train_df["model_text"].tolist()
corpus_te = test_df["model_text"].tolist()

# ---- prior: predict the k commonest strategies ------------------------
k_tr = max(1, round(Y_STRAT_TR.sum(1).mean()))
top_tr = np.argsort(Y_STRAT_TR.sum(0))[::-1][:k_tr]
Pte = np.zeros_like(Y_STRAT_TE); Pte[:, top_tr] = 1
baseline_metrics["prior_topk"] = multilabel_metrics(Y_STRAT_TE, Pte)
print("prior (predict %d commonest: %s)  micro-F1 %.4f"
      % (k_tr, [STRATS[i] for i in top_tr], baseline_metrics["prior_topk"]["micro_f1"]))

# ---- TF-IDF word + char features -------------------------------------
if CONFIG["FAST_DEV_RUN"]:
    wv = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
    Xtr, Xte = wv.fit_transform(corpus_tr), wv.transform(corpus_te)
else:
    wv  = TfidfVectorizer(max_features=40000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    cvz = TfidfVectorizer(max_features=40000, ngram_range=(3, 5), analyzer="char_wb", min_df=3)
    Xtr = hstack([wv.fit_transform(corpus_tr), cvz.fit_transform(corpus_tr)]).tocsr()
    Xte = hstack([wv.transform(corpus_te),     cvz.transform(corpus_te)]).tocsr()
print("TF-IDF matrix:", Xtr.shape)

# ---- Binary Relevance logistic regression ----------------------------
br = OneVsRestClassifier(LogisticRegression(max_iter=300, C=4, class_weight="balanced"), n_jobs=-1)
br.fit(Xtr, Y_STRAT_TR)
baseline_metrics["tfidf_logreg_BR"] = multilabel_metrics(Y_STRAT_TE, br.predict(Xte))
print("BR-LR      micro-F1 %.4f  macro-F1 %.4f"
      % (baseline_metrics["tfidf_logreg_BR"]["micro_f1"],
         baseline_metrics["tfidf_logreg_BR"]["macro_f1"]))
print("\n[read this before comparing rows] `baseline::tfidf_logreg_BR` here and the")
print("`tfidf-lr` MEMBER from S8b are the same estimator and differ a lot anyway,")
print("for two reasons that are worth separating:")
print("  1. this baseline uses the DEFAULT context width (%d turns); the member uses"
      % CONFIG["context_turns"])
print("     the context-free mt0 text, and S8e/the ablation show context costs a")
print("     bag-of-ngrams ~0.15 micro-F1 on this corpus;")
print("  2. this baseline predicts at a hard 0.5 with no decode search, while every")
print("     member gets the same val-fitted thresholds/smoothing everything else does.")
print("Both numbers are honest; they answer different questions. S10 compares the")
print("headline model against the STRONGEST of the two, which is the fair one.")

# ---- ClassifierChain(XGBoost) ----------------------------------------
# Off by default: the `tfidf-xgb` MEMBER built in S8b is the same idea done
# properly (one booster per label on dense SVD + meta features, ~3 min instead
# of 61, and far more accurate), and it is reported in the S10 table as the
# strong classical baseline. This block exists to reproduce the chain number.
if CONFIG["run_xgb_chain"]:
    try:
        from xgboost import XGBClassifier
        from sklearn.multioutput import ClassifierChain
        keep = np.where(Y_STRAT_TR.sum(0) > 0)[0]        # XGB cannot fit a single-class y
        dropped = [STRATS[j] for j in range(len(STRATS)) if j not in set(keep.tolist())]
        if dropped:
            print("  (constant-zero in train, predicted as 0:", dropped, ")")
        base = XGBClassifier(n_estimators=160, max_depth=5, learning_rate=0.15,
                             subsample=0.8, colsample_bytree=0.6, n_jobs=-1,
                             tree_method="hist", eval_metric="logloss", verbosity=0)
        cc = ClassifierChain(base, order="random", random_state=SEED)
        t0 = time.time()
        cc.fit(Xtr, Y_STRAT_TR[:, keep])
        Pcc = np.zeros_like(Y_STRAT_TE)
        Pcc[:, keep] = (np.asarray(cc.predict(Xte)) >= 0.5).astype(int)
        baseline_metrics["tfidf_xgboost_chain"] = multilabel_metrics(Y_STRAT_TE, Pcc)
        print("XGB-chain  micro-F1 %.4f  macro-F1 %.4f  (%.1f min)"
              % (baseline_metrics["tfidf_xgboost_chain"]["micro_f1"],
                 baseline_metrics["tfidf_xgboost_chain"]["macro_f1"], (time.time() - t0) / 60))
    except Exception as e:
        print("xgboost chain skipped:", type(e).__name__, e)
else:
    print("xgboost chain disabled in CONFIG")

json.dump(baseline_metrics, open(os.path.join(CONFIG["out_dir"], "metrics_baselines.json"), "w"), indent=2)
display(pd.DataFrame(baseline_metrics).T[["micro_f1", "macro_f1", "samples_f1",
                                          "hamming_loss", "jaccard_samples", "card_pred"]].round(4))


## 10. Results summary -- models vs baselines (test split)


In [ ]:
# =============================================================
# 10. RESULTS SUMMARY
# =============================================================
rows = []
for k, m in baseline_metrics.items():
    rows.append({"model": f"baseline::{k}", **{mk: m[mk] for mk in METRIC_KEYS}})
for name, r in results.items():
    _rows = [("@0.5", "metrics_0.5")]
    for _mt in ("micro", "macro"):
        if r.get("metrics_at_" + _mt):
            _rows.append(("@%s-opt" % _mt, "metrics_at_" + _mt))
    _rows.append(("@old perlabelF1", "metrics_oldrule"))
    for tag, key in _rows:
        if key in r and r[key] is not None:
            rows.append({"model": f"{name} {tag}", **{mk: r[key][mk] for mk in METRIC_KEYS}})
summary_df = pd.DataFrame(rows).set_index("model")
summary_df.to_csv(os.path.join(CONFIG["out_dir"], "classifier_test_results.csv"))

metrics_all = {"baselines": baseline_metrics,
               "models": {n: {"metrics_0.5": r["metrics_0.5"],
                              "metrics_tuned": r["metrics_tuned"],
                              "metrics_oldrule": r.get("metrics_oldrule"),
                              "metrics_best": r["metrics_best"],
                              "val_cf_micro_f1": r.get("val_cf"),
                              "kind": r.get("kind"),
                              "decision_rule": r.get("rule"),
                              "force_top1": r.get("force_top1"),
                              "rule_selection": r.get("rule_tables"),
                              "thresholds": r["thresholds"],
                              "speaker_roles": r["use_roles"],
                              "epochs_run": r["epochs_run"], "minutes": r["minutes"],
                              "ensemble_members": r.get("members"),
                              "ensemble_candidate": r.get("candidate")}
                          for n, r in results.items()},
               "ensemble_candidates": ens_table,
               "split_mode": split_mode, "seed": SEED,
               "zero_support_strategies": ZERO_SUPPORT}
json.dump(metrics_all, open(os.path.join(CONFIG["out_dir"], "metrics_all.json"), "w"), indent=2)

# HEADLINE MODEL: chosen by CROSS-FITTED VALIDATION in CONFIG["target_metric"],
# never by test.
#
# The previous revision picked the model AND its operating point by test
# micro-F1 ("best over both cuts"). With 11 members plus an ensemble that is a
# 24-way peek at a 1.5k-row test split, worth a few tenths of a point of pure
# optimism. Every model now carries `val_cf` -- the grouped cross-fitted val
# score of its own selected decode -- and that is what decides. `fixed@0.5` is
# one of the candidate rules, so if not tuning really were best, the search
# would already have returned it.
BEST = max(results, key=lambda n: results[n]["val_cf"])
best_r  = results[BEST]
THR     = np.array(best_r["thresholds"])
FORCE1  = bool(best_r.get("force_top1", False))
BEST_CUT = best_r.get("rule", "rule") + ("+top1" if FORCE1 else "")
te_prob = best_r["test_prob"]
te_pred = apply_pred(te_prob, THR, FORCE1)
te_y    = best_r["test_y"]

TGT_METRIC = CONFIG.get("target_metric", "micro")
TGT_KEY = "macro_f1" if TGT_METRIC == "macro" else "micro_f1"
_bm = best_r["metrics_tuned"]

# The comparator pool is every non-transformer model in the run: the S9
# baselines plus the S8b shallow / prototype / cue members, scored on the SAME
# metric that is being optimised.
_nont = [n for n in results
         if results[n].get("kind") in ("shallow", "proto", "cue")]
_base_pool = {("baseline::" + k): v[TGT_KEY] for k, v in baseline_metrics.items()}
_base_pool.update({n: results[n]["metrics_tuned"][TGT_KEY] for n in _nont})
_bb_name, _best_base = max(_base_pool.items(), key=lambda kv: kv[1])

print("headline model (selected on cross-fitted val %s): %s" % (TGT_METRIC, BEST))
print("  operating point: %s" % BEST_CUT)
print("  cross-fitted val %s %.4f  ->  TEST %s %.4f"
      % (TGT_METRIC, best_r["val_cf"], TGT_METRIC, _bm[TGT_KEY]))
print("  TEST  macro-F1 %.4f (all %d strategies)  |  macro-F1 %.4f (the %d with "
      ">=%d test positives)  |  micro-F1 %.4f"
      % (_bm["macro_f1"], len(STRATS), _bm["macro_f1_eval"], _bm["n_eval_labels"],
         CONFIG.get("macro_min_test_support", 10), _bm["micro_f1"]))
print("  vs strongest non-transformer model (%s) %s %.4f   (%+.4f, %+.1f%% relative)"
      % (_bb_name, TGT_METRIC, _best_base, _bm[TGT_KEY] - _best_base,
         100 * (_bm[TGT_KEY] / max(_best_base, 1e-9) - 1)))
_test_leader = max(results, key=lambda n: results[n]["metrics_tuned"][TGT_KEY])
if _test_leader != BEST:
    print("  [note] %s scores higher ON TEST %s (%.4f vs %.4f) but lost on validation."
          % (_test_leader, TGT_METRIC,
             results[_test_leader]["metrics_tuned"][TGT_KEY], _bm[TGT_KEY]))
    print("         The val-selected model is the one reported: switching to the test")
    print("         leader after the fact is selection on the test split.")
# The best each objective can do, across every model, each at its own optimum
# and each selected on validation.
print("\nBEST ACHIEVED PER OBJECTIVE (model chosen on cross-fitted val for that objective):")
_cb = (CEILING or {}).get("consensus_predictor_bound_self")
for _mt, _key in (("micro", "micro_f1"), ("macro", "macro_f1")):
    _cands = [(n, r) for n, r in results.items()
              if r.get("metrics_at_" + _mt) and r.get("val_cf_" + _mt) is not None]
    if not _cands:
        continue
    _n, _r = max(_cands, key=lambda kv: kv[1]["val_cf_" + _mt])
    _m = _r["metrics_at_" + _mt]
    _d = _r.get("decode_" + _mt) or {}
    print("  %-5s-optimal: %-18s %s-F1 %.4f   (micro %.4f | macro %.4f | "
          "macro>=%d %.4f over %d)"
          % (_mt, _n, _mt, _m[_key], _m["micro_f1"], _m["macro_f1"],
             CONFIG.get("macro_min_test_support", 10), _m["macro_f1_eval"],
             _m["n_eval_labels"]))
    print("        rule=%s top1=%s  (cf-val %.4f)%s"
          % (_d.get("rule"), _d.get("force"), _r["val_cf_" + _mt],
             ("   = %.0f%% of the ~%.2f annotation ceiling"
              % (100 * _m["micro_f1"] / _cb, _cb)) if (_cb and _mt == "micro") else ""))

if TGT_METRIC == "macro":
    print("\n  Before quoting a macro-F1: two strategies have ZERO test positives, so")
    print("  the all-41 average is capped at %.3f on this split whatever the model"
          % (te_y.sum(0) > 0).mean())
    print("  does. Section 10c breaks macro-F1 down by label support -- quote the row")
    print("  whose label count matches your claim, and say the count.")
display(summary_df.round(4))


In [ ]:
# =============================================================
# 10a. F1 COMPARISON BAR CHART
# =============================================================
plot_df = (summary_df.reset_index()
           .melt(id_vars="model", value_vars=["micro_f1", "macro_f1", "jaccard_samples"],
                 var_name="metric", value_name="score"))
plt.figure(figsize=(13, 5.5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model")
plt.ylim(0, max(0.05, plot_df.score.max() * 1.25))
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Multi-label strategy classifier vs baselines (test split)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "classifier_comparison_bar.png"), dpi=140); plt.show()


In [ ]:
# =============================================================
# 10c. PER-CATEGORY REPORT  (best model)
# =============================================================
rep = pd.DataFrame({
    "category": STRATS,
    "train_support": Y_STRAT_TR.sum(0).astype(int),
    "test_support": te_y.sum(0).astype(int),
    "precision": [precision_score(te_y[:, i], te_pred[:, i], zero_division=0) for i in range(len(STRATS))],
    "recall":    [recall_score(te_y[:, i], te_pred[:, i], zero_division=0) for i in range(len(STRATS))],
    "f1":        [f1_score(te_y[:, i], te_pred[:, i], zero_division=0) for i in range(len(STRATS))],
    "threshold": THR,
}).sort_values("f1")

# ---- macro-F1 by how scoreable the label is -------------------------------
# With only 11 categories over 10,600 turns, every category should clear a
# reasonable support floor -- this table exists mainly to catch it if one
# doesn't, not because it's expected to matter the way it did at strategy
# granularity.
_tiers = macro_f1_tiers(te_y, te_pred)
tier_tbl = pd.DataFrame([
    {"min_test_support": k, "n_labels": v["n_labels"],
     "macro_f1": round(v["macro_f1"], 4),
     "share_of_taxonomy": round(v["n_labels"] / len(STRATS), 3)}
    for k, v in sorted(_tiers.items())])
tier_tbl["mean_test_support"] = [
    int(rep.loc[rep.test_support >= k, "test_support"].mean()) if (rep.test_support >= k).any() else 0
    for k in tier_tbl.min_test_support]
tier_tbl.to_csv(os.path.join(CONFIG["out_dir"], "macro_f1_by_support_tier.csv"), index=False)
print("macro-F1 by label test support (%s, %s):" % (BEST, BEST_CUT))
print(tier_tbl.to_string(index=False))
print("  all-category macro-F1 is bounded above by %.3f on this split: "
      "%d categories have zero test positives."
      % ((te_y.sum(0) > 0).mean(), int((te_y.sum(0) == 0).sum())))
print("  labels with zero test support:",
      [STRATS[j] for j in np.where(te_y.sum(0) == 0)[0]] or "none")
_zero_pred = [STRATS[j] for j in range(len(STRATS))
              if te_pred[:, j].sum() == 0 and te_y[:, j].sum() > 0]
print("  scoreable but never predicted (%d): %s" % (len(_zero_pred), _zero_pred or "none"))

rep.to_csv(os.path.join(CONFIG["out_dir"], "per_category_f1.csv"), index=False)
print("weakest categories:\n", rep.head(10).round(3).to_string(index=False))
print("\nstrongest categories:\n", rep.tail(10).round(3).to_string(index=False))
print("\ncategories never predicted on test:",
      [STRATS[j] for j in range(len(STRATS)) if te_pred[:, j].sum() == 0] or "none")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(rep.train_support + 1, rep.f1, s=28, color="#4C72B0")
ax.set_xscale("log"); ax.set_xlabel("train support (log)"); ax.set_ylabel("test F1")
ax.set_title("Per-category F1 vs training support")
for _, rw in rep.head(4).iterrows():
    ax.annotate(rw.category, (rw.train_support + 1, rw.f1), fontsize=7, alpha=.7)
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["out_dir"], "f1_vs_support.png"), dpi=140); plt.show()


In [ ]:
# =============================================================
# 10d. METRICS BY ANNOTATOR SEGMENT  (label-density / rater effect)
# =============================================================
# The four annotation segments differ sharply in label density (2.38 vs 1.24
# strategies/turn). If test metrics track that ordering, part of what the model
# is being scored on is rater granularity, not persuasion.
assert len(te_y) == len(test_df), "prediction / test_df length mismatch (loader must be shuffle=False)"
seg_rows = []
for seg, g in test_df.groupby("annotator"):
    m = test_df.annotator.values == seg
    if m.sum() < 25:
        continue
    r = multilabel_metrics(te_y[m], te_pred[m])
    r.update(segment=seg, n=int(m.sum()))
    seg_rows.append(r)
if seg_rows:
    seg_tbl = (pd.DataFrame(seg_rows).set_index("segment")
               [["n", "micro_f1", "macro_f1", "jaccard_samples", "card_true", "card_pred"]]
               .sort_values("card_true", ascending=False))
    seg_tbl.to_csv(os.path.join(CONFIG["out_dir"], "metrics_by_segment.csv"))
    display(seg_tbl.round(4))
    print("card_true is the segment's true labels/turn; if micro_f1 tracks it, the "
          "model is partly learning rater granularity.")
else:
    print("no annotator segment has >= 25 test turns")


In [ ]:
# =============================================================
# 10e. WHERE THE MODEL ERRS - most-confused category pairs
# =============================================================
fp = defaultdict(Counter)
for gt, pr in zip(te_y, te_pred):
    for a in np.where((pr == 1) & (gt == 0))[0]:
        for b in np.where(gt == 1)[0]:
            fp[STRATS[a]][STRATS[b]] += 1
conf = pd.DataFrame([(a, b, n) for a, cc in fp.items() for b, n in cc.most_common(1)],
                    columns=["predicted", "instead_of_gold", "count"]
                    ).sort_values("count", ascending=False)
conf.to_csv(os.path.join(CONFIG["out_dir"], "confused_pairs.csv"), index=False)
print("top category confusions (there is no finer grouping below category to")
print("check these against, unlike the strategy-level version of this pipeline):")
display(conf.head(15))


In [ ]:
# =============================================================
# 10f. TRAINING CURVES
# =============================================================
hist_models = {n: r for n, r in results.items() if r["history"]}
if hist_models:
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
    for name, r in hist_models.items():
        h = pd.DataFrame(r["history"])
        ax[0].plot(h.epoch, h.train_loss, marker="o", label=name)
        ax[1].plot(h.epoch, h.val_micro_f1_05, marker="o", label=name)
        ax[2].plot(h.epoch, h.select, marker="o", label=name)
    ax[0].set_title("train loss"); ax[1].set_title("val micro-F1 @0.5")
    ax[2].set_title("selection metric (tuned-threshold, cross-fitted)")
    for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(CONFIG["out_dir"], "training_curves.png"), dpi=140); plt.show()

    display(pd.concat({n: pd.DataFrame(r["history"]).set_index("epoch")
                       for n, r in hist_models.items()}, names=["model"]).round(4))
else:
    print("no per-epoch history (only the ensemble is present)")


## 11. Label the full 10,600-turn corpus with the best model

Writes `pred_multilabel_full.jsonl` (`turn_id`, `dialogue_id`, `strategies`,
`categories`, `probs`) -- same schema as the Phase-1 LLM outputs, so it drops
into the same downstream tooling. The probabilities were already computed inside
`train_one_encoder`, so this section costs no extra GPU time and the ensemble
gets full-corpus predictions for free.

Note the leakage caveat printed below: 70 % of these turns were in the model's
training split, so corpus-wide predicted label density is optimistic. Phase 4
uses **gold** labels by default for exactly that reason
(`CONFIG["phase4_label_source"]`).


In [ ]:
# =============================================================
# 11. FULL-CORPUS INFERENCE WITH THE BEST MODEL
# =============================================================
full_prob = best_r["full_prob"]
full_pred = apply_pred(full_prob, THR, FORCE1)
if best_r.get("kind") == "ensemble" and "stack" in str(best_r.get("candidate", "")):
    print("[NOTE] the headline model is a level-2 stacker. Its inputs are the member")
    print("       probabilities, and on TRAIN rows those are over-confident (the")
    print("       members were fitted on them), so train-row predictions here are")
    print("       even more optimistic than usual. The val/test rows are unaffected,")
    print("       and Phase 4 defaults to gold labels anyway.\n")

out_path = os.path.join(CONFIG["out_dir"], "pred_multilabel_full.jsonl")
_df = df.reset_index(drop=True)
with open(out_path, "w", encoding="utf-8") as fh:
    for i in range(len(_df)):
        js = np.where(full_pred[i] == 1)[0]
        cset = [STRATS[j] for j in js]
        fh.write(json.dumps({
            "turn_id": _df.turn_id.iloc[i], "dialogue_id": _df.dialogue_id.iloc[i],
            "split": _df.split.iloc[i],
            "categories": cset,
            "n_labels": len(cset),
            "probs": {STRATS[j]: round(float(full_prob[i, j]), 4)
                      for j in np.argsort(full_prob[i])[::-1][:6]},
        }, ensure_ascii=False) + "\n")

pred_df = pd.DataFrame({"turn_id": _df.turn_id, "dialogue_id": _df.dialogue_id,
                        "split": _df.split,
                        "categories": [[STRATS[j] for j in np.where(full_pred[i] == 1)[0]]
                                       for i in range(len(_df))]})
pred_df["n_labels"] = pred_df.categories.apply(len)
print("wrote", out_path)
print("predicted mean categories/turn: %.3f   gold: %.3f"
      % (pred_df.n_labels.mean(), df.n_strat.mean()))
print("\nby split (train rows were seen in training - their agreement is optimistic):")
print(pd.DataFrame({"pred_mean": pred_df.groupby(pred_df.split).n_labels.mean(),
                    "gold_mean": df.groupby("split").n_strat.mean()}).round(3))


## 12. Phase 4 gate

Phase 4 needs the dialogue-level donation outcome, which lives in
`persuader_turns.csv` (`binary_label_norm` / `modifier_norm`). If that file was
not attached, sections 13-17 print what they would have done and skip cleanly --
the classifier half of the notebook is complete on its own.


In [ ]:
# =============================================================
# 12. PHASE 4 AVAILABILITY GATE
# =============================================================
RUN_PHASE4 = bool(HAVE_OUTCOME)
# placeholders so sections 17-18 never reference an undefined name
feat = D = D_ = None
models, mw, guilt, mod_out = [], {}, {}, {}
chi_cat = pd.DataFrame()
cat_cols, pair_cols, top_pairs = [], [], []

if RUN_PHASE4:
    print("Phase 4 ENABLED - donation outcome found in persuader_turns.csv")
else:
    print("Phase 4 SKIPPED - no donation outcome available.")
    print("  Attach persuader_turns.csv (with binary_label_norm / modifier_norm)")
    print("  as a Kaggle input and re-run to get sections 13-17.")


## 13. Phase 4.1 -- per-dialogue category co-occurrence features

`cat_<category>` (11) + `distinct_categories` + `n_category_instances` + the
top-K `A__and__B` dialogue-level category co-occurrence pairs (plus the
explicit `Emotional Appeal__and__Reciprocity_and_Exchange` pair, the
category-level analogue of the paper's Guilt Induction x Reciprocity test).
Category-only: no fine-grained `has_<strategy>` features here, since Phase 3
no longer predicts at that granularity.


In [ ]:
# =============================================================
# 13. PER-DIALOGUE FEATURES  (Phase 4.1)
# =============================================================
# Category-only: a turn's category set is either the classifier's own
# prediction (already category names, since this pipeline's target IS
# category) or derived from the raw gold FINE-strategy labels via S2C.
# Either way, `lab["cats_here"]` below is a per-turn list of category names,
# and everything downstream builds dialogue-level features from that alone.
if RUN_PHASE4:
    if CONFIG["phase4_label_source"] == "pred":
        lab = pred_df[["turn_id", "dialogue_id", "categories"]].copy()
        lab["cats_here"] = lab["categories"]
        print("Phase 4 features from MODEL PREDICTIONS (%s)" % BEST)
    else:
        lab = df[["turn_id", "dialogue_id", "strategies"]].copy()
        lab["cats_here"] = lab["strategies"].apply(
            lambda ss: sorted({S2C[s] for s in ss if s in S2C}))
        print("Phase 4 features from GOLD (human-corrected) labels")

    def build_dialog_features(lab):
        rows = []
        for did, g in lab.groupby("dialogue_id"):
            flat = [c for cc in g["cats_here"] for c in cc]
            d = {"dialogue_id": did, "n_persuader_turns": len(g),
                 "n_category_instances": len(flat),
                 "distinct_categories": len(set(flat))}
            for c in CATS: d[f"cat_{c.replace(' ', '_')}"] = int(c in flat)
            rows.append(d)
        return pd.DataFrame(rows)

    feat = build_dialog_features(lab)

    pair_ct = Counter()
    for _, g in lab.groupby("dialogue_id"):
        present = sorted({c for cc in g["cats_here"] for c in cc})
        for a, b in combinations(present, 2):
            pair_ct[(a, b)] += 1
    top_pairs = [p for p, _ in pair_ct.most_common(CONFIG["cooc_top_k"])]
    # The strategy-level version of this notebook forced in the
    # (guilt_induction, reciprocity) pair specifically to re-test the paper's
    # Guilt-Induction finding. The category-level analogue is their parent
    # categories: (Emotional Appeal, Reciprocity and Exchange).
    _forced_pair = (S2C.get("guilt_induction", "Emotional Appeal"),
                    S2C.get("reciprocity", "Reciprocity and Exchange"))
    if _forced_pair not in top_pairs and _forced_pair[::-1] not in top_pairs:
        top_pairs.append(_forced_pair)
    for a, b in top_pairs:
        ca, cb = a.replace(" ", "_"), b.replace(" ", "_")
        feat[f"has_{ca}__and__{cb}"] = (feat[f"cat_{ca}"] & feat[f"cat_{cb}"]).astype(int)

    feat.to_csv(os.path.join(CONFIG["out_dir"], "dialog_features.csv"), index=False)
    print("dialogues: %d" % len(feat))
    print("distinct categories / dialogue: mean %.2f  median %.1f"
          % (feat.distinct_categories.mean(), feat.distinct_categories.median()))
    print("\ntop co-occurrence pairs:")
    for (a, b), n in pair_ct.most_common(8):
        print(f"  {n:4d}  {a} + {b}")
    display(feat.filter(regex="^(dialogue_id|distinct|n_)").head(3))
else:
    print("(skipped - no donation outcome)")


## 14. Phase 4.2 -- merge the normalized donation outcome

`donated` in {0,1} and `modifier` in {none, deferred, conditional} per dialogue
(from `persuader_turns.csv`, already normalized). Plus persuadee-side
`ee_sentiment` (VADER over persuadee turns; a tiny-lexicon fallback when NLTK's
lexicon cannot be downloaded) and `ee_engagement` (words/turn, z-scored) --
**proxies** for the paper's sentiment + interest covariates.


In [ ]:
# =============================================================
# 14. MERGE DONATION OUTCOME + PERSUADEE COVARIATES  (Phase 4.2)
# =============================================================
if RUN_PHASE4:
    out = (turns.groupby("dialogue_id")
           .agg(donated=("donated", "first"), modifier=("modifier", "first")).reset_index())
    out["donated"] = (out.donated.astype(str).str.lower().str.strip() == "yes").astype(int)
    out["modifier"] = (out.modifier.astype(str).str.lower().str.strip()
                       .replace({"": "none", "nan": "none", "na": "none"}))

    analyzer = None
    _POS = set("thank thanks great good yes happy glad love appreciate help sure agree wonderful kind".split())
    _NEG = set("no not cant can't won't wont sorry unfortunately broke poor never bad hard tight refuse".split())
    try:
        import nltk
        try:
            nltk.data.find("sentiment/vader_lexicon.zip")
        except LookupError:
            nltk.download("vader_lexicon", quiet=True)
        from nltk.sentiment import SentimentIntensityAnalyzer
        analyzer = SentimentIntensityAnalyzer()
        print("VADER sentiment available")
    except Exception as e:
        print("VADER unavailable (%s) - tiny fallback lexicon" % type(e).__name__)

    def _score(t):
        if analyzer: return analyzer.polarity_scores(t)["compound"]
        w = re.findall(r"[a-z']+", t.lower())
        return 0.0 if not w else (sum(x in _POS for x in w) - sum(x in _NEG for x in w)) / len(w)

    _EE_RE = re.compile(r"\[Persuadee\]\s*(.*?)(?=\n\s*\[Persua|\Z)", re.S)
    got_cov = False
    if PATH["manual"]:
        try:
            man = pd.read_csv(PATH["manual"], dtype=str).fillna("")
            tcol = next((c for c in ["dialogue_text", "text", "full_text", "dialogue"]
                         if c in man.columns), None)
            idcol = next((c for c in ["dialogue_id", "conversation_id", "id"]
                          if c in man.columns), None)
            if tcol and idcol:
                ee_sent, ee_eng = {}, {}
                for _, r in man.iterrows():
                    ee = [e.strip() for e in _EE_RE.findall(r[tcol]) if e.strip()]
                    ee_sent[r[idcol]] = np.mean([_score(e) for e in ee]) if ee else 0.0
                    ee_eng[r[idcol]]  = np.mean([len(e.split()) for e in ee]) if ee else 0.0
                out["ee_sentiment"]  = out.dialogue_id.map(ee_sent).fillna(0.0)
                out["ee_engagement"] = out.dialogue_id.map(ee_eng).fillna(0.0)
                sd = out.ee_engagement.std()
                out["ee_engagement"] = (out.ee_engagement - out.ee_engagement.mean()) / (sd + 1e-9)
                got_cov = out.ee_sentiment.abs().sum() > 0
                print(f"persuadee covariates from {os.path.basename(PATH['manual'])} "
                      f"({tcol}/{idcol}); matched {100*out.ee_sentiment.ne(0).mean():.0f}% of dialogues")
            else:
                print(f"  {os.path.basename(PATH['manual'])} lacks a dialogue-text/id column "
                      f"({list(man.columns)[:8]}...) - covariates = 0")
        except Exception as e:
            print("  manual-label parse failed (%s) - covariates = 0" % type(e).__name__)
    if not got_cov:
        out["ee_sentiment"] = out.get("ee_sentiment", 0.0)
        out["ee_engagement"] = out.get("ee_engagement", 0.0)
        out["ee_sentiment"] = pd.to_numeric(out["ee_sentiment"], errors="coerce").fillna(0.0)
        out["ee_engagement"] = pd.to_numeric(out["ee_engagement"], errors="coerce").fillna(0.0)
        print("  -> ee_sentiment / ee_engagement are all-zero; Models 2/3 lose those regressors")

    D = feat.merge(out, on="dialogue_id", how="inner")
    D.to_csv(os.path.join(CONFIG["out_dir"], "phase4_frame.csv"), index=False)
    print("\nmerged dialogues: %d  |  donate rate: %.3f" % (len(D), D.donated.mean()))
    print("modifier balance:", D.modifier.value_counts().to_dict())
else:
    print("(skipped - no donation outcome)")


## 15. Phase 4.3 -- chi2 association tests + logistic Models 1-4

chi2 of each category vs `donated` (Benjamini-Hochberg FDR), Mann-Whitney U on
category richness, and four logistic models reported as **McFadden pseudo-R2**
-- in-sample (comparable to the paper) and 5-fold CV (honest) -- with an
L2-regularised fit (11 categories on ~1,017 dialogues is a much smaller
quasi-separation risk than the 41-strategy version of this pipeline had, but
the same regularised convention is kept for consistency). Compare against the
paper's single-label baseline (**~0.015-0.08**).


In [ ]:
# =============================================================
# 15. CHI-SQUARE + LOGISTIC MODELS 1-4  (Phase 4.3)
# =============================================================
if RUN_PHASE4:
    from scipy.stats import chi2_contingency, mannwhitneyu
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    try:
        from statsmodels.stats.multitest import multipletests
        import statsmodels.formula.api as smf
        HAVE_SM = True
    except Exception as e:
        HAVE_SM = False; print("statsmodels unavailable (%s): no FDR / odds-ratios" % type(e).__name__)

    D_ = D.copy(); D_.columns = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in D_.columns]

    def chi_table(names, kind="cat"):
        rows = []
        for c in names:
            col = re.sub(r"[^0-9a-zA-Z_]", "_", f"cat_{c.replace(' ', '_')}")
            if col not in D_ or D_[col].nunique() < 2: continue
            ctab = pd.crosstab(D_[col], D_.donated)
            if ctab.shape != (2, 2): continue
            chi2, p, _, _ = chi2_contingency(ctab)
            r1 = D_.loc[D_[col] == 1, "donated"].mean(); r0 = D_.loc[D_[col] == 0, "donated"].mean()
            rows.append(dict(feature=c, chi2=chi2, p=p, donate_rate_with=r1,
                             donate_rate_without=r0, delta_pp=100*(r1-r0),
                             n_with=int(ctab.loc[1].sum())))
        t = pd.DataFrame(rows, columns=["feature","chi2","p","donate_rate_with",
                                        "donate_rate_without","delta_pp","n_with"])
        if len(t):
            t["p_fdr"] = (multipletests(t.p, alpha=CONFIG["fdr_alpha"], method="fdr_bh")[1]
                          if HAVE_SM else t.p)
            t = t.sort_values("p")
        else:
            t["p_fdr"] = pd.Series(dtype=float)
        return t

    chi_cat = chi_table(CATS)
    chi_cat.to_csv(os.path.join(CONFIG["out_dir"], "chi2_category.csv"), index=False)
    _sig_c = chi_cat[chi_cat.p_fdr < CONFIG["fdr_alpha"]] if len(chi_cat) else chi_cat
    print("category vs donated (FDR<%.2f):" % CONFIG["fdr_alpha"])
    print(_sig_c[["feature","chi2","p_fdr","delta_pp","n_with"]].round(4).to_string(index=False)
          if len(_sig_c) else "  (none survive FDR)")

    mw = {}
    for col in ["distinct_categories", "n_category_instances"]:
        a, b = D_.loc[D_.donated == 1, col], D_.loc[D_.donated == 0, col]
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        mw[col] = dict(U=float(u), p=float(p), mean_donated=float(a.mean()), mean_not=float(b.mean()))
    print("\ncategory richness by outcome (Mann-Whitney U):")
    display(pd.DataFrame(mw).T.round(4))

    cat_cols  = [c for c in (re.sub(r"[^0-9a-zA-Z_]", "_", f"cat_{c.replace(' ','_')}") for c in CATS)
                 if c in D_ and D_[c].nunique() > 1]
    pair_cols = [c for c in (re.sub(r"[^0-9a-zA-Z_]", "_", f"has_{a}__and__{b}") for a, b in top_pairs)
                 if c in D_ and D_[c].nunique() > 1]
    y = D_["donated"].to_numpy()

    def mcfadden(cols, tag, C=1.0):
        cols = [c for c in cols if c in D_ and D_[c].nunique() > 1]
        if not cols:
            return dict(model=tag, n_features=0, pseudo_r2_in_sample=float("nan"),
                        pseudo_r2_cv=float("nan"), pseudo_r2_cv_std=float("nan"))
        X = D_[cols].to_numpy(float); X = (X - X.mean(0)) / (X.std(0) + 1e-9)
        clf = LogisticRegression(C=C, max_iter=5000).fit(X, y)
        ll  = -log_loss(y, clf.predict_proba(X)[:, 1], normalize=False)
        ll0 = -log_loss(y, np.full(len(y), y.mean()), normalize=False)
        cv = []
        for tri, tei in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y):
            c = LogisticRegression(C=C, max_iter=5000).fit(X[tri], y[tri])
            cv.append(1 - (-log_loss(y[tei], c.predict_proba(X[tei])[:, 1], normalize=False)) /
                          (-log_loss(y[tei], np.full(len(tei), y[tri].mean()), normalize=False)))
        return dict(model=tag, n_features=len(cols), pseudo_r2_in_sample=round(1 - ll/ll0, 4),
                    pseudo_r2_cv=round(float(np.mean(cv)), 4),
                    pseudo_r2_cv_std=round(float(np.std(cv)), 4))

    # M3 used to isolate the paper's specific Guilt-Induction x Reciprocity
    # STRATEGY interaction. The category-level analogue is their parent
    # categories plus the forced co-occurrence pair feature Section 13 built
    # for exactly this purpose.
    _emo_col  = re.sub(r"[^0-9a-zA-Z_]", "_", f"cat_{S2C.get('guilt_induction','Emotional Appeal').replace(' ','_')}")
    _reci_col = re.sub(r"[^0-9a-zA-Z_]", "_", f"cat_{S2C.get('reciprocity','Reciprocity and Exchange').replace(' ','_')}")
    _pair_col = re.sub(r"[^0-9a-zA-Z_]", "_",
                       f"has_{S2C.get('guilt_induction','Emotional Appeal').replace(' ','_')}"
                       f"__and__{S2C.get('reciprocity','Reciprocity and Exchange').replace(' ','_')}")

    models = [
        mcfadden(cat_cols, "M1_categories"),
        mcfadden(cat_cols + ["ee_sentiment", "ee_engagement"], "M2_+sentiment+interest"),
        mcfadden([_emo_col, _reci_col, _pair_col, "ee_sentiment", "ee_engagement"],
                 "M3_emotional+reciprocity+sent+int"),
        mcfadden(cat_cols + ["n_category_instances", "distinct_categories",
                             "ee_sentiment", "ee_engagement"] + pair_cols, "M4_full_cooccurrence"),
    ]
    if HAVE_SM and cat_cols:
        try:
            m1 = smf.logit("donated ~ " + " + ".join(cat_cols), data=D_).fit(disp=0, method="lbfgs", maxiter=1000)
            or_tbl = (pd.DataFrame({"coef": m1.params, "p": m1.pvalues, "odds_ratio": np.exp(m1.params)})
                      .drop("Intercept").sort_values("p").round(3))
            or_tbl.to_csv(os.path.join(CONFIG["out_dir"], "phase4_M1_odds_ratios.csv"))
            print("\nModel 1 odds ratios (category present vs absent):")
            print(or_tbl.to_string())
        except Exception as e:
            print("statsmodels M1 skipped:", type(e).__name__, e)

    json.dump({"models": models, "paper_singlelabel_pseudo_r2": "~0.015-0.08",
               "mann_whitney": mw, "label_source": CONFIG["phase4_label_source"],
               "note": "pseudo_r2_cv is the honest out-of-sample estimate"},
              open(os.path.join(CONFIG["out_dir"], "phase4_models.json"), "w"), indent=2)
    print("\npaper single-label baseline pseudo-R2: ~0.015-0.08")
    display(pd.DataFrame(models))
else:
    print("(skipped - no donation outcome)")


## 15b. Which INDIVIDUAL categories actually move donation probability?

Section 15's chi2 table scores each category **one at a time** against
`donated` -- a useful first pass, but it cannot separate a category's own
association from the fact that it tends to co-occur with other categories.
This section fits every category *simultaneously* in one multivariate model,
controlling for dialogue length and persuadee sentiment/engagement, so each
category's coefficient reflects its own association net of the others.

**Method.** L2-regularised logistic regression (the same convention Models
1-4 above use) on standardized features. Because regularisation invalidates
textbook MLE standard errors, uncertainty comes from an 800-round
**dialogue-level bootstrap** (resample dialogues with replacement, refit,
repeat) rather than an analytic p-value. With only 11 categories over 1,017
dialogues, the "insufficient data" floor (15 dialogues) is not expected to
exclude anything -- unlike the 41-strategy version, where 16 of 41 strategies
fell below it.

**Verdict per category:** "USE" if the bootstrap 95% CI sits entirely above
zero, "AVOID" if entirely below zero, "no clear effect" if the CI straddles
zero. Cross-checked against Section 15's univariate chi2 direction as a
sanity signal -- agreement between the marginal (chi2) and adjusted (this
model) view is stronger evidence than either alone.

**Read this as correlational, not causal**, exactly like every other model in
Phase 4: persuaders were not randomly assigned a category of appeal, so "USE"
means "this category's presence was associated with more donations, net of
the other categories and covariates fit here" -- not "adopting it will cause
a lift." That is the honest ceiling of what this observational corpus can
support, and the same standard the source paper and Section 15 are held to.


In [ ]:
# =============================================================
# 15b. WHICH INDIVIDUAL CATEGORIES PREDICT DONATION -- MULTIVARIATE,
#      BOOTSTRAP-CI EFFECT MODEL  (Phase 4.3b)
# =============================================================
# Section 15's chi2 table is univariate (one category at a time). This fits
# every category SIMULTANEOUSLY in one L2-regularised logistic model (same
# reasoning as `mcfadden()` above), controlling for dialogue length and the
# persuadee covariates, so each category's coefficient is its association
# net of the others -- then quantifies uncertainty with a dialogue-level
# bootstrap since L2 shrinkage invalidates textbook MLE standard errors.
# Correlational, not causal -- see the markdown above.
if RUN_PHASE4:
    from sklearn.linear_model import LogisticRegression as _LR2

    MIN_CAT_SUPPORT = 15   # dialogues; below this even a bootstrap CI is not trustworthy
    mv_cols_all = [c for c in (re.sub(r"[^0-9a-zA-Z_]", "_", f"cat_{c.replace(' ', '_')}") for c in STRATS)
                   if c in D_]
    support = {c: int(D_[c].sum()) for c in mv_cols_all}
    mv_cols = [c for c in mv_cols_all if support[c] >= MIN_CAT_SUPPORT]
    dropped = [c for c in mv_cols_all if support[c] < MIN_CAT_SUPPORT]
    print(f"categories with >= {MIN_CAT_SUPPORT} dialogues: {len(mv_cols)} / {len(mv_cols_all)}")
    if dropped:
        print(f"  excluded for insufficient support: {[c[4:] for c in dropped]}")

    ctrl_cols = [c for c in ["n_persuader_turns", "ee_sentiment", "ee_engagement"] if c in D_]
    feat_cols = mv_cols + ctrl_cols
    Xraw = D_[feat_cols].to_numpy(float)
    mu, sd = Xraw.mean(0), Xraw.std(0) + 1e-9
    X = (Xraw - mu) / sd
    y_ = D_["donated"].to_numpy()

    def _fit_coefs(Xf, yf, C=1.0):
        clf = _LR2(C=C, max_iter=5000).fit(Xf, yf)
        return clf.coef_[0]

    beta_hat = _fit_coefs(X, y_)

    N_BOOT = 800
    rng = np.random.default_rng(SEED)
    n = len(y_)
    boot = np.zeros((N_BOOT, len(feat_cols)))
    for b in range(N_BOOT):
        idx = rng.integers(0, n, n)
        yb = y_[idx]
        if yb.min() == yb.max():        # a resample with only one class -- redraw once
            idx = rng.integers(0, n, n); yb = y_[idx]
        boot[b] = _fit_coefs(X[idx], yb)

    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    tbl = pd.DataFrame({
        "feature": feat_cols,
        "std_coef": beta_hat.round(4),
        "ci_lo": lo.round(4), "ci_hi": hi.round(4),
        "n_with": [support.get(c) for c in feat_cols],
    })

    def _verdict(row):
        if row["feature"] not in mv_cols:
            return "control"
        if row["n_with"] is not None and row["n_with"] < MIN_CAT_SUPPORT:
            return "insufficient data"
        if row["ci_lo"] > 0:
            return "USE -- reliably positive"
        if row["ci_hi"] < 0:
            return "AVOID -- reliably negative"
        return "no clear effect"
    tbl["verdict"] = tbl.apply(_verdict, axis=1)
    tbl["category"] = tbl["feature"].str.replace("^cat_", "", regex=True).str.replace("_", " ")

    # cross-check against Section 15's UNIVARIATE chi2 direction, so a reader
    # can see whether the marginal and adjusted views agree
    _chi_map = chi_cat.set_index("feature")["delta_pp"].to_dict() if len(chi_cat) else {}
    tbl["univariate_delta_pp"] = tbl["category"].map(_chi_map)
    tbl["agrees_with_univariate"] = np.where(
        tbl["feature"].isin(mv_cols) & tbl["univariate_delta_pp"].notna(),
        np.sign(tbl["std_coef"]) == np.sign(tbl["univariate_delta_pp"]), None)

    tbl = tbl[tbl.feature != ""].sort_values("std_coef", ascending=False)
    tbl.to_csv(os.path.join(CONFIG["out_dir"], "phase4_category_effects.csv"), index=False)

    use_tbl   = tbl[tbl.verdict.str.startswith("USE", na=False)]
    avoid_tbl = tbl[tbl.verdict.str.startswith("AVOID", na=False)]
    print(f"\n{len(mv_cols)} categories modelled simultaneously "
          f"(+ {len(ctrl_cols)} controls: {ctrl_cols}), {N_BOOT}-round dialogue bootstrap 95% CI\n")
    print("CATEGORIES TO USE (bootstrap 95% CI entirely above zero, net of co-occurring "
          "categories and dialogue length / persuadee covariates):")
    print(use_tbl[["category","std_coef","ci_lo","ci_hi","n_with","univariate_delta_pp"]]
          .to_string(index=False) if len(use_tbl) else "  (none)")
    print("\nCATEGORIES TO AVOID (bootstrap 95% CI entirely below zero):")
    print(avoid_tbl[["category","std_coef","ci_lo","ci_hi","n_with","univariate_delta_pp"]]
          .to_string(index=False) if len(avoid_tbl) else "  (none)")
    _n_nce = int((tbl.verdict == "no clear effect").sum())
    _n_ins = int((tbl.verdict == "insufficient data").sum())
    print(f"\n{_n_nce} categories: no clear effect once co-occurring categories/controls are "
          f"accounted for. {_n_ins} categories: too few dialogues (<{MIN_CAT_SUPPORT}) to judge.")

    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(mv_cols))))
    _plot = tbl[tbl.feature.isin(mv_cols)].sort_values("std_coef").reset_index(drop=True)
    # NOTE: matplotlib.errorbar()'s ecolor takes ONE color for the whole call
    # (unlike scatter()'s c, which accepts an array) -- looping per verdict
    # group and issuing one errorbar()/scatter() call per group avoids that.
    _COLOR_OF = {"USE -- reliably positive": "tab:green",
                 "AVOID -- reliably negative": "tab:red",
                 "insufficient data": "lightgray",
                 "no clear effect": "tab:gray"}
    xerr_lo = np.maximum(_plot.std_coef - _plot.ci_lo, 0).to_numpy()
    xerr_hi = np.maximum(_plot.ci_hi - _plot.std_coef, 0).to_numpy()
    ypos = np.arange(len(_plot))
    for _verdict, _color in _COLOR_OF.items():
        _m = (_plot.verdict == _verdict).to_numpy()
        if not _m.any():
            continue
        ax.errorbar(_plot.std_coef.to_numpy()[_m], ypos[_m],
                   xerr=[xerr_lo[_m], xerr_hi[_m]],
                   fmt="none", ecolor=_color, elinewidth=1.5, capsize=2)
        ax.scatter(_plot.std_coef.to_numpy()[_m], ypos[_m], c=_color, s=18, zorder=3)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(ypos); ax.set_yticklabels(_plot.category, fontsize=8)
    ax.set_xlabel("standardized log-odds coefficient (95% dialogue-bootstrap CI)")
    ax.set_title("Which categories move donation probability, net of co-occurring categories?")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["out_dir"], "phase4_category_effects.png"), dpi=110)
    plt.show()
else:
    print("(skipped - no donation outcome)")


## 16. Phase 4.4 -- Guilt Induction re-test (not applicable at category granularity)

The paper's Guilt Induction finding is about one specific fine-grained
strategy, `guilt_induction`, which sits inside the broader "Emotional
Appeal" category alongside `emotion_appeal`, `empathy_and_perspective_taking`
and `hope_and_positive_impact` -- several of them positively framed, unlike
guilt. Retesting "Emotional Appeal" as a stand-in for "Guilt Induction" would
misrepresent the original claim rather than translate it, so this section is
skipped in the category-only pipeline. Section 15b's category-level model
still reports whether Emotional Appeal as a whole predicts donation; that's
the closest honest category-level signal available.


In [ ]:
# =============================================================
# 16. GUILT INDUCTION RE-TEST -- skipped (category-only pipeline)
# =============================================================
# See the markdown above: Guilt Induction is a single fine-grained strategy
# with no faithful category-level equivalent, so there is nothing to compute
# here. `guilt = None` so Section 18's summary can check for this cleanly.
guilt = None
print("skipped: Guilt Induction has no category-level equivalent (see markdown above)")


## 17. Phase 4.5 -- conditional / deferred modifier as an outcome

Multinomial logit `modifier ~ categories (+ richness)`. Tests whether specific
category mixes push persuadees toward a *conditional* or *deferred*
non-commitment rather than a flat refusal -- a distinction that exists only
because of the team's own hand annotation.


In [ ]:
# =============================================================
# 17. MODIFIER MULTINOMIAL MODEL  (Phase 4.5)
# =============================================================
if RUN_PHASE4:
    sm_df = D_[D_.modifier.isin(CONFIG["modifier_classes"])].copy()
    sm_df["modifier_code"] = sm_df.modifier.map(
        {c: i for i, c in enumerate(CONFIG["modifier_classes"])}).astype(int)
    mod_out = {"class_balance": sm_df.modifier.value_counts().to_dict(),
               "coding": "0=none (base), 1=deferred, 2=conditional",
               "n_dialogues": int(len(sm_df))}
    if not HAVE_SM:
        mod_out["error"] = "statsmodels unavailable"
    elif sm_df.modifier_code.nunique() < 2:
        mod_out["error"] = "only one modifier class present"
    else:
        try:
            mnl = smf.mnlogit("modifier_code ~ " + " + ".join(cat_cols +
                              ["distinct_categories", "ee_sentiment"]),
                              data=sm_df).fit(disp=0, method="lbfgs", maxiter=1000)
            k = mnl.params.shape[1]
            names = ["deferred_vs_none", "conditional_vs_none"][:k]
            params = mnl.params.copy(); params.columns = names
            pvals  = mnl.pvalues.copy(); pvals.columns  = [n.split("_vs_")[0] + "_p" for n in names]
            tbl = params.join(pvals).round(3)
            tbl.to_csv(os.path.join(CONFIG["out_dir"], "modifier_analysis.csv"))
            mod_out["pseudo_r2"] = round(float(mnl.prsquared), 4)
            pcols = [c for c in tbl.columns if c.endswith("_p")]
            sig = tbl[(tbl[pcols] < 0.05).any(axis=1)]
            mod_out["significant_terms"] = sig.reset_index().to_dict("records")
            print("significant terms:\n", sig.to_string() if len(sig) else "  (none at p<0.05)")
        except Exception as e:
            mod_out["error"] = f"{type(e).__name__}: {e}"; print("mnlogit failed:", e)
    json.dump(mod_out, open(os.path.join(CONFIG["out_dir"], "modifier_analysis.json"), "w"),
              indent=2, default=str)
    print(json.dumps({k: v for k, v in mod_out.items() if k != "significant_terms"},
                     indent=2, default=str))
else:
    print("(skipped - no donation outcome)")


In [ ]:
# =============================================================
# 17b. PHASE 4 HEADLINE FIGURES
# =============================================================
if RUN_PHASE4 and models:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
    pr2 = pd.DataFrame(models).set_index("model")[["pseudo_r2_in_sample", "pseudo_r2_cv"]]
    pr2.plot.bar(ax=ax[0], color=["#55A868", "#4C72B0"])
    ax[0].axhline(0.08, ls="--", c="grey")
    ax[0].set_title("McFadden pseudo-R2 (dashed = paper single-label upper ~0.08)")
    ax[0].tick_params(axis="x", rotation=20, labelsize=8)
    if len(chi_cat):
        topd = chi_cat.head(8).set_index("feature")["delta_pp"].sort_values()
        topd.plot.barh(ax=ax[1], color=["#C44E52" if v < 0 else "#4C72B0" for v in topd])
        ax[1].set_title("Donation-rate delta (pp): category present - absent")
    plt.tight_layout(); plt.savefig(os.path.join(CONFIG["out_dir"], "fig_phase4.png"), dpi=140); plt.show()
else:
    print("(skipped)")


## 18. Summary & artifacts


In [ ]:
# =============================================================
# 18. SUMMARY + ZIP
# =============================================================
best_m = best_r["metrics_best"]
# The strongest classical comparator, whether it came from S9 or from the S8b
# shallow members (which are the same estimator families, decoded properly).
_bpool = {("baseline::" + k): v["micro_f1"] for k, v in baseline_metrics.items()}
_bpool.update({n: r["metrics_tuned"]["micro_f1"] for n, r in results.items()
               if r.get("kind") == "shallow"})
best_base_name, best_base_f1 = max(_bpool.items(), key=lambda kv: kv[1])

lines = [
    f"# Multi-label pipeline - run summary  ({split_mode}, seed {SEED})",
    "",
    f"turns {len(df)} | dialogues {df.dialogue_id.nunique()} | context {CONFIG['context_turns']} turns"
    f" ({'joined' if HAVE_TURNS else 'NOT AVAILABLE - no context'})",
    f"members: {', '.join(n for n in results if n != ENSEMBLE_NAME)}"
    + (f"  ->  {ENSEMBLE_NAME} = {results[ENSEMBLE_NAME].get('candidate')}"
       f" (chosen from {results[ENSEMBLE_NAME].get('candidates_tried')} candidates)"
       if ENSEMBLE_NAME in results else ""),
    f"loss {CONFIG['loss_type']} | pooling {CONFIG['pooling']} | amp {USE_AMP} | "
    f"LLRD {CONFIG['use_llrd']} | augmentation {CONFIG['use_augmentation']}",
    f"-> headline model: {BEST}",
    "",
    f"## Phase 3 - category classifier (test split, decode selected on val {TGT_METRIC})",
    f"macro-F1 {best_m['macro_f1']:.4f} over all {len(STRATS)} categories | "
    f"macro-F1 {best_m['macro_f1_eval']:.4f} over the {best_m['n_eval_labels']} with "
    f">={CONFIG.get('macro_min_test_support', 10)} test positives",
    f"micro-F1 {best_m['micro_f1']:.4f} | "
    f"weighted-F1 {best_m['weighted_f1']:.4f} | samples-F1 {best_m['samples_f1']:.4f}",
    f"Hamming {best_m['hamming_loss']:.4f} | Jaccard {best_m['jaccard_samples']:.4f} | "
    f"subset-acc {best_m['subset_accuracy']:.4f} | card pred/true "
    f"{best_m['card_pred']:.2f}/{Y_STRAT_TE.sum(1).mean():.2f}",
    f"best non-transformer baseline ({best_base_name}) micro-F1: {best_base_f1:.4f}",
    f"selected on cross-fitted val: {best_r.get('val_cf'):.4f} cf-val | operating point "
    f"{best_r.get('rule')}{'+top1' if best_r.get('force_top1') else ''}",
    f"categories with no support in some split: {ZERO_SUPPORT or 'none'}",
    "",
]
if RUN_PHASE4 and models:
    m1r = next((m for m in models if m["model"] == "M1_categories"), None)
    m4r = next((m for m in models if m["model"] == "M4_full_cooccurrence"), None)
    lines += [
        f"## Phase 4 - donation outcome  (features from: {CONFIG['phase4_label_source']})",
        f"dialogues {len(D)} | donate rate {D.donated.mean():.3f}",
        f"distinct categories / dialogue: mean {feat.distinct_categories.mean():.2f}",
        f"McFadden pseudo-R2  M1 categories: in-sample {m1r['pseudo_r2_in_sample']:.4f} / "
        f"CV {m1r['pseudo_r2_cv']:.4f}" if m1r else "",
        f"                    M4 +co-occ    : in-sample {m4r['pseudo_r2_in_sample']:.4f} / "
        f"CV {m4r['pseudo_r2_cv']:.4f}" if m4r else "",
        "                    (paper single-label ~0.015-0.08)",
        (f"Category-level effects (15b, multivariate + dialogue-bootstrap CI): "
         f"{len(use_tbl)} categories reliably USE, {len(avoid_tbl)} reliably AVOID "
         f"(see phase4_category_effects.csv)") if "use_tbl" in globals() else "",
    ]
else:
    lines += ["## Phase 4 - SKIPPED (persuader_turns.csv / donation outcome not attached)"]

summary = "\n".join(l for l in lines if l != "") + "\n"
open(os.path.join(CONFIG["out_dir"], "SUMMARY.md"), "w", encoding="utf-8").write(summary)
print(summary)

zp = os.path.join(CONFIG["out_dir"], "phase3_4_multilabel_artifacts.zip")
pats = ["*.json", "*.csv", "*.png", "SUMMARY.md", "pred_multilabel_full.jsonl"]
if CONFIG["zip_checkpoints"]:
    pats.append(os.path.join(CONFIG["checkpoint_dir_name"], "*.pt"))
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in pats:
        for p in glob.glob(os.path.join(CONFIG["out_dir"], pat)):
            if os.path.abspath(p) != os.path.abspath(zp):
                z.write(p, os.path.relpath(p, CONFIG["out_dir"]))
print("zipped ->", zp, "(%.1f MB)" % (os.path.getsize(zp) / 1e6))
print("\ncheckpoints kept (not zipped by default):",
      [os.path.basename(p) for p in glob.glob(os.path.join(CKPT_DIR, '*.pt'))])

if CONFIG["SMOKE_TEST"]:
    _ran_ensemble = ENSEMBLE_NAME in results
    _ran_xgb      = "tfidf_xgboost_chain" in baseline_metrics
    _ran_aug      = bool(train_df["_augmented"].any())
    _ran_phase4   = RUN_PHASE4 and bool(models)
    _ran_strategy_effects = RUN_PHASE4 and "tbl" in globals() and len(tbl) > 0
    _checks = {"ensemble": _ran_ensemble, "xgboost_chain": _ran_xgb,
               "shallow_members": bool(SHALLOW_NAMES), "level2_stacker": bool(STACK_RAN),
               "augmentation": _ran_aug, "phase4": _ran_phase4,
               "phase4_strategy_effects": bool(_ran_strategy_effects)}
    _all_ok = all(_checks.values()) and not failed
    print("\n" + "=" * 60)
    print(" SMOKE TEST " + ("PASSED" if _all_ok else "COMPLETED WITH GAPS"))
    print("=" * 60)
    print("sections exercised:", json.dumps(_checks, indent=2))
    if failed:
        print("encoders that failed to train:", json.dumps(failed, indent=2))
    if not _all_ok:
        print("\n[NOTE] not every optional section ran (see above) -- this can be a real")
        print("       config gap (e.g. only 1 encoder configured -> no ensemble) rather")
        print("       than a bug; check which of the four before trusting a full run.")
    print("\nNo exception reached this point -> every executed cell completed.")
    print("Set CONFIG['SMOKE_TEST'] = False (and FAST_DEV_RUN = False) for the real run.")


## 19. Notes on this run

1. **This is the category-level pivot.** Every earlier revision of this
   pipeline predicted the 41 (later 39) fine-grained strategies. This one
   predicts the 11 coarser categories instead -- no fine-grained strategy
   output anywhere, Phase 4 rebuilt around category features throughout. The
   trade-off: no "which specific strategy" answer, in exchange for every
   class having far more supporting examples than the rarest strategies ever
   had, and (the thing to actually check once this run completes) whether
   that turns into a real metric improvement or just a smaller, easier label
   space scoring higher for the same underlying reason.
2. **Dialogue-level split, never turn-level.** Turns of one dialogue share
   context text; a turn-level split would leak.
3. **The annotation ceiling (Section 3c) is the number to read every score
   against, not 1.0.** Same-annotator agreement on near-duplicate text: F1
   0.805. Cross-annotator: F1 0.578. Under the standard miss-only noise model
   that implies a practical ceiling around **0.89** micro-F1. Report scores
   as a fraction of that ceiling. Computed on raw text pairs, so it is
   unaffected by the category pivot.
4. **No `[A:<annotator>]` tag, no annotator-ID feature anywhere** (including
   the shallow-member/stacker meta-features) -- a rater-identity leak, not a
   property of the conversation. A live deployment has no annotator ID for a
   new conversation.
5. **No whole-dialogue probability smoothing.** It blended a turn's
   prediction toward the mean over *other* turns of the same dialogue,
   including turns after the one being classified -- non-causal for a live,
   turn-by-turn deployment.
6. **Decoupled long-tail training and tau-normalization were both piloted
   and dropped** in the strategy-level version of this pipeline -- neither
   recovered any performance there. Not revisited here: with only 11
   categories, the long-tail problem those techniques targeted is not
   expected to exist in the first place.
7. **Context width is measured, not assumed.** Trained at multiple widths;
   let cross-fitted validation and the ensemble search pick or drop each one
   rather than assuming either way.
8. **The decision-rule search beats "tune each label's own F1"** on every
   prior version of this corpus at strategy granularity -- kept as a
   reported baseline (`@old perlabelF1`) here too, for continuity.
9. **The ensemble/stacking zoo never assumes a winner** -- mean, weighted
   blends, power-means, and two level-2 stackers all compete on the same
   grouped, cross-fitted validation score, and either stacker is free to
   lose to a plain average.
10. **Phase 4's category-level effects (Section 15b)** are a multivariate,
    bootstrap-CI model of which categories predict donation net of the
    others, cross-checked against the univariate chi2 view. Correlational,
    not causal, like every other Phase 4 model. The strategy-level
    Guilt-Induction re-test (Section 16 in the strategy-level pipeline) has
    no faithful category-level equivalent and is skipped here -- see Section
    16's markdown.
